In [2]:
import requests

# Test OSRM with a simple route
# From Accra to Kumasi
test_url = "http://localhost:5000/route/v1/driving/-0.186964,5.603717;-1.623889,6.688611"
response = requests.get(test_url)
data = response.json()

if data['code'] == 'Ok':
    duration_minutes = round(data['routes'][0]['duration'] / 60, 1)
    distance_km = round(data['routes'][0]['distance'] / 1000, 1)
    print(f"✅ OSRM is working!")
    print(f"   Accra → Kumasi: {duration_minutes} minutes, {distance_km} km")
else:
    print(f"❌ Something went wrong: {data}")

✅ OSRM is working!
   Accra → Kumasi: 201.5 minutes, 247.1 km


In [4]:
import pandas as pd
import numpy as np
import requests
import time
import rasterio
from rasterio.transform import xy

print("✅ Libraries imported!")

✅ Libraries imported!


In [5]:
master = pd.read_csv(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\master_dataset_v3.csv")
print(f"✅ Master dataset: {len(master):,} facilities")

✅ Master dataset: 9,978 facilities


In [6]:
CENSUS_TOTAL = 30_832_019

with rasterio.open(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\gha_pd_2020_1km_UNadj.tif") as src:
    pop_data = src.read(1)
    transform = src.transform
    nodata = src.nodata

rows, cols = np.where((pop_data > 0) & (pop_data != nodata))
populations = pop_data[rows, cols]
lons, lats = xy(transform, rows, cols)

scale_factor = CENSUS_TOTAL / populations.sum()

pop_df = pd.DataFrame({
    'lat': np.array(lats),
    'lon': np.array(lons),
    'population': populations * scale_factor
})

print(f"✅ Population points: {len(pop_df):,}")
print(f"✅ Total population: {pop_df['population'].sum():,.0f}")

✅ Population points: 278,001
✅ Total population: 30,832,016


In [7]:
import requests
import time

print("Speed test — OSRM routing...\n")

# Take 100 sample population points
sample = pop_df.sample(100, random_state=42)

# Take all emergency facilities
emergency = master[master['Facility_Type'].isin(['HOSPITAL', 'DISTRICT HOSPITAL'])]

print(f"Testing 100 population points → {len(emergency)} emergency facilities...\n")

start = time.time()

for i, (_, pop_point) in enumerate(sample.iterrows()):
    # Build coordinate string
    # First coordinate = source (population point)
    # Rest = destinations (facilities)
    coords = f"{pop_point['lon']},{pop_point['lat']}"
    for _, fac in emergency.iterrows():
        coords += f";{fac['Longitude']},{fac['Latitude']}"
    
    # OSRM table API — returns travel times from source to all destinations
    url = f"http://localhost:5000/table/v1/driving/{coords}?sources=0"
    response = requests.get(url)
    data = response.json()

elapsed = time.time() - start
estimated_hours = round((elapsed / 100) * 278001 / 3600, 1)

print(f"100 population points took: {round(elapsed, 1)} seconds")
print(f"Estimated for all 278,001 points: {estimated_hours} hours")

Speed test — OSRM routing...

Testing 100 population points → 726 emergency facilities...

100 population points took: 258.2 seconds
Estimated for all 278,001 points: 199.4 hours


In [9]:
import requests
import time
import numpy as np
from scipy.spatial import cKDTree

print("Speed test — KD-Tree + OSRM combined...\n")

# -------------------------------------------------------
# STEP 1: BUILD K-D TREE FOR EMERGENCY FACILITIES
# We use emergency as our test case (726 facilities)
# -------------------------------------------------------
emergency = master[master['Facility_Type'].isin(['HOSPITAL', 'DISTRICT HOSPITAL'])]

# Build a k-d tree on facility coordinates
# This lets us find nearest facilities by straight line in milliseconds
fac_coords = emergency[['Longitude', 'Latitude']].values
fac_tree = cKDTree(fac_coords)

print(f"✅ KD-Tree built on {len(emergency)} emergency facilities")

# -------------------------------------------------------
# STEP 2: FOR EACH POPULATION POINT, FIND 10 NEAREST
# FACILITIES BY STRAIGHT LINE (no road routing yet)
# -------------------------------------------------------

# Use a sample of 1000 population points for speed test
sample = pop_df.sample(1000, random_state=42)
sample_coords = sample[['lon', 'lat']].values

# k=10 means find 10 nearest facilities by straight line
# This is instant — pure maths, no routing
K = 10
_, neighbor_idx = fac_tree.query(sample_coords, k=K)

print(f"✅ Found {K} nearest candidates per population point")
print(f"   (straight line only — no routing yet)")

# -------------------------------------------------------
# STEP 3: FOR EACH POPULATION POINT, ROUTE TO ITS
# 10 CANDIDATES ONLY USING OSRM
# We send requests in small batches to avoid crashing OSRM
# -------------------------------------------------------

BATCH_SIZE = 25  # number of population points per request
results = []  # store minimum travel time per population point

start = time.time()

# Split sample into batches
for batch_start in range(0, len(sample), BATCH_SIZE):
    batch_end = min(batch_start + BATCH_SIZE, len(sample))
    batch = sample.iloc[batch_start:batch_end]
    batch_neighbors = neighbor_idx[batch_start:batch_end]
    
    # For this batch, collect all unique candidate facility indices
    # Each population point has 10 candidates
    # We deduplicate to avoid sending same facility coordinates twice
    unique_fac_indices = list(set(batch_neighbors.flatten()))
    candidate_facilities = emergency.iloc[unique_fac_indices]
    
    # Build coordinate string for OSRM
    # Format: lon,lat;lon,lat;lon,lat...
    # Population points first, then facilities
    all_coords = []
    
    # Add population points (these are our SOURCES)
    for _, row in batch.iterrows():
        all_coords.append(f"{row['lon']},{row['lat']}")
    
    # Add candidate facilities (these are our DESTINATIONS)
    for _, row in candidate_facilities.iterrows():
        all_coords.append(f"{row['Longitude']},{row['Latitude']}")
    
    coords_str = ";".join(all_coords)
    
    # Source indices = 0 to len(batch)-1 (population points)
    sources = ";".join(str(i) for i in range(len(batch)))
    
    # Destination indices = len(batch) onwards (facilities)
    destinations = ";".join(str(i) for i in range(len(batch), len(batch) + len(candidate_facilities)))
    
    # Send request to OSRM
    # The table API returns a matrix of travel times in seconds
    url = f"http://localhost:5000/table/v1/driving/{coords_str}?sources={sources}&destinations={destinations}"
    response = requests.get(url)
    data = response.json()
    
    if data['code'] == 'Ok':
        # durations is a matrix: rows = population points, cols = facilities
        # Each value is travel time in SECONDS (or null if unreachable)
        durations = np.array(data['durations'], dtype=float)
        
        # For each population point, find the minimum travel time
        # across all its candidate facilities
        # Convert from seconds to minutes
        min_times = np.nanmin(durations, axis=1) / 60
        results.extend(min_times)
    else:
        # If OSRM fails, fill with NaN for this batch
        results.extend([np.nan] * len(batch))

elapsed = time.time() - start

# -------------------------------------------------------
# STEP 4: REPORT RESULTS
# -------------------------------------------------------

estimated_hours = round((elapsed / 1000) * 278001 / 3600, 1)
estimated_minutes = round((elapsed / 1000) * 278001 / 60, 1)

print(f"\n1,000 population points took: {round(elapsed, 1)} seconds")
print(f"Estimated for all 278,001 points: {estimated_hours} hours ({estimated_minutes} minutes)")
print(f"\nSample results:")
print(f"  Average travel time to nearest emergency facility: {round(np.nanmean(results), 1)} min")
print(f"  Minimum: {round(np.nanmin(results), 1)} min")
print(f"  Maximum: {round(np.nanmax(results), 1)} min")

Speed test — KD-Tree + OSRM combined...

✅ KD-Tree built on 726 emergency facilities
✅ Found 10 nearest candidates per population point
   (straight line only — no routing yet)

1,000 population points took: 21.0 seconds
Estimated for all 278,001 points: 1.6 hours (97.2 minutes)

Sample results:
  Average travel time to nearest emergency facility: 46.8 min
  Minimum: 1.0 min
  Maximum: 210.3 min


In [10]:
import requests
import time
import numpy as np
import pandas as pd
import pickle
from scipy.spatial import cKDTree

print("=" * 60)
print("NEAREST FACILITY TRAVEL TIME — ALL 7 GROUPINGS")
print("KD-TREE + OSRM APPROACH")
print("=" * 60)

# -------------------------------------------------------
# CONFIGURATION
# -------------------------------------------------------

# Number of nearest candidates to check by straight line first
# 10 is safe — protects against unreachable nearest facilities
K = 10

# Number of population points per OSRM request
# 25 is safe — avoids crashing OSRM with too large a request
BATCH_SIZE = 25

# Where to save results
SAVE_PATH = r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project"

# -------------------------------------------------------
# STEP 1: DEFINE THE 7 FACILITY GROUPINGS
# -------------------------------------------------------

groupings = {
    'any': master,
    
    'chps': master[
        master['Facility_Type'] == 'CHPS'
    ],
    
    'maternity': master[
        master['Facility_Type'].isin(['CHPS', 'MATERNITY HOME'])
    ],
    
    'outpatient': master[
        master['Facility_Type'].isin(['CLINIC', 'HEALTH CENTRE', 'POLYCLINIC'])
    ],
    
    'emergency': master[
        master['Facility_Type'].isin(['HOSPITAL', 'DISTRICT HOSPITAL'])
    ],
    
    'specialist': master[
        master['Facility_Type'].isin([
            'REGIONAL HOSPITAL',
            'TEACHING HOSPITAL',
            'UNIVERSITY HOSPITAL/CLINIC'
        ])
    ],
    
    'psychiatric': master[
        master['Facility_Type'] == 'PSYCHIATRIC HOSPITAL'
    ],
}

print("\nFacility counts per grouping:")
for name, df in groupings.items():
    print(f"  {name:<12}: {len(df):,} facilities")

print(f"\nPopulation points: {len(pop_df):,}")

# -------------------------------------------------------
# STEP 2: CORE FUNCTION — ONE GROUPING AT A TIME
# -------------------------------------------------------

def compute_grouping(facility_df, label, save_path):
    """
    For each population point:
    1. Use KD-Tree to find K nearest facilities by straight line
    2. Send to OSRM in batches to get real road travel times
    3. Keep the minimum travel time (nearest facility)
    4. Save progress every 10,000 points so we don't lose work
    """
    
    n_pop = len(pop_df)
    pop_coords = pop_df[['lon', 'lat']].values
    
    print(f"\n{'='*60}")
    print(f"  {label.upper()}")
    print(f"  {len(facility_df):,} facilities")
    print(f"{'='*60}")
    
    # --- CHECK FOR EXISTING CHECKPOINT ---
    # If we stopped halfway, load from where we left off
    checkpoint_path = f"{save_path}\\checkpoint_{label}.pkl"
    
    try:
        with open(checkpoint_path, 'rb') as f:
            checkpoint = pickle.load(f)
        results = checkpoint['results']
        start_idx = checkpoint['next_idx']
        print(f"  ⚡ Resuming from checkpoint at index {start_idx:,}")
    except:
        # No checkpoint — start fresh
        results = [np.nan] * n_pop
        start_idx = 0
        print(f"  Starting fresh...")
    
    # --- BUILD KD-TREE FOR THIS GROUPING ---
    fac_coords = facility_df[['Longitude', 'Latitude']].values
    fac_tree = cKDTree(fac_coords)
    
    # Adjust K if fewer facilities than K
    k = min(K, len(facility_df))
    
    # --- FIND K NEAREST BY STRAIGHT LINE ---
    print(f"  Finding {k} nearest candidates by straight line...")
    _, neighbor_idx = fac_tree.query(pop_coords, k=k)
    if k == 1:
        neighbor_idx = neighbor_idx.reshape(-1, 1)
    print(f"  ✅ Done! Starting road network routing...")
    
    # --- ROUTE IN BATCHES ---
    start_time = time.time()
    total_batches = (n_pop - start_idx) // BATCH_SIZE + 1
    batch_times = []  # track time per batch for ETA
    
    for batch_start in range(start_idx, n_pop, BATCH_SIZE):
        batch_end = min(batch_start + BATCH_SIZE, n_pop)
        batch = pop_df.iloc[batch_start:batch_end]
        batch_neighbors = neighbor_idx[batch_start:batch_end]
        
        batch_start_time = time.time()
        
        # Get unique candidate facility indices for this batch
        unique_fac_indices = list(set(batch_neighbors.flatten()))
        candidate_facilities = facility_df.iloc[unique_fac_indices]
        
        # Build OSRM coordinate string
        # Population points first (sources), facilities second (destinations)
        all_coords = []
        for _, row in batch.iterrows():
            all_coords.append(f"{row['lon']},{row['lat']}")
        for _, row in candidate_facilities.iterrows():
            all_coords.append(f"{row['Longitude']},{row['Latitude']}")
        
        coords_str = ";".join(all_coords)
        sources = ";".join(str(i) for i in range(len(batch)))
        destinations = ";".join(str(i) for i in range(len(batch), len(batch) + len(candidate_facilities)))
        
        # Send to OSRM
        url = f"http://localhost:5000/table/v1/driving/{coords_str}?sources={sources}&destinations={destinations}"
        
        try:
            response = requests.get(url, timeout=30)
            data = response.json()
            
            if data['code'] == 'Ok':
                # Get travel time matrix (seconds) → convert to minutes
                durations = np.array(data['durations'], dtype=float)
                min_times = np.nanmin(durations, axis=1) / 60
                
                # Store results for this batch
                for i, t in enumerate(min_times):
                    results[batch_start + i] = t
            else:
                # OSRM returned an error — leave as NaN
                pass
                
        except Exception as e:
            # Network error — leave as NaN and continue
            pass
        
        # Track batch time for ETA calculation
        batch_elapsed = time.time() - batch_start_time
        batch_times.append(batch_elapsed)
        
        # --- PROGRESS UPDATE EVERY 500 POINTS ---
        if batch_start % 500 == 0 or batch_end == n_pop:
            points_done = batch_end - start_idx
            points_remaining = n_pop - batch_end
            
            # Calculate ETA based on average of last 20 batches
            recent_times = batch_times[-20:]
            avg_batch_time = np.mean(recent_times)
            batches_remaining = points_remaining / BATCH_SIZE
            eta_seconds = avg_batch_time * batches_remaining
            eta_minutes = round(eta_seconds / 60, 1)
            eta_hours = round(eta_seconds / 3600, 1)
            
            # Overall progress
            pct = round(batch_end / n_pop * 100, 1)
            elapsed_total = round((time.time() - start_time) / 60, 1)
            
            # Format ETA nicely
            if eta_hours >= 1:
                eta_str = f"{eta_hours} hours"
            else:
                eta_str = f"{eta_minutes} minutes"
            
            print(f"  {batch_end:,}/{n_pop:,} ({pct}%) — "
                  f"Elapsed: {elapsed_total} min — "
                  f"ETA: {eta_str}")
        
        # --- SAVE CHECKPOINT EVERY 10,000 POINTS ---
        if batch_end % 10000 < BATCH_SIZE:
            checkpoint = {
                'results': results,
                'next_idx': batch_end,
                'label': label
            }
            with open(checkpoint_path, 'wb') as f:
                pickle.dump(checkpoint, f)
            print(f"  💾 Checkpoint saved at {batch_end:,} points")
    
    # --- DONE ---
    elapsed_total = round((time.time() - start_time) / 60, 1)
    results_arr = np.array(results)
    
    # Summary statistics
    reachable_30  = np.sum(results_arr <= 30)
    reachable_60  = np.sum(results_arr <= 60)
    reachable_120 = np.sum(results_arr <= 120)
    unreachable   = np.sum(np.isnan(results_arr))
    
    print(f"\n  ✅ {label} DONE in {elapsed_total} minutes!")
    print(f"  Within 30 min:  {reachable_30:,} ({reachable_30/n_pop*100:.1f}%)")
    print(f"  Within 60 min:  {reachable_60:,} ({reachable_60/n_pop*100:.1f}%)")
    print(f"  Within 120 min: {reachable_120:,} ({reachable_120/n_pop*100:.1f}%)")
    print(f"  Unreachable:    {unreachable:,} ({unreachable/n_pop*100:.1f}%)")
    
    # Clean up checkpoint file
    try:
        import os
        os.remove(checkpoint_path)
    except:
        pass
    
    return results_arr

# -------------------------------------------------------
# STEP 3: RUN ALL 7 GROUPINGS AND SAVE RESULTS
# -------------------------------------------------------

all_results = {}

for group_name, facility_df in groupings.items():
    all_results[group_name] = compute_grouping(
        facility_df, 
        group_name, 
        SAVE_PATH
    )
    
    # Save intermediate results after each grouping
    # So if something goes wrong we don't lose completed groupings
    for name, times in all_results.items():
        pop_df[f'nearest_{name}_min'] = times
    
    pop_df.to_csv(
        f"{SAVE_PATH}\\nearest_facility_times.csv",
        index=False
    )
    print(f"\n  💾 Intermediate save complete after {group_name}!")

# -------------------------------------------------------
# STEP 4: FINAL SAVE
# -------------------------------------------------------

print("\n\nSaving final results...")

for group_name, times in all_results.items():
    pop_df[f'nearest_{group_name}_min'] = times

pop_df.to_csv(
    f"{SAVE_PATH}\\nearest_facility_times.csv",
    index=False
)

with open(f"{SAVE_PATH}\\nearest_facility_times.pkl", 'wb') as f:
    pickle.dump(pop_df, f)

print(f"✅ Saved to nearest_facility_times.csv")
print(f"✅ Saved to nearest_facility_times.pkl")
print(f"\n🎉 ALL DONE!")

NEAREST FACILITY TRAVEL TIME — ALL 7 GROUPINGS
KD-TREE + OSRM APPROACH

Facility counts per grouping:
  any         : 9,978 facilities
  chps        : 6,733 facilities
  maternity   : 6,981 facilities
  outpatient  : 2,237 facilities
  emergency   : 726 facilities
  specialist  : 28 facilities
  psychiatric : 5 facilities

Population points: 278,001

  ANY
  9,978 facilities
  Starting fresh...
  Finding 10 nearest candidates by straight line...
  ✅ Done! Starting road network routing...
  25/278,001 (0.0%) — Elapsed: 0.0 min — ETA: 17.6 minutes
  525/278,001 (0.2%) — Elapsed: 0.1 min — ETA: 34.2 minutes
  1,025/278,001 (0.4%) — Elapsed: 0.1 min — ETA: 29.8 minutes
  1,525/278,001 (0.5%) — Elapsed: 0.2 min — ETA: 25.5 minutes
  2,025/278,001 (0.7%) — Elapsed: 0.2 min — ETA: 27.2 minutes
  2,525/278,001 (0.9%) — Elapsed: 0.3 min — ETA: 25.2 minutes
  3,025/278,001 (1.1%) — Elapsed: 0.3 min — ETA: 27.2 minutes
  3,525/278,001 (1.3%) — Elapsed: 0.4 min — ETA: 28.7 minutes
  4,025/278,001 

In [11]:
import pandas as pd
import numpy as np

pop_df = pd.read_csv(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\nearest_facility_times.csv")

groupings = {
    'Any facility': 'nearest_any_min',
    'CHPS': 'nearest_chps_min',
    'Maternity care': 'nearest_maternity_min',
    'Outpatient': 'nearest_outpatient_min',
    'Emergency': 'nearest_emergency_min',
    'Specialist': 'nearest_specialist_min',
    'Psychiatric': 'nearest_psychiatric_min',
}

thresholds = [5, 10, 15, 30, 60, 120]

print("=" * 75)
print("GHANA HEALTHCARE ACCESSIBILITY — TRAVEL TIME ANALYSIS")
print("=" * 75)

for label, col in groupings.items():
    print(f"\n{'─'*75}")
    print(f"  {label.upper()}")
    print(f"{'─'*75}")
    for t in thresholds:
        count = (pop_df[col] <= t).sum()
        pct = count / len(pop_df) * 100
        bar = '█' * int(pct / 2)
        print(f"  Within {t:>3} min: {pct:>5.1f}%  {bar}")
    unreachable = pop_df[col].isna().sum()
    if unreachable > 0:
        print(f"  Unreachable:   {unreachable:,} ({unreachable/len(pop_df)*100:.1f}%)")

GHANA HEALTHCARE ACCESSIBILITY — TRAVEL TIME ANALYSIS

───────────────────────────────────────────────────────────────────────────
  ANY FACILITY
───────────────────────────────────────────────────────────────────────────
  Within   5 min:  40.0%  ███████████████████
  Within  10 min:  63.3%  ███████████████████████████████
  Within  15 min:  76.1%  ██████████████████████████████████████
  Within  30 min:  90.6%  █████████████████████████████████████████████
  Within  60 min:  97.6%  ████████████████████████████████████████████████
  Within 120 min:  99.6%  █████████████████████████████████████████████████

───────────────────────────────────────────────────────────────────────────
  CHPS
───────────────────────────────────────────────────────────────────────────
  Within   5 min:  35.3%  █████████████████
  Within  10 min:  58.9%  █████████████████████████████
  Within  15 min:  73.0%  ████████████████████████████████████
  Within  30 min:  89.8%  █████████████████████████████████████

In [12]:
import pandas as pd
import numpy as np

# Build the summary table
rows = []
thresholds = [5, 10, 15, 30, 60, 120]

groupings = {
    'Any facility': 'nearest_any_min',
    'CHPS': 'nearest_chps_min',
    'Maternity care': 'nearest_maternity_min',
    'Outpatient': 'nearest_outpatient_min',
    'Emergency': 'nearest_emergency_min',
    'Specialist': 'nearest_specialist_min',
    'Psychiatric': 'nearest_psychiatric_min',
}

for label, col in groupings.items():
    row = {'Grouping': label}
    for t in thresholds:
        count = (pop_df[col] <= t).sum()
        pct = round(count / len(pop_df) * 100, 2)
        row[f'Count_within_{t}min'] = count
        row[f'Pct_within_{t}min'] = pct
    # Add median and mean
    row['Median_travel_time_min'] = round(pop_df[col].median(), 1)
    row['Mean_travel_time_min'] = round(pop_df[col].mean(), 1)
    row['Max_travel_time_min'] = round(pop_df[col].max(), 1)
    row['Unreachable_count'] = pop_df[col].isna().sum()
    rows.append(row)

summary_df = pd.DataFrame(rows)

# Save
out_path = r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\travel_time_summary.csv"
summary_df.to_csv(out_path, index=False)

print("✅ Saved to travel_time_summary.csv")
print("\nPreview:")
print(summary_df[['Grouping', 'Pct_within_30min', 'Pct_within_60min', 'Median_travel_time_min']].to_string(index=False))


✅ Saved to travel_time_summary.csv

Preview:
      Grouping  Pct_within_30min  Pct_within_60min  Median_travel_time_min
  Any facility             90.62             97.61                     6.8
          CHPS             89.77             97.47                     7.8
Maternity care             89.82             97.47                     7.8
    Outpatient             72.60             91.89                    17.3
     Emergency             42.13             74.58                    35.3
    Specialist              6.61             20.88                   109.4
   Psychiatric              0.89              3.51                   265.5


In [13]:
import pandas as pd
import numpy as np

# Load the nearest facility times
pop_df = pd.read_csv(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\nearest_facility_times.csv")

# Load the master dataset to get district/region info per population point
master = pd.read_csv(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\master_dataset_v3.csv")

print(f"Population points: {len(pop_df):,}")
print(f"Columns in pop_df: {list(pop_df.columns)}")

Population points: 278,001
Columns in pop_df: ['lat', 'lon', 'population', 'nearest_any_min', 'nearest_chps_min', 'nearest_maternity_min', 'nearest_outpatient_min', 'nearest_emergency_min', 'nearest_specialist_min', 'nearest_psychiatric_min']


In [14]:
import geopandas as gpd
from shapely.geometry import Point

print("Loading GADM district polygons...")
districts = gpd.read_file(r"C:\Users\hp\Downloads\Code & Scripts\gadm41_GHA_2.shp")
print(f"✅ Districts loaded: {len(districts)} polygons")
print(f"   Columns: {list(districts.columns)}")

# Convert pop_df to GeoDataFrame
print("\nConverting population points to GeoDataFrame...")
pop_gdf = gpd.GeoDataFrame(
    pop_df,
    geometry=gpd.points_from_xy(pop_df['lon'], pop_df['lat']),
    crs='EPSG:4326'
)
print(f"✅ Done: {len(pop_gdf):,} points")

# Spatial join — assign each population point to its district
print("\nSpatially joining population points to districts...")
pop_with_district = gpd.sjoin(pop_gdf, districts[['NAME_1', 'NAME_2', 'geometry']], how='left', predicate='within')
print(f"✅ Done!")
print(f"   Matched: {pop_with_district['NAME_2'].notna().sum():,}")
print(f"   Unmatched: {pop_with_district['NAME_2'].isna().sum():,}")

Loading GADM district polygons...
✅ Districts loaded: 260 polygons
   Columns: ['GID_2', 'GID_0', 'COUNTRY', 'GID_1', 'NAME_1', 'NL_NAME_1', 'NAME_2', 'VARNAME_2', 'NL_NAME_2', 'TYPE_2', 'ENGTYPE_2', 'CC_2', 'HASC_2', 'geometry']

Converting population points to GeoDataFrame...
✅ Done: 278,001 points

Spatially joining population points to districts...
✅ Done!
   Matched: 277,195
   Unmatched: 806


In [15]:
# Aggregate travel times to district level
print("Aggregating to district level...")

grouping_cols = [
    'nearest_any_min', 'nearest_chps_min', 'nearest_maternity_min',
    'nearest_outpatient_min', 'nearest_emergency_min',
    'nearest_specialist_min', 'nearest_psychiatric_min'
]

thresholds = [30, 60]

# Group by district
district_rows = []

for (region, district), group in pop_with_district.groupby(['NAME_1', 'NAME_2']):
    row = {
        'Region': region,
        'District': district,
        'Population_points': len(group),
        'Total_population': round(group['population'].sum()),
    }
    
    for col in grouping_cols:
        label = col.replace('nearest_', '').replace('_min', '')
        
        # Median travel time
        row[f'{label}_median_min'] = round(group[col].median(), 1)
        
        # % within 30 and 60 min
        for t in thresholds:
            count = (group[col] <= t).sum()
            row[f'{label}_pct_within_{t}min'] = round(count / len(group) * 100, 1)
    
    district_rows.append(row)

district_df = pd.DataFrame(district_rows)

print(f"✅ District summary: {len(district_df)} districts")

# Save
out_path = r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\district_travel_times.csv"
district_df.to_csv(out_path, index=False)
print(f"✅ Saved to district_travel_times.csv")

# Quick preview — worst districts for emergency access
print("\n=== TOP 10 WORST DISTRICTS FOR EMERGENCY ACCESS ===")
worst = district_df.nlargest(10, 'emergency_median_min')[['District', 'Region', 'emergency_median_min', 'emergency_pct_within_30min', 'emergency_pct_within_60min']]
print(worst.to_string(index=False))

Aggregating to district level...
✅ District summary: 260 districts
✅ Saved to district_travel_times.csv

=== TOP 10 WORST DISTRICTS FOR EMERGENCY ACCESS ===
                   District     Region  emergency_median_min  emergency_pct_within_30min  emergency_pct_within_60min
                 East Gonja   Savannah                 143.1                         0.0                         0.6
   Kwahu Afram Plains South    Eastern                 137.2                         1.1                         7.1
                North Gonja   Savannah                 102.0                         0.0                         2.5
                    Wa East Upper West                  93.7                         2.2                        18.3
 Sekyere Afram Plains North    Ashanti                  90.3                         0.0                        12.6
Twifo-Hemang-Lower Denkyira    Central                  87.2                         0.1                        17.9
                   Pru W

In [16]:
import requests
import time
import numpy as np
from scipy.spatial import cKDTree

print("Speed test — OSRM route API for journey breakdown...\n")

# We need to know the nearest facility for each population point
# We already computed this — it's in nearest_any_min
# But we need the actual facility coordinates to call the route API

# Build KD-Tree on ALL facilities
fac_coords = master[['Longitude', 'Latitude']].values
fac_tree = cKDTree(fac_coords)

# For a sample of 100 population points
sample = pop_df.sample(100, random_state=42)
sample_coords = sample[['lon', 'lat']].values

# Find nearest facility for each sample point
_, nearest_fac_idx = fac_tree.query(sample_coords, k=1)

start = time.time()

results = []

for i in range(100):
    pop_lon = sample_coords[i, 0]
    pop_lat = sample_coords[i, 1]
    fac_lon = fac_coords[nearest_fac_idx[i], 0]
    fac_lat = fac_coords[nearest_fac_idx[i], 1]
    
    # Route API — returns step by step directions
    url = f"http://localhost:5000/route/v1/driving/{pop_lon},{pop_lat};{fac_lon},{fac_lat}?steps=true&annotations=true"
    response = requests.get(url)
    data = response.json()
    
    if data['code'] == 'Ok':
        results.append(data)

elapsed = time.time() - start
estimated_hours = round((elapsed / 100) * 278001 / 3600, 1)

print(f"100 points took: {round(elapsed, 1)} seconds")
print(f"Estimated for all 278,001: {estimated_hours} hours")

Speed test — OSRM route API for journey breakdown...

100 points took: 1.1 seconds
Estimated for all 278,001: 0.8 hours


In [17]:
# Let's see what a route response looks like
sample_point = pop_df.iloc[0]
fac_idx = fac_tree.query([[sample_point['lon'], sample_point['lat']]], k=1)[1][0]

url = f"http://localhost:5000/route/v1/driving/{sample_point['lon']},{sample_point['lat']};{fac_coords[fac_idx,0]},{fac_coords[fac_idx,1]}?steps=true&annotations=true"
response = requests.get(url)
data = response.json()

# Show the road classes used
print("Sample route steps:\n")
for leg in data['routes'][0]['legs']:
    for step in leg['steps']:
        road_class = step.get('road_class', 'unknown')
        mode = step.get('mode', 'unknown')
        duration = round(step['duration'] / 60, 2)
        name = step.get('name', '')
        print(f"  {road_class:<20} {mode:<12} {duration} min  {name}")

Sample route steps:

  unknown              driving      10.83 min  
  unknown              driving      0.0 min  


In [18]:
# Let's see ALL fields in a route step
import json

step = data['routes'][0]['legs'][0]['steps'][0]
print("All fields in a route step:\n")
print(json.dumps(step, indent=2))


All fields in a route step:

{
  "intersections": [
    {
      "out": 0,
      "entry": [
        true
      ],
      "bearings": [
        167
      ],
      "location": [
        -0.263959,
        11.173998
      ]
    },
    {
      "out": 1,
      "in": 2,
      "entry": [
        true,
        true,
        false
      ],
      "bearings": [
        60,
        150,
        315
      ],
      "location": [
        -0.251446,
        11.147416
      ]
    }
  ],
  "driving_side": "right",
  "geometry": "olecAvpr@xB[|KkBzNcBnEmBrPwJ~PeIhK}DzT_FfJ_AjJmDbN}EhLcL|ByAdAeA~DeCxKoGjQ_IfJ{D`Ak@`Dq@z@@|A@",
  "maneuver": {
    "bearing_after": 167,
    "bearing_before": 0,
    "location": [
      -0.263959,
      11.173998
    ],
    "type": "depart"
  },
  "name": "",
  "mode": "driving",
  "weight": 649.6,
  "duration": 649.6,
  "distance": 4510.5
}


In [19]:
try:
    print(f"✅ Graph loaded: {g.vcount():,} nodes")
except:
    print("❌ Graph not in memory — need to rebuild (3.6 min)")

❌ Graph not in memory — need to rebuild (3.6 min)


In [20]:
import igraph as ig
import geopandas as gpd
import numpy as np
import time

print("Rebuilding road network graph...")
start = time.time()

roads = gpd.read_file(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\ghana-260322-free.shp\gis_osm_roads_free_1.shp")
print(f"✅ Roads loaded: {len(roads):,} segments")

speed_map = {
    'motorway': 80, 'motorway_link': 25, 'trunk': 60, 'trunk_link': 25,
    'primary': 45, 'primary_link': 25, 'secondary': 40, 'secondary_link': 20,
    'tertiary': 30, 'tertiary_link': 15, 'residential': 25, 'living_street': 15,
    'service': 15, 'busway': 25, 'unclassified': 20, 'track': 15,
    'track_grade1': 15, 'track_grade2': 10, 'track_grade3': 10,
    'track_grade4': 5, 'track_grade5': 5, 'cycleway': 10, 'unknown': 15,
    'path': 4, 'footway': 4, 'pedestrian': 4, 'steps': 3, 'bridleway': 4
}

mode_map = {
    'motorway': 'major_road', 'motorway_link': 'major_road',
    'trunk': 'major_road', 'trunk_link': 'major_road',
    'primary': 'major_road', 'primary_link': 'major_road',
    'secondary': 'connecting_road', 'secondary_link': 'connecting_road',
    'tertiary': 'connecting_road', 'tertiary_link': 'connecting_road',
    'residential': 'urban_road', 'living_street': 'urban_road',
    'service': 'urban_road', 'busway': 'urban_road',
    'unclassified': 'rural_unpaved', 'track': 'rural_unpaved',
    'track_grade1': 'rural_unpaved', 'track_grade2': 'rural_unpaved',
    'track_grade3': 'rural_unpaved', 'track_grade4': 'rural_unpaved',
    'track_grade5': 'rural_unpaved', 'cycleway': 'rural_unpaved',
    'unknown': 'rural_unpaved', 'path': 'walking', 'footway': 'walking',
    'pedestrian': 'walking', 'steps': 'walking', 'bridleway': 'walking'
}

print("Extracting nodes and edges...")
node_coords = {}
edges_list = []
node_counter = 0

for _, row in roads.iterrows():
    road_type = row.get('fclass', 'unknown')
    speed = speed_map.get(road_type, 15)
    mode = mode_map.get(road_type, 'rural_unpaved')
    if row.geometry is None:
        continue
    coords = list(row.geometry.coords)
    segment_nodes = []
    for coord in coords:
        if coord not in node_coords:
            node_coords[coord] = node_counter
            node_counter += 1
        segment_nodes.append(node_coords[coord])
    for i in range(len(segment_nodes) - 1):
        a = segment_nodes[i]
        b = segment_nodes[i + 1]
        lon1, lat1 = coords[i]
        lon2, lat2 = coords[i + 1]
        dlat = np.radians(lat2 - lat1)
        dlon = np.radians(lon2 - lon1)
        a_val = np.sin(dlat/2)**2 + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dlon/2)**2
        dist_km = 6371 * 2 * np.arcsin(np.sqrt(a_val))
        travel_time = (dist_km / speed) * 60
        edges_list.append((a, b, travel_time, mode))

print(f"✅ Nodes: {node_counter:,}, Edges: {len(edges_list):,}")

print("Building igraph...")
g = ig.Graph()
g.add_vertices(node_counter)
coords_list = [''] * node_counter
for coord, idx in node_coords.items():
    coords_list[idx] = coord
g.vs['coord'] = coords_list
g.add_edges([(e[0], e[1]) for e in edges_list])
g.es['weight'] = [e[2] for e in edges_list]
g.es['mode'] = [e[3] for e in edges_list]

print("Finding largest connected component...")
components = g.connected_components()
g = components.giant()

elapsed = round((time.time() - start) / 60, 1)
print(f"✅ Graph built: {g.vcount():,} nodes, {g.ecount():,} edges in {elapsed} min")

Rebuilding road network graph...
✅ Roads loaded: 373,884 segments
Extracting nodes and edges...
✅ Nodes: 4,374,664, Edges: 4,542,352
Building igraph...
Finding largest connected component...
✅ Graph built: 4,286,670 nodes, 4,454,860 edges in 5.0 min


In [21]:
from scipy.spatial import cKDTree
import numpy as np

print("Setting up coordinate lookup...")

all_coords = g.vs['coord']
lon_lat = np.array([[c[0], c[1]] for c in all_coords])

print("Building KD-Tree...")
tree = cKDTree(lon_lat)
print(f"✅ KD-Tree built on {len(lon_lat):,} road nodes")

# Snap facilities to road network
print("\nSnapping facilities to road network...")
facility_coords = master[['Longitude', 'Latitude']].values
_, facility_node_ids = tree.query(facility_coords, k=1)
master['node_id'] = facility_node_ids
print(f"✅ {len(master):,} facilities snapped")

# Snap population points to road network
print("\nSnapping population points to road network...")
pop_coords = pop_df[['lon', 'lat']].values
_, pop_node_ids = tree.query(pop_coords, k=1)
pop_df['node_id'] = pop_node_ids
print(f"✅ {len(pop_df):,} population points snapped")

# Build KD-Tree on ALL facilities for nearest facility lookup
print("\nBuilding facility KD-Tree for nearest facility lookup...")
fac_tree = cKDTree(facility_coords)
print("✅ All done! Ready for journey breakdown.")

Setting up coordinate lookup...
Building KD-Tree...
✅ KD-Tree built on 4,286,670 road nodes

Snapping facilities to road network...
✅ 9,978 facilities snapped

Snapping population points to road network...
✅ 278,001 population points snapped

Building facility KD-Tree for nearest facility lookup...
✅ All done! Ready for journey breakdown.


In [22]:
import time
import numpy as np

print("Speed test — journey breakdown via igraph...\n")

# Sample 100 population points
sample_idx = np.random.choice(len(pop_df), 100, replace=False)

start = time.time()

for i in sample_idx:
    pop_node = int(pop_df['node_id'].iloc[i])
    
    # Find nearest facility by straight line (k=10 candidates)
    pop_coord = pop_coords[i:i+1]
    _, fac_candidates = fac_tree.query(pop_coord, k=10)
    candidate_nodes = list(set([int(master['node_id'].iloc[j]) for j in fac_candidates[0]]))
    
    # Get shortest path from population point to nearest facility
    result = g.get_shortest_paths(
        pop_node,
        to=candidate_nodes,
        weights='weight',
        output='epath'  # return EDGE path not vertex path
    )
    
    # Find the shortest one
    best_path_edges = min(result, key=lambda path: sum(g.es[e]['weight'] for e in path) if path else float('inf'))
    
    # Sum up time by mode
    breakdown = {'major_road': 0, 'connecting_road': 0, 'urban_road': 0, 'rural_unpaved': 0, 'walking': 0}
    for edge_id in best_path_edges:
        edge = g.es[edge_id]
        breakdown[edge['mode']] += edge['weight']

elapsed = time.time() - start
estimated_hours = round((elapsed / 100) * 278001 / 3600, 1)

print(f"100 points took: {round(elapsed, 1)} seconds")
print(f"Estimated for all 278,001: {estimated_hours} hours")
print(f"\nSample breakdown from last point:")
for mode, mins in breakdown.items():
    print(f"  {mode:<20}: {round(mins, 1)} min")

Speed test — journey breakdown via igraph...

100 points took: 80.9 seconds
Estimated for all 278,001: 62.5 hours

Sample breakdown from last point:
  major_road          : 0 min
  connecting_road     : 0 min
  urban_road          : 0 min
  rural_unpaved       : 41.1 min
  walking             : 0 min


In [25]:
from scipy.spatial import cKDTree
import numpy as np

print("Finding nearest facility for each population point...")

# Build k-d tree on all facilities
fac_coords = master[['Longitude', 'Latitude']].values
fac_tree = cKDTree(fac_coords)

# For each population point, find nearest facility
pop_coords = pop_df[['lon', 'lat']].values
_, nearest_fac_idx = fac_tree.query(pop_coords, k=1)

# Get the nearest facility coordinates
pop_df['nearest_fac_lon'] = fac_coords[nearest_fac_idx, 0]
pop_df['nearest_fac_lat'] = fac_coords[nearest_fac_idx, 1]
pop_df['nearest_fac_name'] = master['Name'].iloc[nearest_fac_idx].values
pop_df['nearest_fac_type'] = master['Facility_Type'].iloc[nearest_fac_idx].values

print(f"✅ Done!")
print(f"\nSample:")
print(pop_df[['lon', 'lat', 'nearest_fac_name', 'nearest_fac_type', 'nearest_any_min']].head())

Finding nearest facility for each population point...
✅ Done!

Sample:
        lon        lat nearest_fac_name nearest_fac_type  nearest_any_min
0 -0.280417  11.170417       BADOR CHPS             CHPS        10.826667
1 -0.272083  11.170417       BADOR CHPS             CHPS        10.436667
2 -0.263750  11.170417       BADOR CHPS             CHPS         9.883333
3 -0.255417  11.170417       BADOR CHPS             CHPS         8.716667
4 -0.247083  11.170417       BADOR CHPS             CHPS         7.798333


In [28]:
import requests
import json

# Take one population point and its actual nearest facility
pop_point = pop_df.iloc[1000]

In [29]:
import requests
import json

pop_point = pop_df.iloc[1000]

url = (f"http://localhost:5000/route/v1/driving/"
       f"{pop_point['lon']},{pop_point['lat']};"
       f"{pop_point['nearest_fac_lon']},{pop_point['nearest_fac_lat']}"
       f"?steps=true&geometries=geojson&overview=full&annotations=nodes")

print(f"URL: {url}\n")

response = requests.get(url)
print(f"Status code: {response.status_code}")
print(f"Response: {response.text[:500]}")

URL: http://localhost:5000/route/v1/driving/-1.413749994711092,10.995416827532884;-1.3296589,10.9789448?steps=true&geometries=geojson&overview=full&annotations=nodes

Status code: 200
Response: {"code":"Ok","routes":[{"legs":[{"annotation":{"nodes":[9302083687,9302083688,9302083689,9302083692,9302083693,9302083694,9302083644,9302083695,9302083696,9302083697,9302083698,9302083699,9302083624,9302083623,9302083701,9302083702,9302083703,9302083679,9302083681,9302083705,9302083682,9302083683,9302083707,9302083684,9302083685,9302083708,9302083710]},"duration":160.2,"summary":"","weight":160.2,"distance":1112.7,"steps":[{"intersections":[{"out":0,"entry":[true],"bearings":[103],"location":[-1


In [30]:
import geopandas as gpd

roads = gpd.read_file(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\ghana-260322-free.shp\gis_osm_roads_free_1.shp")

print("Road shapefile columns:")
print(roads.columns.tolist())
print("\nSample rows:")
print(roads[['osm_id', 'fclass', 'name']].head(10))

Road shapefile columns:
['osm_id', 'code', 'fclass', 'name', 'ref', 'oneway', 'maxspeed', 'layer', 'bridge', 'tunnel', 'geometry']

Sample rows:
    osm_id         fclass                   name
0  4790591   unclassified           Airport Road
1  4790592    residential     Nortei Ababio Road
2  4790594       tertiary           Airport Road
3  4790596   unclassified           Airport Road
4  4790597    residential             Volta Road
5  4790599       tertiary  South Liberation Link
6  4790600       tertiary           Airport Road
7  4790601  tertiary_link                   None
8  4790602   unclassified           Airport Road
9  4790603   unclassified                   None


In [31]:
import requests
import time
import pickle
import numpy as np

print("Journey breakdown — OSRM route API...\n")

# Test with just 3 points first to confirm it works
for i in [0, 1000, 50000]:
    row = pop_df.iloc[i]
    url = (f"http://localhost:5000/route/v1/driving/"
           f"{row['lon']},{row['lat']};"
           f"{row['nearest_fac_lon']},{row['nearest_fac_lat']}"
           f"?steps=true&annotations=nodes")
    
    response = requests.get(url)
    data = response.json()
    total_min = round(data['routes'][0]['duration'] / 60, 1)
    n_nodes = len(data['routes'][0]['legs'][0]['annotation']['nodes'])
    
    print(f"Row {i}: {row['nearest_fac_name']} — {total_min} min — {n_nodes} road nodes")

Journey breakdown — OSRM route API...

Row 0: BADOR CHPS — 10.8 min — 23 road nodes
Row 1000: BALIU CHPS — 2.7 min — 27 road nodes
Row 50000: BONAA CHPS — 12.5 min — 52 road nodes


In [32]:
# Build a lookup: OSM node ID → road fclass
# Each road segment has a geometry with node coordinates
# We need to extract start/end node IDs from the OSM pbf

# Actually let's check what node IDs OSRM returns
# and see if they match OSM node IDs in our data

pop_point = pop_df.iloc[1000]
url = (f"http://localhost:5000/route/v1/driving/"
       f"{pop_point['lon']},{pop_point['lat']};"
       f"{pop_point['nearest_fac_lon']},{pop_point['nearest_fac_lat']}"
       f"?steps=true&annotations=nodes&geometries=geojson")

response = requests.get(url)
data = response.json()

nodes = data['routes'][0]['legs'][0]['annotation']['nodes']
geometry = data['routes'][0]['geometry']['coordinates']

print(f"Number of nodes: {len(nodes)}")
print(f"Number of geometry points: {len(geometry)}")
print(f"\nFirst 5 node IDs: {nodes[:5]}")
print(f"First 5 coordinates: {geometry[:5]}")

Number of nodes: 27
Number of geometry points: 17

First 5 node IDs: [9302083687, 9302083688, 9302083689, 9302083692, 9302083693]
First 5 coordinates: [[-1.340337, 10.977695], [-1.340005, 10.977621], [-1.339619, 10.977447], [-1.338664, 10.976726], [-1.337886, 10.976483]]


In [33]:
import geopandas as gpd
from shapely.geometry import Point

print("Building road segment spatial index...")

# We already have roads loaded — if not, reload
try:
    print(f"Roads already loaded: {len(roads):,} segments")
except:
    roads = gpd.read_file(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\ghana-260322-free.shp\gis_osm_roads_free_1.shp")
    print(f"✅ Roads loaded: {len(roads):,} segments")

# Map fclass to our 5 categories
mode_map = {
    'motorway': 'major_road', 'motorway_link': 'major_road',
    'trunk': 'major_road', 'trunk_link': 'major_road',
    'primary': 'major_road', 'primary_link': 'major_road',
    'secondary': 'connecting_road', 'secondary_link': 'connecting_road',
    'tertiary': 'connecting_road', 'tertiary_link': 'connecting_road',
    'residential': 'urban_road', 'living_street': 'urban_road',
    'service': 'urban_road', 'busway': 'urban_road',
    'unclassified': 'rural_unpaved', 'track': 'rural_unpaved',
    'track_grade1': 'rural_unpaved', 'track_grade2': 'rural_unpaved',
    'track_grade3': 'rural_unpaved', 'track_grade4': 'rural_unpaved',
    'track_grade5': 'rural_unpaved', 'cycleway': 'rural_unpaved',
    'unknown': 'rural_unpaved', 'path': 'walking', 'footway': 'walking',
    'pedestrian': 'walking', 'steps': 'walking', 'bridleway': 'walking'
}

roads['mode'] = roads['fclass'].map(mode_map).fillna('rural_unpaved')

# Build spatial index
print("Building spatial index on road segments...")
roads_sindex = roads.sindex
print(f"✅ Spatial index built on {len(roads):,} road segments")

# Test — find road type for one coordinate
test_point = Point(-1.340337, 10.977695)
possible_matches_idx = list(roads_sindex.nearest(test_point, 1))
nearest_road = roads.iloc[possible_matches_idx[0]]
print(f"\nTest point road type: {nearest_road['fclass']} → {nearest_road['mode']}")

Building road segment spatial index...
Roads already loaded: 373,884 segments
Building spatial index on road segments...
✅ Spatial index built on 373,884 road segments

Test point road type: 0    unclassified
Name: fclass, dtype: object → 0    rural_unpaved
Name: mode, dtype: object


In [34]:
import time
import numpy as np
from shapely.geometry import Point

print("Speed test — full journey breakdown pipeline...\n")

sample = pop_df.sample(100, random_state=42)

start = time.time()

for _, row in sample.iterrows():
    
    # Step 1 — get route from OSRM
    url = (f"http://localhost:5000/route/v1/driving/"
           f"{row['lon']},{row['lat']};"
           f"{row['nearest_fac_lon']},{row['nearest_fac_lat']}"
           f"?geometries=geojson&overview=full")
    
    response = requests.get(url)
    data = response.json()
    
    if data['code'] != 'Ok':
        continue
    
    route = data['routes'][0]
    total_duration = route['duration']  # seconds
    coords = route['geometry']['coordinates']  # list of [lon, lat]
    
    # Step 2 — for each segment, find road type
    segment_modes = []
    for i in range(len(coords) - 1):
        mid_lon = (coords[i][0] + coords[i+1][0]) / 2
        mid_lat = (coords[i][1] + coords[i+1][1]) / 2
        pt = Point(mid_lon, mid_lat)
        nearest_idx = list(roads_sindex.nearest(pt, 1))[0]
        mode = roads.iloc[nearest_idx]['mode']
        segment_modes.append(mode)
    
    # Step 3 — distribute total time equally across segments
    if len(segment_modes) > 0:
        time_per_segment = total_duration / len(segment_modes)
        breakdown = {'major_road': 0, 'connecting_road': 0, 
                    'urban_road': 0, 'rural_unpaved': 0, 'walking': 0}
        for mode in segment_modes:
            breakdown[mode] += time_per_segment / 60  # convert to minutes

elapsed = time.time() - start
estimated_hours = round((elapsed / 100) * 278001 / 3600, 1)
estimated_minutes = round((elapsed / 100) * 278001 / 60, 1)

print(f"100 points took: {round(elapsed, 1)} seconds")
print(f"Estimated for all 278,001: {estimated_hours} hours ({estimated_minutes} minutes)")
print(f"\nSample breakdown:")
for mode, mins in breakdown.items():
    print(f"  {mode:<20}: {round(mins, 1)} min")

Speed test — full journey breakdown pipeline...



TypeError: unhashable type: 'Series'

In [35]:
import time
import numpy as np
from shapely.geometry import Point

print("Speed test — full journey breakdown pipeline...\n")

sample = pop_df.sample(100, random_state=42)

start = time.time()

for _, row in sample.iterrows():
    
    # Step 1 — get route from OSRM
    url = (f"http://localhost:5000/route/v1/driving/"
           f"{row['lon']},{row['lat']};"
           f"{row['nearest_fac_lon']},{row['nearest_fac_lat']}"
           f"?geometries=geojson&overview=full")
    
    response = requests.get(url)
    data = response.json()
    
    if data['code'] != 'Ok':
        continue
    
    route = data['routes'][0]
    total_duration = route['duration']  # seconds
    coords = route['geometry']['coordinates']
    
    # Step 2 — for each segment, find road type
    segment_modes = []
    for i in range(len(coords) - 1):
        mid_lon = (coords[i][0] + coords[i+1][0]) / 2
        mid_lat = (coords[i][1] + coords[i+1][1]) / 2
        pt = Point(mid_lon, mid_lat)
        nearest_idx = list(roads_sindex.nearest(pt, 1))[0]
        mode = roads.iloc[nearest_idx]['mode'].iloc[0]  # fix here
        segment_modes.append(mode)
    
    # Step 3 — distribute total time equally across segments
    if len(segment_modes) > 0:
        time_per_segment = total_duration / len(segment_modes)
        breakdown = {'major_road': 0, 'connecting_road': 0, 
                    'urban_road': 0, 'rural_unpaved': 0, 'walking': 0}
        for mode in segment_modes:
            breakdown[mode] += time_per_segment / 60

elapsed = time.time() - start
estimated_hours = round((elapsed / 100) * 278001 / 3600, 1)
estimated_minutes = round((elapsed / 100) * 278001 / 60, 1)

print(f"100 points took: {round(elapsed, 1)} seconds")
print(f"Estimated for all 278,001: {estimated_hours} hours ({estimated_minutes} minutes)")
print(f"\nSample breakdown:")
for mode, mins in breakdown.items():
    print(f"  {mode:<20}: {round(mins, 1)} min")

Speed test — full journey breakdown pipeline...

100 points took: 22.4 seconds
Estimated for all 278,001: 17.3 hours (1036.3 minutes)

Sample breakdown:
  major_road          : 0 min
  connecting_road     : 0 min
  urban_road          : 0 min
  rural_unpaved       : 1.4 min
  walking             : 0 min


In [36]:
import time
from shapely.geometry import Point
import numpy as np

print("Speed test v2 — faster spatial lookup...\n")

# Pre-build a coordinate array for all road midpoints
# So we can use KD-Tree instead of spatial index (much faster)
print("Pre-computing road segment midpoints...")
from scipy.spatial import cKDTree

road_midpoints = []
road_modes = []

for _, road in roads.iterrows():
    coords = list(road.geometry.coords)
    for i in range(len(coords) - 1):
        mid_lon = (coords[i][0] + coords[i+1][0]) / 2
        mid_lat = (coords[i][1] + coords[i+1][1]) / 2
        road_midpoints.append([mid_lon, mid_lat])
        road_modes.append(road['mode'])

road_midpoints = np.array(road_midpoints)
road_modes = np.array(road_modes)
road_kd = cKDTree(road_midpoints)

print(f"✅ Road KD-Tree built on {len(road_midpoints):,} segments")

# Speed test
sample = pop_df.sample(100, random_state=42)
start = time.time()

for _, row in sample.iterrows():
    url = (f"http://localhost:5000/route/v1/driving/"
           f"{row['lon']},{row['lat']};"
           f"{row['nearest_fac_lon']},{row['nearest_fac_lat']}"
           f"?geometries=geojson&overview=full&annotations=duration")
    
    response = requests.get(url)
    data = response.json()
    if data['code'] != 'Ok':
        continue
    
    route = data['routes'][0]
    coords = route['geometry']['coordinates']
    leg = route['legs'][0]
    
    # Get per-segment durations from annotations
    seg_durations = leg['annotation']['duration']  # seconds per segment
    
    # Get midpoints of each geometry segment
    if len(coords) < 2:
        continue
        
    mids = np.array([
        [(coords[i][0] + coords[i+1][0]) / 2,
         (coords[i][1] + coords[i+1][1]) / 2]
        for i in range(len(coords) - 1)
    ])
    
    # Find nearest road type for each midpoint in one batch call
    _, idxs = road_kd.query(mids)
    modes = road_modes[idxs]
    
    # Distribute annotation durations across segments
    # annotation has more segments than geometry — distribute evenly
    total_duration = sum(seg_durations)
    time_per_geom_seg = total_duration / len(mids) if len(mids) > 0 else 0
    
    breakdown = {'major_road': 0, 'connecting_road': 0,
                'urban_road': 0, 'rural_unpaved': 0, 'walking': 0}
    for i, mode in enumerate(modes):
        breakdown[mode] += time_per_geom_seg / 60

elapsed = time.time() - start
estimated_hours = round((elapsed / 100) * 278001 / 3600, 1)

print(f"\n100 points took: {round(elapsed, 1)} seconds")
print(f"Estimated for all 278,001: {estimated_hours} hours")
print(f"\nSample breakdown:")
total = sum(breakdown.values())
for mode, mins in breakdown.items():
    print(f"  {mode:<20}: {round(mins, 1)} min ({round(mins/total*100 if total > 0 else 0, 1)}%)")

Speed test v2 — faster spatial lookup...

Pre-computing road segment midpoints...
✅ Road KD-Tree built on 4,542,352 segments

100 points took: 0.9 seconds
Estimated for all 278,001: 0.7 hours

Sample breakdown:
  major_road          : 0 min (0.0%)
  connecting_road     : 0 min (0.0%)
  urban_road          : 0 min (0.0%)
  rural_unpaved       : 1.4 min (100.0%)
  walking             : 0 min (0.0%)


In [37]:
# Quick fix test on one point
row = pop_df.iloc[0]

url = (f"http://localhost:5000/route/v1/driving/"
       f"{row['lon']},{row['lat']};"
       f"{row['nearest_fac_lon']},{row['nearest_fac_lat']}"
       f"?geometries=geojson&overview=full&annotations=duration")

response = requests.get(url)
data = response.json()
route = data['routes'][0]
leg = route['legs'][0]

total_duration_min = round(route['duration'] / 60, 1)
coords = route['geometry']['coordinates']
seg_durations = leg['annotation']['duration']

print(f"Total duration: {total_duration_min} min")
print(f"Geometry points: {len(coords)}")
print(f"Geometry segments: {len(coords)-1}")
print(f"Annotation segments: {len(seg_durations)}")
print(f"Sum of annotation durations: {round(sum(seg_durations)/60, 1)} min")

Total duration: 10.8 min
Geometry points: 23
Geometry segments: 22
Annotation segments: 22
Sum of annotation durations: 10.8 min


In [38]:
import time
import numpy as np

print("Speed test v3 — correct time distribution...\n")

sample = pop_df.sample(100, random_state=42)
start = time.time()

for _, row in sample.iterrows():
    url = (f"http://localhost:5000/route/v1/driving/"
           f"{row['lon']},{row['lat']};"
           f"{row['nearest_fac_lon']},{row['nearest_fac_lat']}"
           f"?geometries=geojson&overview=full&annotations=duration")
    
    response = requests.get(url)
    data = response.json()
    if data['code'] != 'Ok':
        continue
    
    route = data['routes'][0]
    coords = route['geometry']['coordinates']
    seg_durations = route['legs'][0]['annotation']['duration']
    
    if len(coords) < 2 or len(seg_durations) == 0:
        continue
    
    # Midpoint of each geometry segment
    mids = np.array([
        [(coords[i][0] + coords[i+1][0]) / 2,
         (coords[i][1] + coords[i+1][1]) / 2]
        for i in range(len(coords) - 1)
    ])
    
    # Find nearest road type for each midpoint
    _, idxs = road_kd.query(mids)
    modes = road_modes[idxs]
    
    # Use actual duration per segment
    breakdown = {'major_road': 0, 'connecting_road': 0,
                'urban_road': 0, 'rural_unpaved': 0, 'walking': 0}
    
    for i, mode in enumerate(modes):
        if i < len(seg_durations):
            breakdown[mode] += seg_durations[i] / 60  # seconds to minutes

elapsed = time.time() - start
estimated_hours = round((elapsed / 100) * 278001 / 3600, 1)

print(f"100 points took: {round(elapsed, 1)} seconds")
print(f"Estimated for all 278,001: {estimated_hours} hours")
print(f"\nSample breakdown (last point):")
total = sum(breakdown.values())
for mode, mins in breakdown.items():
    pct = round(mins/total*100 if total > 0 else 0, 1)
    print(f"  {mode:<20}: {round(mins, 1)} min ({pct}%)")
print(f"  {'TOTAL':<20}: {round(total, 1)} min")


Speed test v3 — correct time distribution...

100 points took: 1.1 seconds
Estimated for all 278,001: 0.9 hours

Sample breakdown (last point):
  major_road          : 0 min (0.0%)
  connecting_road     : 0 min (0.0%)
  urban_road          : 0 min (0.0%)
  rural_unpaved       : 1.4 min (100.0%)
  walking             : 0 min (0.0%)
  TOTAL               : 1.4 min


In [39]:
# Check with overview=false to get full unsimplified geometry
row = pop_df.iloc[0]

url_full = (f"http://localhost:5000/route/v1/driving/"
            f"{row['lon']},{row['lat']};"
            f"{row['nearest_fac_lon']},{row['nearest_fac_lat']}"
            f"?geometries=geojson&overview=false&annotations=duration&steps=true")

response = requests.get(url_full)
data = response.json()
route = data['routes'][0]
leg = route['legs'][0]

# With overview=false, geometry is per step
total_coords = 0
total_ann_segs = len(leg['annotation']['duration'])
for step in leg['steps']:
    if 'geometry' in step:
        total_coords += len(step['geometry']['coordinates'])

print(f"Total duration: {round(route['duration']/60, 1)} min")
print(f"Annotation segments: {total_ann_segs}")
print(f"Sum annotation: {round(sum(leg['annotation']['duration'])/60, 1)} min")
print(f"Steps: {len(leg['steps'])}")
print(f"Total step geometry coords: {total_coords}")

Total duration: 10.8 min
Annotation segments: 22
Sum annotation: 10.8 min
Steps: 2
Total step geometry coords: 25


In [40]:
# Debug one point in detail
row = pop_df.iloc[0]

url = (f"http://localhost:5000/route/v1/driving/"
       f"{row['lon']},{row['lat']};"
       f"{row['nearest_fac_lon']},{row['nearest_fac_lat']}"
       f"?geometries=geojson&overview=full&annotations=duration")

response = requests.get(url)
data = response.json()
route = data['routes'][0]
coords = route['geometry']['coordinates']
seg_durations = route['legs'][0]['annotation']['duration']

print(f"Total duration: {round(route['duration']/60, 1)} min")
print(f"Coords: {len(coords)}, Segments: {len(coords)-1}, Annotations: {len(seg_durations)}")
print(f"Sum annotations: {round(sum(seg_durations)/60, 1)} min")

# Check each segment
mids = np.array([
    [(coords[i][0] + coords[i+1][0]) / 2,
     (coords[i][1] + coords[i+1][1]) / 2]
    for i in range(len(coords) - 1)
])

_, idxs = road_kd.query(mids)
modes = road_modes[idxs]

print(f"\nSegment breakdown:")
total = 0
for i in range(len(mids)):
    dur = seg_durations[i] if i < len(seg_durations) else 0
    total += dur
    print(f"  seg {i}: {round(dur/60,2)} min → {modes[i]}")

print(f"\nTotal accounted: {round(total/60, 1)} min")

Total duration: 10.8 min
Coords: 23, Segments: 22, Annotations: 22
Sum annotations: 10.8 min

Segment breakdown:
  seg 0: 0.17 min → rural_unpaved
  seg 1: 0.57 min → rural_unpaved
  seg 2: 0.69 min → rural_unpaved
  seg 3: 0.31 min → rural_unpaved
  seg 4: 0.9 min → rural_unpaved
  seg 5: 0.88 min → rural_unpaved
  seg 6: 0.58 min → rural_unpaved
  seg 7: 0.97 min → rural_unpaved
  seg 8: 0.49 min → rural_unpaved
  seg 9: 0.54 min → rural_unpaved
  seg 10: 0.71 min → rural_unpaved
  seg 11: 0.79 min → rural_unpaved
  seg 12: 0.21 min → rural_unpaved
  seg 13: 0.13 min → rural_unpaved
  seg 14: 0.31 min → rural_unpaved
  seg 15: 0.65 min → rural_unpaved
  seg 16: 0.89 min → rural_unpaved
  seg 17: 0.54 min → rural_unpaved
  seg 18: 0.1 min → rural_unpaved
  seg 19: 0.22 min → rural_unpaved
  seg 20: 0.08 min → rural_unpaved
  seg 21: 0.12 min → rural_unpaved

Total accounted: 10.8 min


In [41]:
import time
import numpy as np
import pandas as pd
import pickle

print("=" * 60)
print("JOURNEY BREAKDOWN — ALL 278,001 POPULATION POINTS")
print("=" * 60)

n_pop = len(pop_df)
BATCH_SAVE = 10000  # save every 10,000 points

# Results storage
results = {
    'major_road': np.zeros(n_pop),
    'connecting_road': np.zeros(n_pop),
    'urban_road': np.zeros(n_pop),
    'rural_unpaved': np.zeros(n_pop),
    'walking': np.zeros(n_pop),
    'total': np.zeros(n_pop),
    'failed': np.zeros(n_pop, dtype=bool)
}

start = time.time()

for i, (_, row) in enumerate(pop_df.iterrows()):
    
    # Progress update every 500 points
    if i % 500 == 0:
        elapsed = time.time() - start
        pct = i / n_pop * 100
        if i > 0:
            eta = (elapsed / i) * (n_pop - i) / 60
            print(f"  {i:,}/{n_pop:,} ({pct:.1f}%) — ETA: {eta:.1f} min")
        else:
            print(f"  {i:,}/{n_pop:,} — starting...")
    
    # Get route from OSRM
    url = (f"http://localhost:5000/route/v1/driving/"
           f"{row['lon']},{row['lat']};"
           f"{row['nearest_fac_lon']},{row['nearest_fac_lat']}"
           f"?geometries=geojson&overview=full&annotations=duration")
    
    try:
        response = requests.get(url, timeout=10)
        data = response.json()
        
        if data['code'] != 'Ok':
            results['failed'][i] = True
            continue
        
        route = data['routes'][0]
        coords = route['geometry']['coordinates']
        seg_durations = route['legs'][0]['annotation']['duration']
        
        if len(coords) < 2 or len(seg_durations) == 0:
            results['failed'][i] = True
            continue
        
        # Midpoints of each segment
        mids = np.array([
            [(coords[j][0] + coords[j+1][0]) / 2,
             (coords[j][1] + coords[j+1][1]) / 2]
            for j in range(len(coords) - 1)
        ])
        
        # Find nearest road type for each midpoint
        _, idxs = road_kd.query(mids)
        modes = road_modes[idxs]
        
        # Accumulate time per mode
        for j, mode in enumerate(modes):
            if j < len(seg_durations):
                results[mode][i] += seg_durations[j] / 60
        
        results['total'][i] = sum(seg_durations) / 60
        
    except Exception:
        results['failed'][i] = True
    
    # Save checkpoint every 10,000 points
    if i > 0 and i % BATCH_SAVE == 0:
        with open(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\journey_breakdown_checkpoint.pkl", 'wb') as f:
            pickle.dump({'results': results, 'next_idx': i}, f)
        print(f"  💾 Checkpoint saved at {i:,}")

# Final save
print("\nSaving final results...")

for mode in ['major_road', 'connecting_road', 'urban_road', 'rural_unpaved', 'walking', 'total']:
    pop_df[f'journey_{mode}_min'] = results[mode]
pop_df['journey_failed'] = results['failed']

out_path = r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\journey_breakdown.csv"
pop_df.to_csv(out_path, index=False)

with open(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\journey_breakdown.pkl", 'wb') as f:
    pickle.dump(pop_df, f)

print(f"✅ Saved to journey_breakdown.csv and .pkl")
print(f"\n🎉 ALL DONE!")

# Summary
print("\n=== JOURNEY BREAKDOWN SUMMARY ===")
for mode in ['major_road', 'connecting_road', 'urban_road', 'rural_unpaved', 'walking']:
    col = f'journey_{mode}_min'
    mean = round(pop_df[col].mean(), 1)
    pct = round(pop_df[col].sum() / pop_df['journey_total_min'].sum() * 100, 1)
    print(f"  {mode:<20}: avg {mean} min ({pct}% of total journey time)")
print(f"  Failed: {results['failed'].sum():,} points")

JOURNEY BREAKDOWN — ALL 278,001 POPULATION POINTS
  0/278,001 — starting...
  500/278,001 (0.2%) — ETA: 91.0 min
  1,000/278,001 (0.4%) — ETA: 89.5 min
  1,500/278,001 (0.5%) — ETA: 88.5 min
  2,000/278,001 (0.7%) — ETA: 85.0 min
  2,500/278,001 (0.9%) — ETA: 84.2 min
  3,000/278,001 (1.1%) — ETA: 83.2 min
  3,500/278,001 (1.3%) — ETA: 80.2 min
  4,000/278,001 (1.4%) — ETA: 81.2 min
  4,500/278,001 (1.6%) — ETA: 82.2 min
  5,000/278,001 (1.8%) — ETA: 83.4 min
  5,500/278,001 (2.0%) — ETA: 83.2 min
  6,000/278,001 (2.2%) — ETA: 82.6 min
  6,500/278,001 (2.3%) — ETA: 82.1 min
  7,000/278,001 (2.5%) — ETA: 81.4 min
  7,500/278,001 (2.7%) — ETA: 81.2 min
  8,000/278,001 (2.9%) — ETA: 81.6 min
  8,500/278,001 (3.1%) — ETA: 80.7 min
  9,000/278,001 (3.2%) — ETA: 80.6 min
  9,500/278,001 (3.4%) — ETA: 80.6 min
  10,000/278,001 (3.6%) — ETA: 80.6 min
  💾 Checkpoint saved at 10,000
  10,500/278,001 (3.8%) — ETA: 79.6 min
  11,000/278,001 (4.0%) — ETA: 79.0 min
  11,500/278,001 (4.1%) — ETA: 79.

In [42]:
import pandas as pd

journey_df = pd.read_csv(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\journey_breakdown.csv")

print(f"Shape: {journey_df.shape}")
print(f"\nColumns: {list(journey_df.columns)}")
print(f"\nFirst 5 rows:")
journey_df.head()

Shape: (278001, 22)

Columns: ['lat', 'lon', 'population', 'nearest_any_min', 'nearest_chps_min', 'nearest_maternity_min', 'nearest_outpatient_min', 'nearest_emergency_min', 'nearest_specialist_min', 'nearest_psychiatric_min', 'node_id', 'nearest_fac_lon', 'nearest_fac_lat', 'nearest_fac_name', 'nearest_fac_type', 'journey_major_road_min', 'journey_connecting_road_min', 'journey_urban_road_min', 'journey_rural_unpaved_min', 'journey_walking_min', 'journey_total_min', 'journey_failed']

First 5 rows:


,lat,lon,population,nearest_any_min,nearest_chps_min,nearest_maternity_min,nearest_outpatient_min,nearest_emergency_min,nearest_specialist_min,nearest_psychiatric_min,...,nearest_fac_lat,nearest_fac_name,nearest_fac_type,journey_major_road_min,journey_connecting_road_min,journey_urban_road_min,journey_rural_unpaved_min,journey_walking_min,journey_total_min,journey_failed
0,11.170417,-0.280417,53.884277,10.826667,10.826667,10.826667,19.851667,31.205000,96.846667,50.118333,...,11.137765,BADOR CHPS,CHPS,0.0,0.0,0.0,10.826667,0.0,10.826667,False
1,11.170417,-0.272083,59.233740,10.436667,10.436667,10.436667,19.461667,30.815000,96.456667,49.728333,...,11.137765,BADOR CHPS,CHPS,0.0,0.0,0.0,10.436667,0.0,10.436667,False
2,11.170417,-0.263750,57.277393,9.883333,9.883333,9.883333,18.908333,30.261667,95.903333,49.175000,...,11.137765,BADOR CHPS,CHPS,0.0,0.0,0.0,9.883333,0.0,9.883333,False
3,11.170417,-0.255417,54.462917,8.716667,8.716667,8.716667,17.741667,29.095000,94.736667,48.008333,...,11.137765,BADOR CHPS,CHPS,0.0,0.0,0.0,8.716667,0.0,8.716667,False
4,11.170417,-0.247083,55.727630,7.798333,7.798333,7.798333,16.823333,28.176667,93.818333,47.090000,...,11.137765,BADOR CHPS,CHPS,0.0,0.0,0.0,7.798333,0.0,7.798333,False


In [43]:
import geopandas as gpd
import pandas as pd

print("Loading GADM district polygons...")
districts = gpd.read_file(r"C:\Users\hp\Downloads\Code & Scripts\gadm41_GHA_2.shp")
print(f"✅ Districts loaded: {len(districts)} polygons")

# Convert journey_df to GeoDataFrame
print("\nConverting to GeoDataFrame...")
journey_gdf = gpd.GeoDataFrame(
    journey_df,
    geometry=gpd.points_from_xy(journey_df['lon'], journey_df['lat']),
    crs='EPSG:4326'
)

# Spatial join — assign each point to its district
print("Spatially joining to districts...")
journey_with_district = gpd.sjoin(
    journey_gdf,
    districts[['NAME_1', 'NAME_2', 'geometry']],
    how='left',
    predicate='within'
)

# Rename columns
journey_with_district = journey_with_district.rename(columns={
    'NAME_1': 'Region',
    'NAME_2': 'District'
})

# Drop geometry and index columns
journey_with_district = journey_with_district.drop(columns=['geometry', 'index_right'])

print(f"✅ Done!")
print(f"   Matched: {journey_with_district['District'].notna().sum():,}")
print(f"   Unmatched: {journey_with_district['District'].isna().sum():,}")
print(f"\nSample:")
print(journey_with_district[['lon', 'lat', 'Region', 'District', 'journey_total_min', 'journey_rural_unpaved_min']].head())

Loading GADM district polygons...
✅ Districts loaded: 260 polygons

Converting to GeoDataFrame...
Spatially joining to districts...
✅ Done!
   Matched: 277,195
   Unmatched: 806

Sample:
        lon        lat      Region District  journey_total_min  \
0 -0.280417  11.170417  Upper East    Bawku          10.826667   
1 -0.272083  11.170417  Upper East    Bawku          10.436667   
2 -0.263750  11.170417  Upper East    Bawku           9.883333   
3 -0.255417  11.170417         NaN      NaN           8.716667   
4 -0.247083  11.170417         NaN      NaN           7.798333   

   journey_rural_unpaved_min  
0                  10.826667  
1                  10.436667  
2                   9.883333  
3                   8.716667  
4                   7.798333  


In [45]:
import pandas as pd

print("Loading all datasets...")

journey_df = pd.read_csv(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\journey_breakdown.csv")
travel_df = pd.read_csv(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\nearest_facility_times.csv")
e2sfca_df = pd.read_csv(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\e2sfca_v2_scores.csv")

print(f"✅ Journey breakdown: {journey_df.shape}")
print(f"✅ Travel times: {travel_df.shape}")
print(f"✅ E2SFCA scores: {e2sfca_df.shape}")

print(f"\nJourney columns: {list(journey_df.columns)}")
print(f"\nTravel columns: {list(travel_df.columns)}")
print(f"\nE2SFCA columns: {list(e2sfca_df.columns)}")

Loading all datasets...
✅ Journey breakdown: (278001, 22)
✅ Travel times: (278001, 10)
✅ E2SFCA scores: (278001, 7)

Journey columns: ['lat', 'lon', 'population', 'nearest_any_min', 'nearest_chps_min', 'nearest_maternity_min', 'nearest_outpatient_min', 'nearest_emergency_min', 'nearest_specialist_min', 'nearest_psychiatric_min', 'node_id', 'nearest_fac_lon', 'nearest_fac_lat', 'nearest_fac_name', 'nearest_fac_type', 'journey_major_road_min', 'journey_connecting_road_min', 'journey_urban_road_min', 'journey_rural_unpaved_min', 'journey_walking_min', 'journey_total_min', 'journey_failed']

Travel columns: ['lat', 'lon', 'population', 'nearest_any_min', 'nearest_chps_min', 'nearest_maternity_min', 'nearest_outpatient_min', 'nearest_emergency_min', 'nearest_specialist_min', 'nearest_psychiatric_min']

E2SFCA columns: ['lat', 'lon', 'population', 'population_normalized', 'node_idx', 'node_id', 'accessibility_score']


In [46]:
# Journey breakdown already has all the travel time columns! So we just need to add the E2SFCA score and the district/region. 

import pandas as pd

print("Building master population dataset...")

# Start with journey_with_district (has district/region + all journey data)
master_pop = journey_with_district.copy()

# Add E2SFCA accessibility score
master_pop = master_pop.merge(
    e2sfca_df[['lat', 'lon', 'accessibility_score']],
    on=['lat', 'lon'],
    how='left'
)

print(f"✅ Shape: {master_pop.shape}")
print(f"\nColumns: {list(master_pop.columns)}")
print(f"\nSample:")
print(master_pop[['lat', 'lon', 'Region', 'District', 'population',
                   'nearest_any_min', 'nearest_emergency_min',
                   'journey_rural_unpaved_min', 'journey_total_min',
                   'accessibility_score']].head())

# Save
out_path = r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\master_population_data.csv"
master_pop.to_csv(out_path, index=False)
print(f"\n✅ Saved to master_population_data.csv")

Building master population dataset...
✅ Shape: (278001, 25)

Columns: ['lat', 'lon', 'population', 'nearest_any_min', 'nearest_chps_min', 'nearest_maternity_min', 'nearest_outpatient_min', 'nearest_emergency_min', 'nearest_specialist_min', 'nearest_psychiatric_min', 'node_id', 'nearest_fac_lon', 'nearest_fac_lat', 'nearest_fac_name', 'nearest_fac_type', 'journey_major_road_min', 'journey_connecting_road_min', 'journey_urban_road_min', 'journey_rural_unpaved_min', 'journey_walking_min', 'journey_total_min', 'journey_failed', 'Region', 'District', 'accessibility_score']

Sample:
         lat       lon      Region District  population  nearest_any_min  \
0  11.170417 -0.280417  Upper East    Bawku   53.884277        10.826667   
1  11.170417 -0.272083  Upper East    Bawku   59.233740        10.436667   
2  11.170417 -0.263750  Upper East    Bawku   57.277393         9.883333   
3  11.170417 -0.255417         NaN      NaN   54.462917         8.716667   
4  11.170417 -0.247083         NaN  

In [47]:
print("Creating district level summary...")

district_summary = master_pop.groupby(['Region', 'District']).agg(
    total_population=('population', 'sum'),
    population_points=('population', 'count'),
    avg_travel_any=('nearest_any_min', 'mean'),
    avg_travel_emergency=('nearest_emergency_min', 'mean'),
    avg_travel_specialist=('nearest_specialist_min', 'mean'),
    avg_travel_psychiatric=('nearest_psychiatric_min', 'mean'),
    pct_within_30_any=('nearest_any_min', lambda x: (x<=30).mean()*100),
    pct_within_30_emergency=('nearest_emergency_min', lambda x: (x<=30).mean()*100),
    pct_within_60_emergency=('nearest_emergency_min', lambda x: (x<=60).mean()*100),
    avg_rural_unpaved_pct=('journey_rural_unpaved_min', lambda x: (x / master_pop.loc[x.index, 'journey_total_min'].replace(0, np.nan) * 100).mean()),
    avg_accessibility_score=('accessibility_score', 'mean'),
).reset_index().round(2)

print(f"✅ District summary: {district_summary.shape}")

# Save
out_path = r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\district_summary.csv"
district_summary.to_csv(out_path, index=False)
print(f"✅ Saved to district_summary.csv")

print("\nTop 10 worst districts for emergency access:")
worst = district_summary.nlargest(10, 'avg_travel_emergency')[
    ['District', 'Region', 'avg_travel_emergency', 'pct_within_30_emergency', 'pct_within_60_emergency']
]
print(worst.to_string(index=False))

Creating district level summary...
✅ District summary: (260, 13)
✅ Saved to district_summary.csv

Top 10 worst districts for emergency access:
                   District     Region  avg_travel_emergency  pct_within_30_emergency  pct_within_60_emergency
                 East Gonja   Savannah                139.53                     0.00                     0.60
   Kwahu Afram Plains South    Eastern                129.14                     1.08                     7.05
                North Gonja   Savannah                108.09                     0.00                     2.52
 Sekyere Afram Plains North    Ashanti                 94.70                     0.00                    12.61
                    Wa East Upper West                 91.55                     2.16                    18.29
                   Pru West  Bono East                 84.92                     0.00                    10.15
   Kwahu Afram Plains North    Eastern                 84.30                    

In [1]:
import pandas as pd
import numpy as np

print("Loading datasets...")
district_df = pd.read_csv(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\district_summary.csv")
master_pop = pd.read_csv(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\master_population_data.csv")

print(f"District summary: {district_df.shape}")
print(f"Master population: {master_pop.shape}")
print(f"\nDistrict columns: {list(district_df.columns)}")
print(f"\nMaster pop columns: {list(master_pop.columns)}")

Loading datasets...
District summary: (260, 13)
Master population: (278001, 25)

District columns: ['Region', 'District', 'total_population', 'population_points', 'avg_travel_any', 'avg_travel_emergency', 'avg_travel_specialist', 'avg_travel_psychiatric', 'pct_within_30_any', 'pct_within_30_emergency', 'pct_within_60_emergency', 'avg_rural_unpaved_pct', 'avg_accessibility_score']

Master pop columns: ['lat', 'lon', 'population', 'nearest_any_min', 'nearest_chps_min', 'nearest_maternity_min', 'nearest_outpatient_min', 'nearest_emergency_min', 'nearest_specialist_min', 'nearest_psychiatric_min', 'node_id', 'nearest_fac_lon', 'nearest_fac_lat', 'nearest_fac_name', 'nearest_fac_type', 'journey_major_road_min', 'journey_connecting_road_min', 'journey_urban_road_min', 'journey_rural_unpaved_min', 'journey_walking_min', 'journey_total_min', 'journey_failed', 'Region', 'District', 'accessibility_score']


In [2]:
import pandas as pd
import numpy as np

print("Building comprehensive district summary from master population data...")

ROAD_SCORES = {
    'journey_major_road_min': 100,
    'journey_urban_road_min': 85,
    'journey_connecting_road_min': 60,
    'journey_rural_unpaved_min': 15,
    'journey_walking_min': 0
}

def band_score(series, thresholds=[30, 60, 90, 120]):
    scores = np.zeros(len(series))
    scores[series <= 30] = 100
    scores[(series > 30) & (series <= 60)] = 70
    scores[(series > 60) & (series <= 90)] = 40
    scores[(series > 90) & (series <= 120)] = 15
    scores[series > 120] = 0
    return scores

rows = []

total = master_pop.groupby(['Region', 'District']).ngroups
count = 0

for (region, district), group in master_pop.groupby(['Region', 'District']):
    if pd.isna(region) or pd.isna(district):
        continue
    count += 1
    if count % 50 == 0:
        print(f"  {count}/{total} districts processed...")

    row = {
        'Region': region,
        'District': district,
        'total_population': round(group['population'].sum()),
        'population_points': len(group),
    }

    # ACCESS BANDS
    for col, label in [
        ('nearest_emergency_min', 'emergency'),
        ('nearest_any_min', 'any'),
        ('nearest_specialist_min', 'specialist'),
    ]:
        vals = group[col].dropna()
        row[f'{label}_pct_within_30'] = round((vals <= 30).mean() * 100, 1)
        row[f'{label}_pct_30_60']     = round(((vals > 30) & (vals <= 60)).mean() * 100, 1)
        row[f'{label}_pct_60_90']     = round(((vals > 60) & (vals <= 90)).mean() * 100, 1)
        row[f'{label}_pct_90_120']    = round(((vals > 90) & (vals <= 120)).mean() * 100, 1)
        row[f'{label}_pct_120plus']   = round((vals > 120).mean() * 100, 1)
        row[f'{label}_band_score']    = round(band_score(vals).mean(), 2)

    # ROAD QUALITY SCORE
    road_cols = list(ROAD_SCORES.keys())
    avg_road_times = group[road_cols].mean()
    total_road_time = avg_road_times.sum()
    if total_road_time > 0:
        road_score = sum(
            avg_road_times[col] * ROAD_SCORES[col]
            for col in road_cols
        ) / total_road_time
    else:
        road_score = 0
    row['road_quality_score'] = round(road_score, 2)

    # Road type breakdown
    for col in road_cols:
        label = col.replace('journey_', '').replace('_min', '')
        row[f'road_pct_{label}'] = round(
            avg_road_times[col] / total_road_time * 100 if total_road_time > 0 else 0, 1
        )

    # E2SFCA
    row['avg_accessibility_score'] = round(group['accessibility_score'].mean(), 6)

    rows.append(row)

district_scores = pd.DataFrame(rows)

# Save as brand new dataset
out_path = r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\district_accessibility_scores.csv"
district_scores.to_csv(out_path, index=False)

print(f"\n✅ Done! {len(district_scores)} districts")
print(f"✅ Saved to district_accessibility_scores.csv")
print(f"\nColumns: {list(district_scores.columns)}")
print(f"\nSample:")
print(district_scores[['Region', 'District', 'emergency_band_score', 'road_quality_score', 'avg_accessibility_score']].head(10))

Building comprehensive district summary from master population data...
  50/260 districts processed...
  100/260 districts processed...
  150/260 districts processed...
  200/260 districts processed...
  250/260 districts processed...

✅ Done! 260 districts
✅ Saved to district_accessibility_scores.csv

Columns: ['Region', 'District', 'total_population', 'population_points', 'emergency_pct_within_30', 'emergency_pct_30_60', 'emergency_pct_60_90', 'emergency_pct_90_120', 'emergency_pct_120plus', 'emergency_band_score', 'any_pct_within_30', 'any_pct_30_60', 'any_pct_60_90', 'any_pct_90_120', 'any_pct_120plus', 'any_band_score', 'specialist_pct_within_30', 'specialist_pct_30_60', 'specialist_pct_60_90', 'specialist_pct_90_120', 'specialist_pct_120plus', 'specialist_band_score', 'road_quality_score', 'road_pct_major_road', 'road_pct_urban_road', 'road_pct_connecting_road', 'road_pct_rural_unpaved', 'road_pct_walking', 'avg_accessibility_score']

Sample:
    Region         District  emergenc

In [ ]:
----------------------------------------------------------------------
-----------------------------------------------------------------------

In [3]:
import pandas as pd
import numpy as np

# ── NORMALIZE E2SFCA TO 0-100 ──
e2sfca_min = district_scores['avg_accessibility_score'].min()
e2sfca_max = district_scores['avg_accessibility_score'].max()

district_scores['e2sfca_normalized'] = round(
    (district_scores['avg_accessibility_score'] - e2sfca_min) / 
    (e2sfca_max - e2sfca_min) * 100, 2
)

# ── COMPOSITE SCORE ──
# Emergency access: 35%
# E2SFCA:          25%
# Specialist:      20%
# Road quality:    10%
# Any facility:    10%

district_scores['composite_score'] = round(
    (district_scores['emergency_band_score']  * 0.35) +
    (district_scores['e2sfca_normalized']     * 0.25) +
    (district_scores['specialist_band_score'] * 0.20) +
    (district_scores['road_quality_score']    * 0.10) +
    (district_scores['any_band_score']        * 0.10),
    2
)

# ── BIN INTO CATEGORIES ──
def categorize(score):
    if score >= 80: return 'Thriving'
    elif score >= 60: return 'Decent'
    elif score >= 40: return 'Getting By'
    elif score >= 20: return 'Struggling'
    else: return 'Dire'

district_scores['category'] = district_scores['composite_score'].apply(categorize)

# ── SAVE ──
out_path = r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\district_accessibility_scores.csv"
district_scores.to_csv(out_path, index=False)
print(f"✅ Saved!")

# ── SUMMARY ──
print(f"\n=== COMPOSITE SCORE SUMMARY ===")
print(f"\nTop 10 best districts:")
print(district_scores.nlargest(10, 'composite_score')[['District', 'Region', 'composite_score', 'category']].to_string(index=False))

print(f"\nTop 10 worst districts:")
print(district_scores.nsmallest(10, 'composite_score')[['District', 'Region', 'composite_score', 'category']].to_string(index=False))

print(f"\nCategory distribution:")
print(district_scores['category'].value_counts())

✅ Saved!

=== COMPOSITE SCORE SUMMARY ===

Top 10 best districts:
       District        Region  composite_score category
     Bolga East    Upper East            95.37 Thriving
   Wa Municipal    Upper West            90.92 Thriving
  Nadowli-Kaleo    Upper West            83.96 Thriving
     Bolgatanga    Upper East            82.70 Thriving
Ayawaso Central Greater Accra            82.54 Thriving
   Ayawaso East Greater Accra            82.49 Thriving
 Ablekuma North Greater Accra            82.24 Thriving
         Nabdam    Upper East            81.98 Thriving
  Ayawaso North Greater Accra            81.83 Thriving
 Okaikwei North Greater Accra            81.76 Thriving

Top 10 worst districts:
                   District     Region  composite_score   category
                 East Gonja   Savannah            13.51       Dire
   Kwahu Afram Plains South    Eastern            18.63       Dire
                   Pru West  Bono East            24.38 Struggling
 Sekyere Afram Plains Nor

In [4]:
# Pull raw numbers for the top 10 and bottom 10 districts
# So we can sanity check with our own eyes

check_cols = [
    'Region', 'District',
    'emergency_pct_within_30',   # % reaching emergency care in 30 min
    'emergency_pct_120plus',     # % taking over 2 hours for emergency
    'specialist_pct_within_30',  # % reaching specialist in 30 min
    'any_pct_within_30',         # % reaching any facility in 30 min
    'road_pct_rural_unpaved',    # % of journey on dirt roads
    'road_pct_major_road',       # % of journey on major roads
    'avg_accessibility_score',   # raw E2SFCA
    'composite_score',
    'category'
]

print("=== TOP 10 DISTRICTS — RAW NUMBERS ===")
top10 = district_scores.nlargest(10, 'composite_score')[check_cols]
print(top10.to_string(index=False))

print("\n=== BOTTOM 10 DISTRICTS — RAW NUMBERS ===")
bottom10 = district_scores.nsmallest(10, 'composite_score')[check_cols]
print(bottom10.to_string(index=False))

=== TOP 10 DISTRICTS — RAW NUMBERS ===
       Region        District  emergency_pct_within_30  emergency_pct_120plus  specialist_pct_within_30  any_pct_within_30  road_pct_rural_unpaved  road_pct_major_road  avg_accessibility_score  composite_score category
   Upper East      Bolga East                    100.0                    0.0                      98.8              100.0                    41.4                  4.5                 0.000834            95.37 Thriving
   Upper West    Wa Municipal                     93.9                    0.0                      90.0              100.0                    52.4                  9.7                 0.000769            90.92 Thriving
   Upper West   Nadowli-Kaleo                     84.6                    0.0                      37.9              100.0                    61.6                  5.2                 0.000717            83.96 Thriving
   Upper East      Bolgatanga                     78.3                    0.0        

In [5]:
#these outputs are suspicious. i am tryign a different approach again.

In [6]:
import pandas as pd
import numpy as np

print("Loading data...")
district_scores = pd.read_csv(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\district_accessibility_scores.csv")
print(f"✅ {len(district_scores)} districts loaded")

# ══════════════════════════════════════════
# STEP 1 — EMERGENCY BAND SCORE (already computed)
# bands: 0-30=100, 30-60=70, 60-90=40, 90-120=15, 120+=0
# this is already in emergency_band_score column ✅

# ══════════════════════════════════════════
# STEP 2 — SPECIALIST BAND SCORE (wider bands)
# bands: 0-60=100, 60-120=70, 120-180=40, 180+=0

print("\nRecomputing specialist band score with wider bands...")

master_pop = pd.read_csv(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\master_population_data.csv")

def specialist_band_score(series):
    scores = np.zeros(len(series))
    scores[series <= 60] = 100
    scores[(series > 60) & (series <= 120)] = 70
    scores[(series > 120) & (series <= 180)] = 40
    scores[series > 180] = 0
    return scores

specialist_scores = {}
for (region, district), group in master_pop.groupby(['Region', 'District']):
    if pd.isna(region) or pd.isna(district):
        continue
    vals = group['nearest_specialist_min'].dropna()
    specialist_scores[f"{region}|{district}"] = round(specialist_band_score(vals).mean(), 2)

district_scores['specialist_band_score_wide'] = district_scores.apply(
    lambda r: specialist_scores.get(f"{r['Region']}|{r['District']}", np.nan), axis=1
)

print("✅ Specialist band score recomputed with wider bands")

# ══════════════════════════════════════════
# STEP 3 — JOURNEY ACCESS SCORE
# Emergency 50% + Specialist 35% + Any facility 15%

print("\nComputing Journey Access Score...")

district_scores['journey_access_score'] = round(
    (district_scores['emergency_band_score']      * 0.50) +
    (district_scores['specialist_band_score_wide'] * 0.35) +
    (district_scores['any_band_score']            * 0.15),
    2
)

print("✅ Journey Access Score computed")

# ══════════════════════════════════════════
# STEP 4 — SUPPLY ADEQUACY SCORE
# Normalize E2SFCA to 0-100

print("\nNormalizing E2SFCA to 0-100...")

e2sfca_min = district_scores['avg_accessibility_score'].min()
e2sfca_max = district_scores['avg_accessibility_score'].max()

district_scores['supply_adequacy_score'] = round(
    (district_scores['avg_accessibility_score'] - e2sfca_min) /
    (e2sfca_max - e2sfca_min) * 100, 2
)

print("✅ Supply Adequacy Score computed")

# ══════════════════════════════════════════
# STEP 5 — FINAL COMPOSITE SCORE

print("\nComputing Final Composite Score...")

district_scores['composite_score'] = round(
    (district_scores['journey_access_score']  * 0.70) +
    (district_scores['supply_adequacy_score'] * 0.30),
    2
)

print("✅ Final Composite Score computed")

# ══════════════════════════════════════════
# STEP 6 — CATEGORIES

def categorize(score):
    if score >= 80:   return 'Thriving'
    elif score >= 60: return 'Decent'
    elif score >= 40: return 'Getting By'
    elif score >= 20: return 'Struggling'
    else:             return 'Dire'

district_scores['category'] = district_scores['composite_score'].apply(categorize)

# ══════════════════════════════════════════
# STEP 7 — SAVE

out_path = r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\district_accessibility_scores.csv"
district_scores.to_csv(out_path, index=False)
print(f"\n✅ Saved!")

# ══════════════════════════════════════════
# STEP 8 — RESULTS

print(f"\n=== CATEGORY DISTRIBUTION ===")
print(district_scores['category'].value_counts())

print(f"\n=== TOP 10 DISTRICTS ===")
print(district_scores.nlargest(10, 'composite_score')[
    ['District', 'Region', 'journey_access_score', 
     'supply_adequacy_score', 'composite_score', 'category']
].to_string(index=False))

print(f"\n=== BOTTOM 10 DISTRICTS ===")
print(district_scores.nsmallest(10, 'composite_score')[
    ['District', 'Region', 'journey_access_score',
     'supply_adequacy_score', 'composite_score', 'category']
].to_string(index=False))

Loading data...
✅ 260 districts loaded

Recomputing specialist band score with wider bands...
✅ Specialist band score recomputed with wider bands

Computing Journey Access Score...
✅ Journey Access Score computed

Normalizing E2SFCA to 0-100...
✅ Supply Adequacy Score computed

Computing Final Composite Score...
✅ Final Composite Score computed

✅ Saved!

=== CATEGORY DISTRIBUTION ===
category
Decent        161
Getting By     62
Thriving       26
Struggling      9
Dire            2
Name: count, dtype: int64

=== TOP 10 DISTRICTS ===
     District     Region  journey_access_score  supply_adequacy_score  composite_score category
   Bolga East Upper East                100.00                 100.00           100.00 Thriving
 Wa Municipal Upper West                 99.08                  90.82            96.60 Thriving
Nadowli-Kaleo Upper West                 97.54                  83.47            93.32 Thriving
       Nabdam Upper East                 98.69                  73.45        

In [7]:
# Check which district is missing
import pandas as pd

# Load master population data
master_pop = pd.read_csv(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\master_population_data.csv")

# Check all unique districts
districts_in_data = set(master_pop['District'].dropna().unique())
districts_in_scores = set(district_scores['District'].unique())

print(f"Districts in population data: {len(districts_in_data)}")
print(f"Districts in scores: {len(districts_in_scores)}")
print(f"\nMissing from scores:")
print(districts_in_data - districts_in_scores)

# Also check unmatched population points
print(f"\nUnmatched population points (no district): {master_pop['District'].isna().sum()}")

Districts in population data: 260
Districts in scores: 260

Missing from scores:
set()

Unmatched population points (no district): 806


In [8]:
# Check for Jasikan and Biakoye
print("Searching for Jasikan and Biakoye...")

print("\nIn master_population_data:")
for district in ['Jasikan', 'Biakoye', 'Guan']:
    count = (master_pop['District'] == district).sum()
    print(f"  {district}: {count} population points")

print("\nIn district_scores:")
for district in ['Jasikan', 'Biakoye', 'Guan']:
    found = district in district_scores['District'].values
    print(f"  {district}: {'✅ found' if found else '❌ not found'}")

print("\nAll Oti Region districts in scores:")
oti = district_scores[district_scores['Region'] == 'Oti']
print(oti[['Region', 'District']].to_string(index=False))

Searching for Jasikan and Biakoye...

In master_population_data:
  Jasikan: 632 population points
  Biakoye: 1008 population points
  Guan: 0 population points

In district_scores:
  Jasikan: ✅ found
  Biakoye: ✅ found
  Guan: ❌ not found

All Oti Region districts in scores:
Region        District
   Oti         Biakoye
   Oti         Jasikan
   Oti         Kadjebi
   Oti     Krachi East
   Oti Krachi Nchumuru
   Oti     Krachi West
   Oti   Nkwanta North
   Oti   Nkwanta South


### **Note:**

#### 261 districts were targeted. GUAN district (Oti Region, created 2021) is excluded from all spatial analysis due to its absence from GADM boundary datasets and WorldPop population grids at the time of this study. All other 260 districts are represented, This is because the Guan District (which covers the Santrokofi, Akpafu, Likpe, and Lolobi areas—often called the SALL areas) has a slightly two-part creation history because of the gap between the law passing and the actual setup:The legal creation: It was legally carved out of the Biakoye District in 2020 under Legislative Instrument (LI) 2416.The official inauguration: The district was formally inaugurated and its assembly members sworn into office on October 8, 2021, with its capital at Likpe-Mate.  Because it's one of Ghana's newest districts, it is a textbook example of why your composite score redesign is so vital. If you are looking at historical health data, Guan doesn't even show up in datasets prior to 2021, its data as it would still be folded into Biakoye.

In [9]:
import geopandas as gpd

regions = gpd.read_file(r"C:\Users\hp\Downloads\Code & Scripts\gadm41_GHA_1.shp")
print(f"Regions: {len(regions)}")
print(regions['NAME_1'].tolist())

Regions: 16
['Ahafo', 'Ashanti', 'Bono', 'Bono East', 'Central', 'Eastern', 'Greater Accra', 'North East', 'Northern', 'Oti', 'Savannah', 'Upper East', 'Upper West', 'Volta', 'Western', 'Western North']


In [10]:
import geopandas as gpd
import json

print("Loading GADM Level 1 (regions)...")
regions = gpd.read_file(r"C:\Users\hp\Downloads\Code & Scripts\gadm41_GHA_1.shp")

# Simplify geometry slightly to reduce file size
print("Simplifying geometry...")
regions_simplified = regions.copy()
regions_simplified['geometry'] = regions.geometry.simplify(0.01, preserve_topology=True)

# Keep only what we need
regions_simplified = regions_simplified[['NAME_1', 'geometry']]

# Convert to GeoJSON
print("Converting to GeoJSON...")
geojson = json.loads(regions_simplified.to_json())

# Save to file
out_path = r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\ghana_regions_16.geojson"
with open(out_path, 'w') as f:
    json.dump(geojson, f)

print(f"✅ Saved to ghana_regions_16.geojson")
print(f"Regions in file: {len(geojson['features'])}")
print(f"Region names: {[f['properties']['NAME_1'] for f in geojson['features']]}")

# Check file size
import os
size_kb = os.path.getsize(out_path) / 1024
print(f"File size: {round(size_kb)} KB")

Loading GADM Level 1 (regions)...
Simplifying geometry...
Converting to GeoJSON...
✅ Saved to ghana_regions_16.geojson
Regions in file: 16
Region names: ['Ahafo', 'Ashanti', 'Bono', 'Bono East', 'Central', 'Eastern', 'Greater Accra', 'North East', 'Northern', 'Oti', 'Savannah', 'Upper East', 'Upper West', 'Volta', 'Western', 'Western North']
File size: 87 KB


In [11]:
# moving unto final stage

In [12]:
import geopandas as gpd
import json
import pandas as pd
import os

project_path = r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project"

# ── 1. SAVE 16-REGION GEOJSON (already done but let's confirm) ──
geojson_path = os.path.join(project_path, "ghana_regions_16.geojson")
if os.path.exists(geojson_path):
    print(f"✅ ghana_regions_16.geojson already saved")
else:
    regions = gpd.read_file(r"C:\Users\hp\Downloads\Code & Scripts\gadm41_GHA_1.shp")
    regions['geometry'] = regions.geometry.simplify(0.01, preserve_topology=True)
    regions = regions[['NAME_1', 'geometry']]
    geojson = json.loads(regions.to_json())
    with open(geojson_path, 'w') as f:
        json.dump(geojson, f)
    print(f"✅ ghana_regions_16.geojson saved")

# ── 2. SAVE DISTRICT SCORES AS JSON ──
scores_df = pd.read_csv(os.path.join(project_path, "district_accessibility_scores.csv"))

# Build clean JSON for HTML injection
district_json = []
for _, row in scores_df.iterrows():
    district_json.append({
        "region": row['Region'],
        "district": row['District'],
        "composite_score": round(row['composite_score'], 1),
        "journey_access_score": round(row['journey_access_score'], 1),
        "supply_adequacy_score": round(row['supply_adequacy_score'], 1),
        "category": row['category'],
        # Emergency bands
        "emergency_within_30": round(row['emergency_pct_within_30'], 1),
        "emergency_30_60": round(row['emergency_pct_30_60'], 1),
        "emergency_60_90": round(row['emergency_pct_60_90'], 1),
        "emergency_90_120": round(row['emergency_pct_90_120'], 1),
        "emergency_120plus": round(row['emergency_pct_120plus'], 1),
        # Specialist bands
        "specialist_within_30": round(row['specialist_pct_within_30'], 1),
        "specialist_30_60": round(row['specialist_pct_30_60'], 1),
        "specialist_60_90": round(row['specialist_pct_60_90'], 1),
        "specialist_90_120": round(row['specialist_pct_90_120'], 1),
        "specialist_120plus": round(row['specialist_pct_120plus'], 1),
        # Any facility bands
        "any_within_30": round(row['any_pct_within_30'], 1),
        "any_30_60": round(row['any_pct_30_60'], 1),
        "any_60_90": round(row['any_pct_60_90'], 1),
        "any_90_120": round(row['any_pct_90_120'], 1),
        "any_120plus": round(row['any_pct_120plus'], 1),
        # Road breakdown
        "road_pct_major": round(row['road_pct_major_road'], 1),
        "road_pct_urban": round(row['road_pct_urban_road'], 1),
        "road_pct_connecting": round(row['road_pct_connecting_road'], 1),
        "road_pct_unpaved": round(row['road_pct_rural_unpaved'], 1),
        "road_pct_walking": round(row['road_pct_walking'], 1),
        "road_quality_score": round(row['road_quality_score'], 1),
        "total_population": int(row['total_population']),
    })

json_path = os.path.join(project_path, "district_scores.json")
with open(json_path, 'w') as f:
    json.dump(district_json, f)
print(f"✅ district_scores.json saved — {len(district_json)} districts")

# ── 3. SAVE REGION SUMMARY ──
region_summary = scores_df.groupby('Region').agg(
    avg_composite=('composite_score', 'mean'),
    avg_journey=('journey_access_score', 'mean'),
    avg_supply=('supply_adequacy_score', 'mean'),
    total_population=('total_population', 'sum'),
    num_districts=('District', 'count'),
    worst_district_emergency=('emergency_pct_within_30', 'min'),
    best_district_emergency=('emergency_pct_within_30', 'max'),
).round(1).reset_index()

region_path = os.path.join(project_path, "region_summary.csv")
region_summary.to_csv(region_path, index=False)
print(f"✅ region_summary.csv saved — {len(region_summary)} regions")

# ── 4. CONFIRM ALL KEY FILES EXIST ──
print(f"\n=== FILE CHECK ===")
files = [
    "ghana_regions_16.geojson",
    "district_scores.json",
    "district_accessibility_scores.csv",
    "region_summary.csv",
    "master_population_data.csv",
    "master_dataset_v3.csv",
    "travel_time_summary.csv",
    "journey_breakdown.csv",
]
for f in files:
    path = os.path.join(project_path, f)
    exists = os.path.exists(path)
    size = round(os.path.getsize(path) / 1024) if exists else 0
    print(f"  {'✅' if exists else '❌'} {f} — {size} KB")

✅ ghana_regions_16.geojson already saved
✅ district_scores.json saved — 260 districts
✅ region_summary.csv saved — 16 regions

=== FILE CHECK ===
  ✅ ghana_regions_16.geojson — 87 KB
  ✅ district_scores.json — 179 KB
  ✅ district_accessibility_scores.csv — 49 KB
  ✅ region_summary.csv — 1 KB
  ✅ master_population_data.csv — 82562 KB
  ✅ master_dataset_v3.csv — 2672 KB
  ✅ travel_time_summary.csv — 1 KB
  ✅ journey_breakdown.csv — 72165 KB


In [13]:
import pandas as pd
import numpy as np

print("Building rich region summary from master population data...")

master_pop = pd.read_csv(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\master_population_data.csv")
district_scores = pd.read_csv(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\district_accessibility_scores.csv")

def band_score_emergency(series):
    scores = np.zeros(len(series))
    scores[series <= 30] = 100
    scores[(series > 30) & (series <= 60)] = 70
    scores[(series > 60) & (series <= 90)] = 40
    scores[(series > 90) & (series <= 120)] = 15
    scores[series > 120] = 0
    return scores

def band_score_specialist(series):
    scores = np.zeros(len(series))
    scores[series <= 60] = 100
    scores[(series > 60) & (series <= 120)] = 70
    scores[(series > 120) & (series <= 180)] = 40
    scores[series > 180] = 0
    return scores

rows = []

for region, group in master_pop.groupby('Region'):
    if pd.isna(region):
        continue

    row = {
        'Region': region,
        'total_population': round(group['population'].sum()),
        'population_points': len(group),
        'num_districts': group['District'].nunique(),
    }

    # ── EMERGENCY BANDS ──
    emerg = group['nearest_emergency_min'].dropna()
    row['emergency_pct_within_30'] = round((emerg <= 30).mean() * 100, 1)
    row['emergency_pct_30_60']     = round(((emerg > 30) & (emerg <= 60)).mean() * 100, 1)
    row['emergency_pct_60_90']     = round(((emerg > 60) & (emerg <= 90)).mean() * 100, 1)
    row['emergency_pct_90_120']    = round(((emerg > 90) & (emerg <= 120)).mean() * 100, 1)
    row['emergency_pct_120plus']   = round((emerg > 120).mean() * 100, 1)
    row['emergency_median_min']    = round(emerg.median(), 1)
    row['emergency_band_score']    = round(band_score_emergency(emerg).mean(), 2)

    # ── ANY FACILITY BANDS ──
    any_fac = group['nearest_any_min'].dropna()
    row['any_pct_within_30']  = round((any_fac <= 30).mean() * 100, 1)
    row['any_pct_30_60']      = round(((any_fac > 30) & (any_fac <= 60)).mean() * 100, 1)
    row['any_pct_60_90']      = round(((any_fac > 60) & (any_fac <= 90)).mean() * 100, 1)
    row['any_pct_90_120']     = round(((any_fac > 90) & (any_fac <= 120)).mean() * 100, 1)
    row['any_pct_120plus']    = round((any_fac > 120).mean() * 100, 1)
    row['any_median_min']     = round(any_fac.median(), 1)
    row['any_band_score']     = round(band_score_emergency(any_fac).mean(), 2)

    # ── SPECIALIST BANDS ──
    spec = group['nearest_specialist_min'].dropna()
    row['specialist_pct_within_60']  = round((spec <= 60).mean() * 100, 1)
    row['specialist_pct_60_120']     = round(((spec > 60) & (spec <= 120)).mean() * 100, 1)
    row['specialist_pct_120_180']    = round(((spec > 120) & (spec <= 180)).mean() * 100, 1)
    row['specialist_pct_180plus']    = round((spec > 180).mean() * 100, 1)
    row['specialist_median_min']     = round(spec.median(), 1)
    row['specialist_band_score']     = round(band_score_specialist(spec).mean(), 2)

    # ── PSYCHIATRIC ──
    psych = group['nearest_psychiatric_min'].dropna()
    row['psychiatric_pct_within_60'] = round((psych <= 60).mean() * 100, 1)
    row['psychiatric_median_min']    = round(psych.median(), 1)

    # ── ROAD QUALITY ──
    road_cols = {
        'journey_major_road_min': 100,
        'journey_urban_road_min': 85,
        'journey_connecting_road_min': 60,
        'journey_rural_unpaved_min': 15,
        'journey_walking_min': 0
    }
    avg_road = group[list(road_cols.keys())].mean()
    total_road = avg_road.sum()
    if total_road > 0:
        road_score = sum(avg_road[c] * s for c, s in road_cols.items()) / total_road
    else:
        road_score = 0
    row['road_quality_score']       = round(road_score, 2)
    row['road_pct_major_road']      = round(avg_road['journey_major_road_min'] / total_road * 100, 1)
    row['road_pct_urban_road']      = round(avg_road['journey_urban_road_min'] / total_road * 100, 1)
    row['road_pct_connecting_road'] = round(avg_road['journey_connecting_road_min'] / total_road * 100, 1)
    row['road_pct_rural_unpaved']   = round(avg_road['journey_rural_unpaved_min'] / total_road * 100, 1)
    row['road_pct_walking']         = round(avg_road['journey_walking_min'] / total_road * 100, 1)

    # ── E2SFCA ──
    row['avg_accessibility_score'] = round(group['accessibility_score'].mean(), 6)

    # ── DISTRICT LEVEL STATS ──
    region_districts = district_scores[district_scores['Region'] == region]
    if len(region_districts) > 0:
        row['avg_composite_score']    = round(region_districts['composite_score'].mean(), 1)
        row['avg_journey_score']      = round(region_districts['journey_access_score'].mean(), 1)
        row['avg_supply_score']       = round(region_districts['supply_adequacy_score'].mean(), 1)
        row['best_district']          = region_districts.loc[region_districts['composite_score'].idxmax(), 'District']
        row['best_district_score']    = round(region_districts['composite_score'].max(), 1)
        row['worst_district']         = region_districts.loc[region_districts['composite_score'].idxmin(), 'District']
        row['worst_district_score']   = round(region_districts['composite_score'].min(), 1)
        row['num_thriving']           = (region_districts['category'] == 'Thriving').sum()
        row['num_decent']             = (region_districts['category'] == 'Decent').sum()
        row['num_getting_by']         = (region_districts['category'] == 'Getting By').sum()
        row['num_struggling']         = (region_districts['category'] == 'Struggling').sum()
        row['num_dire']               = (region_districts['category'] == 'Dire').sum()

    rows.append(row)

region_summary = pd.DataFrame(rows)

# Normalize E2SFCA
e_min = region_summary['avg_accessibility_score'].min()
e_max = region_summary['avg_accessibility_score'].max()
region_summary['e2sfca_normalized'] = round(
    (region_summary['avg_accessibility_score'] - e_min) /
    (e_max - e_min) * 100, 2
)

# Save
out_path = r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\region_summary.csv"
region_summary.to_csv(out_path, index=False)

print(f"✅ Saved! {len(region_summary)} regions × {len(region_summary.columns)} columns")
print(f"\nColumns: {list(region_summary.columns)}")
print(f"\nSample:")
print(region_summary[['Region', 'emergency_pct_within_30', 'specialist_median_min',
                        'road_pct_rural_unpaved', 'avg_composite_score',
                        'worst_district', 'worst_district_score']].to_string(index=False))

Building rich region summary from master population data...
✅ Saved! 16 regions × 46 columns

Columns: ['Region', 'total_population', 'population_points', 'num_districts', 'emergency_pct_within_30', 'emergency_pct_30_60', 'emergency_pct_60_90', 'emergency_pct_90_120', 'emergency_pct_120plus', 'emergency_median_min', 'emergency_band_score', 'any_pct_within_30', 'any_pct_30_60', 'any_pct_60_90', 'any_pct_90_120', 'any_pct_120plus', 'any_median_min', 'any_band_score', 'specialist_pct_within_60', 'specialist_pct_60_120', 'specialist_pct_120_180', 'specialist_pct_180plus', 'specialist_median_min', 'specialist_band_score', 'psychiatric_pct_within_60', 'psychiatric_median_min', 'road_quality_score', 'road_pct_major_road', 'road_pct_urban_road', 'road_pct_connecting_road', 'road_pct_rural_unpaved', 'road_pct_walking', 'avg_accessibility_score', 'avg_composite_score', 'avg_journey_score', 'avg_supply_score', 'best_district', 'best_district_score', 'worst_district', 'worst_district_score', 'num_

In [14]:
# Compute proper regional composite score
# Using same methodology as districts

# Journey Access Score per region
region_summary['journey_access_score'] = round(
    (region_summary['emergency_band_score']  * 0.50) +
    (region_summary['specialist_band_score'] * 0.35) +
    (region_summary['any_band_score']        * 0.15),
    2
)

# Supply Adequacy Score — E2SFCA already normalized
region_summary['supply_adequacy_score'] = region_summary['e2sfca_normalized']

# Final composite
region_summary['composite_score'] = round(
    (region_summary['journey_access_score']  * 0.70) +
    (region_summary['supply_adequacy_score'] * 0.30),
    2
)

# Categories
def categorize(score):
    if score >= 80:   return 'Thriving'
    elif score >= 60: return 'Decent'
    elif score >= 40: return 'Getting By'
    elif score >= 20: return 'Struggling'
    else:             return 'Dire'

region_summary['category'] = region_summary['composite_score'].apply(categorize)

# Save
out_path = r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\region_summary.csv"
region_summary.to_csv(out_path, index=False)
print(f"✅ Saved!")

# Show results
print(f"\n=== REGIONAL SCORES ===")
print(region_summary[['Region', 'journey_access_score', 
                        'supply_adequacy_score', 
                        'composite_score', 
                        'category']].sort_values('composite_score', ascending=False).to_string(index=False))

✅ Saved!

=== REGIONAL SCORES ===
       Region  journey_access_score  supply_adequacy_score  composite_score   category
   Upper East                 86.10                  91.04            87.58   Thriving
   Upper West                 66.86                 100.00            76.80     Decent
Greater Accra                 96.62                  20.07            73.66     Decent
        Ahafo                 85.09                  39.43            71.39     Decent
         Bono                 77.94                  44.44            67.89     Decent
        Volta                 88.64                  11.11            65.38     Decent
Western North                 65.65                  59.86            63.91     Decent
      Western                 73.28                  37.99            62.69     Decent
      Central                 86.87                   3.58            61.88     Decent
   North East                 71.29                  20.07            55.92 Getting By
      Ash

In [15]:
import json

region_json = []
for _, row in region_summary.iterrows():
    region_json.append({k: (None if str(v) == 'nan' else v) for k, v in row.items()})

out_path = r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\region_scores.json"
with open(out_path, 'w') as f:
    json.dump(region_json, f)

print(f"✅ Saved region_scores.json — {len(region_json)} regions")

✅ Saved region_scores.json — 16 regions


In [16]:
import pandas as pd

PROJECT_DIR = r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project"

df = pd.read_csv(PROJECT_DIR + r"\master_population_data.csv")

district_times = df.groupby(["Region", "District"]).agg(
    emergency_median_min=("nearest_emergency_min", "median"),
    any_median_min=("nearest_any_min", "median"),
    specialist_median_min=("nearest_specialist_min", "median"),
    psychiatric_median_min=("nearest_psychiatric_min", "median"),
).round(1).reset_index()

district_times.to_csv(PROJECT_DIR + r"\district_travel_times.csv", index=False)
print("Done!")
print(district_times.head())

Done!
  Region       District  emergency_median_min  any_median_min  \
0  Ahafo  Asunafo North                  35.2             7.7   
1  Ahafo  Asunafo South                  24.0             6.1   
2  Ahafo  Asutifi North                  29.6             7.5   
3  Ahafo  Asutifi South                  20.0             6.1   
4  Ahafo     Tano North                  17.8             5.3   

   specialist_median_min  psychiatric_median_min  
0                  102.4                   329.2  
1                  104.4                   312.9  
2                   63.5                   313.1  
3                   71.5                   293.4  
4                   42.7                   265.3  


In [17]:
import pandas as pd
import numpy as np

print("Adding outpatient bands to district and region data...")

# Load existing files
district_scores = pd.read_csv(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\district_accessibility_scores.csv")
region_summary = pd.read_csv(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\region_summary.csv")
master_pop = pd.read_csv(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\master_population_data.csv")

def band_score_outpatient(series):
    scores = np.zeros(len(series))
    scores[series <= 30] = 100
    scores[(series > 30) & (series <= 60)] = 70
    scores[(series > 60) & (series <= 90)] = 40
    scores[(series > 90) & (series <= 120)] = 15
    scores[series > 120] = 0
    return scores

# ── DISTRICT LEVEL ──
print("\nComputing outpatient bands for districts...")
district_outpatient = {}

for (region, district), group in master_pop.groupby(['Region', 'District']):
    if pd.isna(region) or pd.isna(district):
        continue
    vals = group['nearest_outpatient_min'].dropna()
    district_outpatient[f"{region}|{district}"] = {
        'outpatient_pct_within_30': round((vals <= 30).mean() * 100, 1),
        'outpatient_pct_30_60':     round(((vals > 30) & (vals <= 60)).mean() * 100, 1),
        'outpatient_pct_60_90':     round(((vals > 60) & (vals <= 90)).mean() * 100, 1),
        'outpatient_pct_90_120':    round(((vals > 90) & (vals <= 120)).mean() * 100, 1),
        'outpatient_pct_120plus':   round((vals > 120).mean() * 100, 1),
        'outpatient_median_min':    round(vals.median(), 1),
        'outpatient_band_score':    round(band_score_outpatient(vals).mean(), 2),
    }

# Add to district_scores
for col in ['outpatient_pct_within_30', 'outpatient_pct_30_60', 'outpatient_pct_60_90',
            'outpatient_pct_90_120', 'outpatient_pct_120plus', 
            'outpatient_median_min', 'outpatient_band_score']:
    district_scores[col] = district_scores.apply(
        lambda r: district_outpatient.get(f"{r['Region']}|{r['District']}", {}).get(col, np.nan), axis=1
    )

print(f"✅ Districts done!")

# ── REGION LEVEL ──
print("\nComputing outpatient bands for regions...")

for region, group in master_pop.groupby('Region'):
    if pd.isna(region):
        continue
    vals = group['nearest_outpatient_min'].dropna()
    idx = region_summary[region_summary['Region'] == region].index
    if len(idx) == 0:
        continue
    region_summary.loc[idx, 'outpatient_pct_within_30'] = round((vals <= 30).mean() * 100, 1)
    region_summary.loc[idx, 'outpatient_pct_30_60']     = round(((vals > 30) & (vals <= 60)).mean() * 100, 1)
    region_summary.loc[idx, 'outpatient_pct_60_90']     = round(((vals > 60) & (vals <= 90)).mean() * 100, 1)
    region_summary.loc[idx, 'outpatient_pct_90_120']    = round(((vals > 90) & (vals <= 120)).mean() * 100, 1)
    region_summary.loc[idx, 'outpatient_pct_120plus']   = round((vals > 120).mean() * 100, 1)
    region_summary.loc[idx, 'outpatient_median_min']    = round(vals.median(), 1)
    region_summary.loc[idx, 'outpatient_band_score']    = round(band_score_outpatient(vals).mean(), 2)

print(f"✅ Regions done!")

# ── SAVE BOTH FILES ──
district_path = r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\district_accessibility_scores.csv"
region_path = r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\region_summary.csv"

district_scores.to_csv(district_path, index=False)
region_summary.to_csv(region_path, index=False)

print(f"\n✅ district_accessibility_scores.csv saved — {district_scores.shape}")
print(f"✅ region_summary.csv saved — {region_summary.shape}")

# ── QUICK SENSE CHECK ──
print(f"\n=== OUTPATIENT ACCESS BY REGION ===")
print(region_summary[['Region', 'outpatient_pct_within_30', 
                        'outpatient_median_min']].sort_values(
    'outpatient_pct_within_30', ascending=False).to_string(index=False))

Adding outpatient bands to district and region data...

Computing outpatient bands for districts...
✅ Districts done!

Computing outpatient bands for regions...
✅ Regions done!

✅ district_accessibility_scores.csv saved — (260, 42)
✅ region_summary.csv saved — (16, 57)

=== OUTPATIENT ACCESS BY REGION ===
       Region  outpatient_pct_within_30  outpatient_median_min
Greater Accra                      97.8                    7.3
        Volta                      94.0                    8.5
      Central                      92.1                   10.9
   Upper East                      92.0                   10.5
        Ahafo                      89.2                   14.5
      Ashanti                      84.4                   13.0
         Bono                      83.5                   13.5
      Western                      83.1                   14.7
   North East                      79.2                   17.7
Western North                      78.9                   17.0


In [18]:
import geopandas as gpd
import json

gdf = gpd.read_file(r"C:\Users\hp\Downloads\Code & Scripts\gadm41_GHA_2.shp")
print(gdf.columns.tolist())
print(gdf.shape)
print(gdf.head(3))

['GID_2', 'GID_0', 'COUNTRY', 'GID_1', 'NAME_1', 'NL_NAME_1', 'NAME_2', 'VARNAME_2', 'NL_NAME_2', 'TYPE_2', 'ENGTYPE_2', 'CC_2', 'HASC_2', 'geometry']
(260, 14)
      GID_2 GID_0 COUNTRY   GID_1 NAME_1 NL_NAME_1         NAME_2 VARNAME_2  \
0  GHA1.1_2   GHA   Ghana  GHA1_2  Ahafo        NA  Asunafo North        NA   
1  GHA1.2_2   GHA   Ghana  GHA1_2  Ahafo        NA  Asunafo South        NA   
2  GHA1.3_2   GHA   Ghana  GHA1_2  Ahafo        NA  Asutifi North        NA   

  NL_NAME_2        TYPE_2     ENGTYPE_2 CC_2 HASC_2  \
0        NA  Municipality  Municipality   NA     NA   
1        NA      District      District   NA     NA   
2        NA      District      District   NA     NA   

                                            geometry  
0  POLYGON ((-2.8775 6.65303, -2.87756 6.65323, -...  
1  POLYGON ((-2.83203 6.63043, -2.83157 6.63067, ...  
2  POLYGON ((-2.49059 7.21745, -2.49002 7.2167, -...  


In [19]:
import geopandas as gpd
import json

PROJECT_DIR = r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project"

gdf = gpd.read_file(r"C:\Users\hp\Downloads\Code & Scripts\gadm41_GHA_2.shp")

# Simplify geometry slightly to reduce file size (tolerance in degrees)
gdf['geometry'] = gdf['geometry'].simplify(tolerance=0.005, preserve_topology=True)

# Keep only the columns we need
gdf = gdf[['NAME_1', 'NAME_2', 'geometry']]

# Save as GeoJSON
gdf.to_file(PROJECT_DIR + r"\ghana_districts.geojson", driver='GeoJSON')

# Check file size
import os
size = os.path.getsize(PROJECT_DIR + r"\ghana_districts.geojson") / 1024
print(f"Done! File size: {size:.0f} KB")
print(f"Districts: {len(gdf)}")

Done! File size: 571 KB
Districts: 260


In [20]:
# For each region, estimate emergency journey breakdown
# by applying the known road mix to the emergency travel time

region_summary['emergency_journey_major_road_min'] = round(
    region_summary['road_pct_major_road'] / 100 * region_summary['emergency_median_min'], 1)

region_summary['emergency_journey_connecting_road_min'] = round(
    region_summary['road_pct_connecting_road'] / 100 * region_summary['emergency_median_min'], 1)

region_summary['emergency_journey_urban_road_min'] = round(
    region_summary['road_pct_urban_road'] / 100 * region_summary['emergency_median_min'], 1)

region_summary['emergency_journey_rural_unpaved_min'] = round(
    region_summary['road_pct_rural_unpaved'] / 100 * region_summary['emergency_median_min'], 1)

region_summary['emergency_journey_walking_min'] = round(
    region_summary['road_pct_walking'] / 100 * region_summary['emergency_median_min'], 1)

# Save
out_path = r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\region_summary.csv"
region_summary.to_csv(out_path, index=False)
print(f"✅ Saved!")

# Show results
print(f"\n=== ESTIMATED EMERGENCY JOURNEY BREAKDOWN BY REGION ===")
print(region_summary[['Region', 'emergency_median_min',
                        'emergency_journey_rural_unpaved_min',
                        'emergency_journey_major_road_min',
                        'emergency_journey_connecting_road_min']].sort_values(
    'emergency_journey_rural_unpaved_min', ascending=False).to_string(index=False))

✅ Saved!

=== ESTIMATED EMERGENCY JOURNEY BREAKDOWN BY REGION ===
       Region  emergency_median_min  emergency_journey_rural_unpaved_min  emergency_journey_major_road_min  emergency_journey_connecting_road_min
     Savannah                  65.9                                 44.1                              10.3                                   10.4
   Upper West                  53.1                                 37.5                               8.7                                    6.2
    Bono East                  51.3                                 36.4                               2.7                                   11.6
     Northern                  34.6                                 22.8                               3.4                                    7.4
   North East                  35.2                                 22.6                               7.1                                    4.9
      Eastern                  34.8                       

In [21]:
print("Journey breakdown by road type — regional level...")

region_journey = master_pop.groupby('Region').agg(
    avg_major_road_min=('journey_major_road_min', 'mean'),
    avg_connecting_road_min=('journey_connecting_road_min', 'mean'),
    avg_urban_road_min=('journey_urban_road_min', 'mean'),
    avg_rural_unpaved_min=('journey_rural_unpaved_min', 'mean'),
    avg_walking_min=('journey_walking_min', 'mean'),
    avg_total_min=('journey_total_min', 'mean'),
).round(1).reset_index()

# Add percentages
for col, label in [
    ('avg_major_road_min', 'pct_major_road'),
    ('avg_connecting_road_min', 'pct_connecting_road'),
    ('avg_urban_road_min', 'pct_urban_road'),
    ('avg_rural_unpaved_min', 'pct_rural_unpaved'),
    ('avg_walking_min', 'pct_walking'),
]:
    region_journey[label] = round(
        region_journey[col] / region_journey['avg_total_min'] * 100, 1
    )

print(region_journey[['Region', 'avg_total_min', 
                        'pct_rural_unpaved', 
                        'pct_connecting_road',
                        'pct_major_road',
                        'pct_urban_road',
                        'pct_walking']].sort_values(
    'pct_rural_unpaved', ascending=False).to_string(index=False))

Journey breakdown by road type — regional level...
       Region  avg_total_min  pct_rural_unpaved  pct_connecting_road  pct_major_road  pct_urban_road  pct_walking
    Bono East           43.9               70.8                 22.6             5.2             1.1          0.0
   Upper West           16.7               70.7                 11.4            16.2             1.2          0.0
Western North           16.2               67.9                 23.5             6.2             2.5          0.0
     Savannah           40.3               67.0                 15.9            15.6             1.7          0.0
     Northern           17.1               66.1                 21.6             9.9             2.9          0.0
   North East           15.3               64.1                 13.7            20.3             2.0          0.0
   Upper East           12.4               63.7                 20.2            11.3             4.8          0.0
        Ahafo           12.6         

In [22]:
print(district_scores[['Region', 'District', 
                         'road_pct_rural_unpaved',
                         'road_pct_connecting_road',
                         'road_pct_major_road',
                         'road_pct_urban_road',
                         'road_pct_walking',
                         'road_quality_score']].sort_values(
    'road_pct_rural_unpaved', ascending=False).to_string(index=False))

       Region                     District  road_pct_rural_unpaved  road_pct_connecting_road  road_pct_major_road  road_pct_urban_road  road_pct_walking  road_quality_score
        Volta                 Akatsi North                    88.1                       1.2                  8.6                  2.1               0.0               24.36
     Northern               Tatale Sanguli                    87.3                       9.8                  0.2                  2.7               0.0               21.47
     Northern                Nanumba South                    83.4                       7.6                  7.6                  1.4               0.0               25.83
     Savannah             Sawla-Tuna-Kalba                    82.0                       9.0                  7.4                  1.7               0.0               26.50
          Oti                Nkwanta North                    81.8                       7.5                  8.6                  2.1 

In [23]:
import pandas as pd

district_scores = pd.read_csv(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\district_accessibility_scores.csv")
master_pop = pd.read_csv(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\master_population_data.csv")

# Average journey minutes per road type per district
road_by_district = master_pop.groupby(['Region', 'District']).agg(
    avg_major_road_min=('journey_major_road_min', 'mean'),
    avg_connecting_road_min=('journey_connecting_road_min', 'mean'),
    avg_urban_road_min=('journey_urban_road_min', 'mean'),
    avg_rural_unpaved_min=('journey_rural_unpaved_min', 'mean'),
    avg_walking_min=('journey_walking_min', 'mean'),
    avg_total_min=('journey_total_min', 'mean'),
).round(1).reset_index()

print(f"✅ Shape: {road_by_district.shape}")
print(f"\nSample — worst districts by rural unpaved time:")
print(road_by_district.nlargest(10, 'avg_rural_unpaved_min')[
    ['Region', 'District', 'avg_rural_unpaved_min', 
     'avg_connecting_road_min', 'avg_major_road_min',
     'avg_total_min']
].to_string(index=False))

✅ Shape: (260, 8)

Sample — worst districts by rural unpaved time:
   Region                 District  avg_rural_unpaved_min  avg_connecting_road_min  avg_major_road_min  avg_total_min
 Savannah               East Gonja                   55.8                      8.1                 5.1           70.3
Bono East                Sene East                   50.3                     22.4                 0.2           73.9
Bono East                Sene West                   44.4                     19.5                 1.7           65.6
Bono East           Kintampo North                   39.1                      5.3                 6.8           51.4
     Bono                    Banda                   31.4                     30.5                 5.7           68.1
 Savannah              North Gonja                   28.3                      7.7                 9.5           45.8
 Savannah               West Gonja                   27.4                      5.5                14.9     

In [24]:
regions_of_interest = ['Savannah', 'Upper West', 'Bono East', 'North East', 'Northern']

filtered = road_by_district[road_by_district['Region'].isin(regions_of_interest)]

print(f"Districts in selected regions: {len(filtered)}")
print(f"\n=== AVERAGE JOURNEY TIME BY ROAD TYPE ===")
print(filtered.sort_values(['Region', 'avg_rural_unpaved_min'], ascending=[True, False])[
    ['Region', 'District', 
     'avg_rural_unpaved_min',
     'avg_connecting_road_min', 
     'avg_major_road_min',
     'avg_urban_road_min',
     'avg_walking_min',
     'avg_total_min']
].to_string(index=False))

Districts in selected regions: 51

=== AVERAGE JOURNEY TIME BY ROAD TYPE ===
    Region              District  avg_rural_unpaved_min  avg_connecting_road_min  avg_major_road_min  avg_urban_road_min  avg_walking_min  avg_total_min
 Bono East             Sene East                   50.3                     22.4                 0.2                 1.0              0.0           73.9
 Bono East             Sene West                   44.4                     19.5                 1.7                 0.0              0.0           65.6
 Bono East        Kintampo North                   39.1                      5.3                 6.8                 0.1              0.0           51.4
 Bono East              Pru East                   23.9                      2.3                 4.5                 1.5              0.0           32.2
 Bono East       Atebubu-Amantin                   19.0                      5.1                 1.5                 0.2              0.0           25.8
 Bono

In [25]:
import json
with open(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\region_scores.json") as f:
    data = json.load(f)

for r in data:
    print(f"{r['Region']:20s}  emg_within_30={r['emergency_pct_within_30']}  spec_within_60={r.get('specialist_pct_within_60', 'N/A')}")

Ahafo                 emg_within_30=62.5  spec_within_60=29.7
Ashanti               emg_within_30=55.9  spec_within_60=29.0
Bono                  emg_within_30=53.4  spec_within_60=25.3
Bono East             emg_within_30=30.3  spec_within_60=0.3
Central               emg_within_30=64.9  spec_within_60=55.4
Eastern               emg_within_30=42.9  spec_within_60=26.5
Greater Accra         emg_within_30=88.7  spec_within_60=84.1
North East            emg_within_30=41.1  spec_within_60=8.5
Northern              emg_within_30=42.1  spec_within_60=19.1
Oti                   emg_within_30=43.7  spec_within_60=8.2
Savannah              emg_within_30=15.7  spec_within_60=6.0
Upper East            emg_within_30=58.3  spec_within_60=42.3
Upper West            emg_within_30=26.6  spec_within_60=25.5
Volta                 emg_within_30=66.2  spec_within_60=53.2
Western               emg_within_30=46.6  spec_within_60=17.6
Western North         emg_within_30=51.3  spec_within_60=0.0


In [6]:
import pandas as pd
import numpy as np

print("Loading data...")
master_pop = pd.read_csv(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\master_population_data.csv")
district_scores = pd.read_csv(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\district_accessibility_scores.csv")

print(f"✅ Master population: {master_pop.shape}")
print(f"✅ District scores: {district_scores.shape}")

# Pull Weija Gbawe
weija = master_pop[master_pop['District'] == 'Weija Gbawe']
weija_scores = district_scores[district_scores['District'] == 'Weija Gbawe']

print(f"\nWeija Gbawe population points: {len(weija)}")
print(f"\n=== WEIJA GBAWE TRAVEL TIME STATISTICS ===")
print(f"\nAny facility:")
print(f"  Median: {weija['nearest_any_min'].median():.1f} min")
print(f"  Within 30 min: {(weija['nearest_any_min'] <= 30).mean()*100:.1f}%")

print(f"\nEmergency care:")
print(f"  Median: {weija['nearest_emergency_min'].median():.1f} min")
print(f"  Within 30 min: {(weija['nearest_emergency_min'] <= 30).mean()*100:.1f}%")

print(f"\nSpecialist care:")
print(f"  Median: {weija['nearest_specialist_min'].median():.1f} min")
print(f"  Within 60 min: {(weija['nearest_specialist_min'] <= 60).mean()*100:.1f}%")

print(f"\nJourney breakdown:")
print(f"  Rural unpaved: {weija['journey_rural_unpaved_min'].mean():.1f} min ({weija['journey_rural_unpaved_min'].mean()/weija['journey_total_min'].mean()*100:.1f}%)")
print(f"  Major road: {weija['journey_major_road_min'].mean():.1f} min ({weija['journey_major_road_min'].mean()/weija['journey_total_min'].mean()*100:.1f}%)")
print(f"  Urban road: {weija['journey_urban_road_min'].mean():.1f} min ({weija['journey_urban_road_min'].mean()/weija['journey_total_min'].mean()*100:.1f}%)")
print(f"  Connecting road: {weija['journey_connecting_road_min'].mean():.1f} min ({weija['journey_connecting_road_min'].mean()/weija['journey_total_min'].mean()*100:.1f}%)")

print(f"\nComposite score: {weija_scores['composite_score'].values[0]}")
print(f"Journey access: {weija_scores['journey_access_score'].values[0]}")
print(f"Supply adequacy: {weija_scores['supply_adequacy_score'].values[0]}")
print(f"Category: {weija_scores['category'].values[0]}")

Loading data...
✅ Master population: (278001, 25)
✅ District scores: (260, 42)

Weija Gbawe population points: 186

=== WEIJA GBAWE TRAVEL TIME STATISTICS ===

Any facility:
  Median: 2.4 min
  Within 30 min: 100.0%

Emergency care:
  Median: 6.1 min
  Within 30 min: 100.0%

Specialist care:
  Median: 29.6 min
  Within 60 min: 100.0%

Journey breakdown:
  Rural unpaved: 0.7 min (15.5%)
  Major road: 0.2 min (4.5%)
  Urban road: 2.8 min (64.0%)
  Connecting road: 0.7 min (16.0%)

Composite score: 77.67
Journey access: 100.0
Supply adequacy: 25.56
Category: Decent


In [7]:
print("=== WEIJA GBAWE — MEAN vs MEDIAN ===\n")

cols = {
    'Any facility': 'nearest_any_min',
    'Emergency': 'nearest_emergency_min',
    'Outpatient': 'nearest_outpatient_min',
    'Specialist': 'nearest_specialist_min',
}

for label, col in cols.items():
    mean = weija[col].mean()
    median = weija[col].median()
    print(f"{label}:")
    print(f"  Mean:   {mean:.1f} min")
    print(f"  Median: {median:.1f} min")
    print()

=== WEIJA GBAWE — MEAN vs MEDIAN ===

Any facility:
  Mean:   3.2 min
  Median: 2.4 min

Emergency:
  Mean:   7.5 min
  Median: 6.1 min

Outpatient:
  Mean:   5.0 min
  Median: 4.3 min

Specialist:
  Mean:   30.5 min
  Median: 29.6 min



In [8]:
# Check which columns in our files have median in the name
print("=== COLUMNS WITH MEDIAN IN NAME ===")

print("\ndistrict_accessibility_scores.csv:")
median_cols_district = [c for c in district_scores.columns if 'median' in c.lower()]
print(median_cols_district if median_cols_district else "None!")

print("\nregion_summary.csv:")
region_summary = pd.read_csv(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\region_summary.csv")
median_cols_region = [c for c in region_summary.columns if 'median' in c.lower()]
print(median_cols_region if median_cols_region else "None!")

print("\ntravel_time_summary.csv:")
travel_summary = pd.read_csv(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\travel_time_summary.csv")
median_cols_travel = [c for c in travel_summary.columns if 'median' in c.lower()]
print(median_cols_travel if median_cols_travel else "None!")

=== COLUMNS WITH MEDIAN IN NAME ===

district_accessibility_scores.csv:
['outpatient_median_min']

region_summary.csv:
['emergency_median_min', 'any_median_min', 'specialist_median_min', 'psychiatric_median_min', 'outpatient_median_min']

travel_time_summary.csv:
['Median_travel_time_min']


In [9]:
print("=== GA CENTRAL — MEAN vs MEDIAN ===\n")

ga_central = master_pop[master_pop['District'] == 'Ga Central']
print(f"Population points: {len(ga_central)}")

cols = {
    'Any facility': 'nearest_any_min',
    'Emergency': 'nearest_emergency_min',
    'Outpatient': 'nearest_outpatient_min',
    'Specialist': 'nearest_specialist_min',
}

for label, col in cols.items():
    mean = ga_central[col].mean()
    median = ga_central[col].median()
    print(f"{label}:")
    print(f"  Average: {mean:.1f} min")
    print(f"  Median:  {median:.1f} min")
    print()

=== GA CENTRAL — MEAN vs MEDIAN ===

Population points: 58
Any facility:
  Average: 2.3 min
  Median:  2.0 min

Emergency:
  Average: 7.3 min
  Median:  6.5 min

Outpatient:
  Average: 2.9 min
  Median:  2.9 min

Specialist:
  Average: 25.1 min
  Median:  24.8 min



In [10]:
for district_name in ['East Gonja', 'Kwahu Afram Plains South', 'Ga Central', 'Weija Gbawe']:
    d = master_pop[master_pop['District'] == district_name]
    mean = d['nearest_emergency_min'].mean()
    median = d['nearest_emergency_min'].median()
    print(f"{district_name}:")
    print(f"  Average: {mean:.1f} min")
    print(f"  Median:  {median:.1f} min")
    print()

East Gonja:
  Average: 139.5 min
  Median:  143.1 min

Kwahu Afram Plains South:
  Average: 129.1 min
  Median:  137.2 min

Ga Central:
  Average: 7.3 min
  Median:  6.5 min

Weija Gbawe:
  Average: 7.5 min
  Median:  6.1 min



In [11]:
for district_name in ['East Gonja', 'Kwahu Afram Plains South', 'Ga Central', 'Weija Gbawe']:
    d = master_pop[master_pop['District'] == district_name]
    col = 'nearest_emergency_min'
    print(f"{district_name}:")
    print(f"  Min:     {d[col].min():.1f} min")
    print(f"  Average: {d[col].mean():.1f} min")
    print(f"  Median:  {d[col].median():.1f} min")
    print(f"  Max:     {d[col].max():.1f} min")
    print()
    

East Gonja:
  Min:     46.8 min
  Average: 139.5 min
  Median:  143.1 min
  Max:     198.0 min

Kwahu Afram Plains South:
  Min:     18.0 min
  Average: 129.1 min
  Median:  137.2 min
  Max:     252.3 min

Ga Central:
  Min:     1.4 min
  Average: 7.3 min
  Median:  6.5 min
  Max:     18.0 min

Weija Gbawe:
  Min:     0.7 min
  Average: 7.5 min
  Median:  6.1 min
  Max:     20.3 min



In [12]:
import pandas as pd
import numpy as np

print("=== POPULATION WEIGHTED vs CELL WEIGHTED ===\n")

# Total population
total_pop = master_pop['population'].sum()
total_cells = len(master_pop)

print(f"Total population: {total_pop:,.0f}")
print(f"Total grid cells: {total_cells:,}")
print(f"Average population per cell: {total_pop/total_cells:.1f}")

print(f"\n=== ANY FACILITY WITHIN 30 MIN ===")

# Cell weighted (what we currently report)
cell_weighted = (master_pop['nearest_any_min'] <= 30).sum() / total_cells * 100

# Population weighted (what we SHOULD report)
pop_weighted = master_pop.loc[
    master_pop['nearest_any_min'] <= 30, 'population'
].sum() / total_pop * 100

print(f"Cell weighted:       {cell_weighted:.1f}%")
print(f"Population weighted: {pop_weighted:.1f}%")

print(f"\n=== EMERGENCY WITHIN 30 MIN ===")
cell_weighted_emerg = (master_pop['nearest_emergency_min'] <= 30).sum() / total_cells * 100
pop_weighted_emerg = master_pop.loc[
    master_pop['nearest_emergency_min'] <= 30, 'population'
].sum() / total_pop * 100

print(f"Cell weighted:       {cell_weighted_emerg:.1f}%")
print(f"Population weighted: {pop_weighted_emerg:.1f}%")

print(f"\n=== SPECIALIST WITHIN 30 MIN ===")
cell_weighted_spec = (master_pop['nearest_specialist_min'] <= 30).sum() / total_cells * 100
pop_weighted_spec = master_pop.loc[
    master_pop['nearest_specialist_min'] <= 30, 'population'
].sum() / total_pop * 100

print(f"Cell weighted:       {cell_weighted_spec:.1f}%")
print(f"Population weighted: {pop_weighted_spec:.1f}%")

print(f"\n=== PSYCHIATRIC WITHIN 30 MIN ===")
cell_weighted_psych = (master_pop['nearest_psychiatric_min'] <= 30).sum() / total_cells * 100
pop_weighted_psych = master_pop.loc[
    master_pop['nearest_psychiatric_min'] <= 30, 'population'
].sum() / total_pop * 100

print(f"Cell weighted:       {cell_weighted_psych:.1f}%")
print(f"Population weighted: {pop_weighted_psych:.1f}%")

=== POPULATION WEIGHTED vs CELL WEIGHTED ===

Total population: 30,832,018
Total grid cells: 278,001
Average population per cell: 110.9

=== ANY FACILITY WITHIN 30 MIN ===
Cell weighted:       90.6%
Population weighted: 98.9%

=== EMERGENCY WITHIN 30 MIN ===
Cell weighted:       42.1%
Population weighted: 81.0%

=== SPECIALIST WITHIN 30 MIN ===
Cell weighted:       6.6%
Population weighted: 45.2%

=== PSYCHIATRIC WITHIN 30 MIN ===
Cell weighted:       0.9%
Population weighted: 16.8%


In [13]:
print("Sample population values per grid cell:")
print(master_pop[['lat', 'lon', 'Region', 'District', 'population']].head(10).to_string(index=False))

print(f"\nMin population per cell: {master_pop['population'].min():.2f}")
print(f"Max population per cell: {master_pop['population'].max():.2f}")
print(f"Average population per cell: {master_pop['population'].mean():.1f}")
print(f"Total population: {master_pop['population'].sum():,.0f}")

Sample population values per grid cell:
      lat       lon     Region District  population
11.170417 -0.280417 Upper East    Bawku   53.884277
11.170417 -0.272083 Upper East    Bawku   59.233740
11.170417 -0.263750 Upper East    Bawku   57.277393
11.170417 -0.255417        NaN      NaN   54.462917
11.170417 -0.247083        NaN      NaN   55.727630
11.162083 -0.280417 Upper East    Bawku   69.824730
11.162083 -0.272083 Upper East    Bawku   91.809390
11.162083 -0.263750 Upper East    Bawku   75.181460
11.162083 -0.255417 Upper East    Bawku   63.525470
11.162083 -0.247083 Upper East    Bawku   68.440530

Min population per cell: 0.15
Max population per cell: 20756.96
Average population per cell: 110.9
Total population: 30,832,018


In [ ]:
ERRORS IDENTIFIED IN INITAL REASONING OF POPULATION AND PERCENTAGE BANDS. RECOMPUTING ....

In [14]:
import pandas as pd
import numpy as np

print("Loading data...")
master_pop = pd.read_csv(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\master_population_data.csv")
print(f"✅ {len(master_pop):,} population points loaded")

# ── BAND SCORE FUNCTIONS (population weighted) ──

def emergency_band_score_pop(group):
    """Population weighted emergency band score"""
    total_pop = group['population'].sum()
    if total_pop == 0:
        return 0
    col = 'nearest_emergency_min'
    score = (
        group.loc[group[col] <= 30, 'population'].sum() * 100 +
        group.loc[(group[col] > 30) & (group[col] <= 60), 'population'].sum() * 70 +
        group.loc[(group[col] > 60) & (group[col] <= 90), 'population'].sum() * 40 +
        group.loc[(group[col] > 90) & (group[col] <= 120), 'population'].sum() * 15 +
        group.loc[group[col] > 120, 'population'].sum() * 0
    ) / total_pop
    return round(score, 2)

def any_band_score_pop(group):
    """Population weighted any facility band score"""
    total_pop = group['population'].sum()
    if total_pop == 0:
        return 0
    col = 'nearest_any_min'
    score = (
        group.loc[group[col] <= 30, 'population'].sum() * 100 +
        group.loc[(group[col] > 30) & (group[col] <= 60), 'population'].sum() * 70 +
        group.loc[(group[col] > 60) & (group[col] <= 90), 'population'].sum() * 40 +
        group.loc[(group[col] > 90) & (group[col] <= 120), 'population'].sum() * 15 +
        group.loc[group[col] > 120, 'population'].sum() * 0
    ) / total_pop
    return round(score, 2)

def outpatient_band_score_pop(group):
    """Population weighted outpatient band score"""
    total_pop = group['population'].sum()
    if total_pop == 0:
        return 0
    col = 'nearest_outpatient_min'
    score = (
        group.loc[group[col] <= 30, 'population'].sum() * 100 +
        group.loc[(group[col] > 30) & (group[col] <= 60), 'population'].sum() * 70 +
        group.loc[(group[col] > 60) & (group[col] <= 90), 'population'].sum() * 40 +
        group.loc[(group[col] > 90) & (group[col] <= 120), 'population'].sum() * 15 +
        group.loc[group[col] > 120, 'population'].sum() * 0
    ) / total_pop
    return round(score, 2)

def specialist_band_score_pop(group):
    """Population weighted specialist band score — wider bands"""
    total_pop = group['population'].sum()
    if total_pop == 0:
        return 0
    col = 'nearest_specialist_min'
    score = (
        group.loc[group[col] <= 60, 'population'].sum() * 100 +
        group.loc[(group[col] > 60) & (group[col] <= 120), 'population'].sum() * 70 +
        group.loc[(group[col] > 120) & (group[col] <= 180), 'population'].sum() * 40 +
        group.loc[group[col] > 180, 'population'].sum() * 0
    ) / total_pop
    return round(score, 2)

def pop_pct(group, col, lower, upper=None):
    """Population weighted percentage in a time band"""
    total_pop = group['population'].sum()
    if total_pop == 0:
        return 0
    if upper is None:
        mask = group[col] > lower
    elif lower == 0:
        mask = group[col] <= upper
    else:
        mask = (group[col] > lower) & (group[col] <= upper)
    return round(group.loc[mask, 'population'].sum() / total_pop * 100, 1)

# ── BUILD DISTRICT SCORES ──
print("\nBuilding population weighted district scores...")
rows = []
total = master_pop.groupby(['Region', 'District']).ngroups
count = 0

for (region, district), group in master_pop.groupby(['Region', 'District']):
    if pd.isna(region) or pd.isna(district):
        continue
    count += 1
    if count % 50 == 0:
        print(f"  {count}/{total} districts...")

    total_pop = group['population'].sum()

    row = {
        'Region': region,
        'District': district,
        'total_population': round(total_pop),
        'population_points': len(group),

        # Emergency bands
        'emergency_pct_within_30': pop_pct(group, 'nearest_emergency_min', 0, 30),
        'emergency_pct_30_60':     pop_pct(group, 'nearest_emergency_min', 30, 60),
        'emergency_pct_60_90':     pop_pct(group, 'nearest_emergency_min', 60, 90),
        'emergency_pct_90_120':    pop_pct(group, 'nearest_emergency_min', 90, 120),
        'emergency_pct_120plus':   pop_pct(group, 'nearest_emergency_min', 120),
        'emergency_band_score':    emergency_band_score_pop(group),

        # Any facility bands
        'any_pct_within_30': pop_pct(group, 'nearest_any_min', 0, 30),
        'any_pct_30_60':     pop_pct(group, 'nearest_any_min', 30, 60),
        'any_pct_60_90':     pop_pct(group, 'nearest_any_min', 60, 90),
        'any_pct_90_120':    pop_pct(group, 'nearest_any_min', 90, 120),
        'any_pct_120plus':   pop_pct(group, 'nearest_any_min', 120),
        'any_band_score':    any_band_score_pop(group),

        # Specialist bands
        'specialist_pct_within_60':  pop_pct(group, 'nearest_specialist_min', 0, 60),
        'specialist_pct_60_120':     pop_pct(group, 'nearest_specialist_min', 60, 120),
        'specialist_pct_120_180':    pop_pct(group, 'nearest_specialist_min', 120, 180),
        'specialist_pct_180plus':    pop_pct(group, 'nearest_specialist_min', 180),
        'specialist_band_score_wide': specialist_band_score_pop(group),

        # Outpatient bands
        'outpatient_pct_within_30': pop_pct(group, 'nearest_outpatient_min', 0, 30),
        'outpatient_pct_30_60':     pop_pct(group, 'nearest_outpatient_min', 30, 60),
        'outpatient_pct_60_90':     pop_pct(group, 'nearest_outpatient_min', 60, 90),
        'outpatient_pct_90_120':    pop_pct(group, 'nearest_outpatient_min', 90, 120),
        'outpatient_pct_120plus':   pop_pct(group, 'nearest_outpatient_min', 120),
        'outpatient_band_score':    outpatient_band_score_pop(group),

        # Road breakdown
        'road_pct_major_road':      round(group['journey_major_road_min'].mean() / group['journey_total_min'].mean() * 100, 1) if group['journey_total_min'].mean() > 0 else 0,
        'road_pct_urban_road':      round(group['journey_urban_road_min'].mean() / group['journey_total_min'].mean() * 100, 1) if group['journey_total_min'].mean() > 0 else 0,
        'road_pct_connecting_road': round(group['journey_connecting_road_min'].mean() / group['journey_total_min'].mean() * 100, 1) if group['journey_total_min'].mean() > 0 else 0,
        'road_pct_rural_unpaved':   round(group['journey_rural_unpaved_min'].mean() / group['journey_total_min'].mean() * 100, 1) if group['journey_total_min'].mean() > 0 else 0,
        'road_pct_walking':         round(group['journey_walking_min'].mean() / group['journey_total_min'].mean() * 100, 1) if group['journey_total_min'].mean() > 0 else 0,

        # Road quality score
        'road_quality_score': round(
            (group['journey_major_road_min'].mean() * 100 +
             group['journey_urban_road_min'].mean() * 85 +
             group['journey_connecting_road_min'].mean() * 60 +
             group['journey_rural_unpaved_min'].mean() * 15 +
             group['journey_walking_min'].mean() * 0) /
            group['journey_total_min'].mean(), 2
        ) if group['journey_total_min'].mean() > 0 else 0,

        # Travel time stats
        'emergency_mean_min':   round(group['nearest_emergency_min'].mean(), 1),
        'emergency_median_min': round(group['nearest_emergency_min'].median(), 1),
        'emergency_min_min':    round(group['nearest_emergency_min'].min(), 1),
        'emergency_max_min':    round(group['nearest_emergency_min'].max(), 1),

        'any_mean_min':   round(group['nearest_any_min'].mean(), 1),
        'any_min_min':    round(group['nearest_any_min'].min(), 1),
        'any_max_min':    round(group['nearest_any_min'].max(), 1),

        'specialist_mean_min':   round(group['nearest_specialist_min'].mean(), 1),
        'specialist_min_min':    round(group['nearest_specialist_min'].min(), 1),
        'specialist_max_min':    round(group['nearest_specialist_min'].max(), 1),

        'outpatient_mean_min':   round(group['nearest_outpatient_min'].mean(), 1),
        'outpatient_min_min':    round(group['nearest_outpatient_min'].min(), 1),
        'outpatient_max_min':    round(group['nearest_outpatient_min'].max(), 1),

        # E2SFCA
        'avg_accessibility_score': round(group['accessibility_score'].mean(), 6),
    }
    rows.append(row)

district_scores = pd.DataFrame(rows)
print(f"✅ {len(district_scores)} districts computed")

# ── NORMALIZE E2SFCA ──
e_min = district_scores['avg_accessibility_score'].min()
e_max = district_scores['avg_accessibility_score'].max()
district_scores['e2sfca_normalized'] = round(
    (district_scores['avg_accessibility_score'] - e_min) /
    (e_max - e_min) * 100, 2
)

# ── COMPOSITE SCORES ──
district_scores['journey_access_score'] = round(
    (district_scores['emergency_band_score']       * 0.50) +
    (district_scores['specialist_band_score_wide'] * 0.35) +
    (district_scores['any_band_score']             * 0.15),
    2
)

district_scores['supply_adequacy_score'] = district_scores['e2sfca_normalized']

district_scores['composite_score'] = round(
    (district_scores['journey_access_score']  * 0.70) +
    (district_scores['supply_adequacy_score'] * 0.30),
    2
)

# ── CATEGORIES ──
def categorize(score):
    if score >= 80:   return 'Thriving'
    elif score >= 60: return 'Decent'
    elif score >= 40: return 'Getting By'
    elif score >= 20: return 'Struggling'
    else:             return 'Dire'

district_scores['category'] = district_scores['composite_score'].apply(categorize)

# ── SAVE ──
out_path = r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\district_accessibility_scores.csv"
district_scores.to_csv(out_path, index=False)
print(f"✅ Saved! {district_scores.shape}")

# ── RESULTS ──
print(f"\n=== CATEGORY DISTRIBUTION ===")
print(district_scores['category'].value_counts())

print(f"\n=== TOP 10 ===")
print(district_scores.nlargest(10, 'composite_score')[
    ['District', 'Region', 'journey_access_score',
     'supply_adequacy_score', 'composite_score', 'category']
].to_string(index=False))

print(f"\n=== BOTTOM 10 ===")
print(district_scores.nsmallest(10, 'composite_score')[
    ['District', 'Region', 'journey_access_score',
     'supply_adequacy_score', 'composite_score', 'category']
].to_string(index=False))

# ── NATIONAL HEADLINE NUMBERS ──
print(f"\n=== NATIONAL POPULATION WEIGHTED HEADLINE NUMBERS ===")
total_pop = master_pop['population'].sum()
for label, col, threshold in [
    ('Any facility within 30 min', 'nearest_any_min', 30),
    ('Emergency within 30 min', 'nearest_emergency_min', 30),
    ('Outpatient within 30 min', 'nearest_outpatient_min', 30),
    ('Specialist within 60 min', 'nearest_specialist_min', 60),
]:
    pct = master_pop.loc[master_pop[col] <= threshold, 'population'].sum() / total_pop * 100
    print(f"  {label}: {pct:.1f}%")

Loading data...
✅ 278,001 population points loaded

Building population weighted district scores...
  50/260 districts...
  100/260 districts...
  150/260 districts...
  200/260 districts...
  250/260 districts...
✅ 260 districts computed
✅ Saved! (260, 52)

=== CATEGORY DISTRIBUTION ===
category
Decent        170
Getting By     53
Thriving       29
Struggling      6
Dire            2
Name: count, dtype: int64

=== TOP 10 ===
     District     Region  journey_access_score  supply_adequacy_score  composite_score category
   Bolga East Upper East                100.00                 100.00           100.00 Thriving
 Wa Municipal Upper West                 99.75                  90.82            97.07 Thriving
Nadowli-Kaleo Upper West                 98.15                  83.47            93.75 Thriving
       Nabdam Upper East                 99.60                  73.45            91.75 Thriving
   Bolgatanga Upper East                 99.45                  70.48            90.76 Thr

In [15]:
# Check what % cannot reach each facility type within 30 min
# Population weighted
total_pop = master_pop['population'].sum()

print("=== % CANNOT REACH WITHIN 30 MIN (population weighted) ===\n")
for label, col in [
    ('Any facility (incl CHPS)', 'nearest_any_min'),
    ('CHPS only', 'nearest_chps_min'),
    ('Outpatient (clinic+hospital)', 'nearest_outpatient_min'),
    ('Emergency (hospital)', 'nearest_emergency_min'),
]:
    pct_within = master_pop.loc[master_pop[col] <= 30, 'population'].sum() / total_pop * 100
    pct_outside = 100 - pct_within
    print(f"{label}:")
    print(f"  Within 30 min: {pct_within:.1f}%")
    print(f"  Beyond 30 min: {pct_outside:.1f}%")
    print()

=== % CANNOT REACH WITHIN 30 MIN (population weighted) ===

Any facility (incl CHPS):
  Within 30 min: 98.9%
  Beyond 30 min: 1.1%

CHPS only:
  Within 30 min: 98.7%
  Beyond 30 min: 1.3%

Outpatient (clinic+hospital):
  Within 30 min: 94.3%
  Beyond 30 min: 5.7%

Emergency (hospital):
  Within 30 min: 81.0%
  Beyond 30 min: 19.0%



In [16]:
# Check if there are suspiciously short travel times
print("Distribution of travel times to any facility:\n")

bins = [0, 5, 10, 15, 20, 30, 60, 120, 999]
labels = ['0-5', '5-10', '10-15', '15-20', '20-30', '30-60', '60-120', '120+']

master_pop['time_bin'] = pd.cut(
    master_pop['nearest_any_min'], 
    bins=bins, 
    labels=labels
)

bin_counts = master_pop.groupby('time_bin').agg(
    cells=('population', 'count'),
    population=('population', 'sum')
).reset_index()

bin_counts['pct_cells'] = round(bin_counts['cells'] / len(master_pop) * 100, 1)
bin_counts['pct_population'] = round(bin_counts['population'] / master_pop['population'].sum() * 100, 1)

print(bin_counts.to_string(index=False))

Distribution of travel times to any facility:

time_bin  cells   population  pct_cells  pct_population
     0-5 107851 2.306239e+07       38.8            74.8
    5-10  64751 4.653601e+06       23.3            15.1
   10-15  35814 1.499228e+06       12.9             4.9
   15-20  19521 6.472970e+05        7.0             2.1
   20-30  20717 5.034802e+05        7.5             1.6
   30-60  19413 2.841184e+05        7.0             0.9
  60-120   5561 5.455654e+04        2.0             0.2
    120+   1093 8.695043e+03        0.4             0.0


C:\Users\hp\AppData\Local\Temp\ipykernel_19828\4265537286.py:13: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bin_counts = master_pop.groupby('time_bin').agg(


In [17]:
# Check facility types and their distribution
master_dataset = pd.read_csv(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\master_dataset_v3.csv")

print("Facility type counts:")
print(master_dataset['Facility_Type'].value_counts())

print(f"\nTotal facilities: {len(master_dataset)}")
print(f"\nFacilities per region:")
print(master_dataset['Region'].value_counts())

Facility type counts:
Facility_Type
CHPS                          6733
HEALTH CENTRE                 1215
CLINIC                         929
HOSPITAL                       581
MATERNITY HOME                 248
DISTRICT HOSPITAL              145
POLYCLINIC                      93
TEACHING HOSPITAL               10
REGIONAL HOSPITAL               10
UNIVERSITY HOSPITAL/CLINIC       8
PSYCHIATRIC HOSPITAL             5
LEPROSARIUM                      1
Name: count, dtype: int64

Total facilities: 9978

Facilities per region:
Region
ASHANTI          1645
GREATER ACCRA    1468
EASTERN          1125
CENTRAL           773
UPPER EAST        673
WESTERN           659
NORTHERN          562
VOLTA             527
UPPER WEST        513
BONO              478
BONO EAST         408
WESTERN NORTH     324
OTI               247
SAVANNAH          241
AHAFO             197
NORTH EAST        138
Name: count, dtype: int64


In [18]:
total_pop = master_pop['population'].sum()

print("=== WITHIN 30 MIN BY FACILITY TYPE ===\n")
for label, col in [
    ('Any (incl CHPS)', 'nearest_any_min'),
    ('Outpatient (clinic+above)', 'nearest_outpatient_min'),
    ('Emergency (hospital)', 'nearest_emergency_min'),
    ('Specialist', 'nearest_specialist_min'),
]:
    within = master_pop.loc[
        master_pop[col] <= 30, 'population'
    ].sum() / total_pop * 100
    beyond = 100 - within
    print(f"{label}:")
    print(f"  Within 30 min: {within:.1f}%")
    print(f"  Beyond 30 min: {beyond:.1f}%")
    print()

=== WITHIN 30 MIN BY FACILITY TYPE ===

Any (incl CHPS):
  Within 30 min: 98.9%
  Beyond 30 min: 1.1%

Outpatient (clinic+above):
  Within 30 min: 94.3%
  Beyond 30 min: 5.7%

Emergency (hospital):
  Within 30 min: 81.0%
  Beyond 30 min: 19.0%

Specialist:
  Within 30 min: 45.2%
  Beyond 30 min: 54.8%



In [24]:
import requests
import numpy as np

print("Testing OSRM routing on suspicious short-time population points...\n")

# Get population points with very short travel times (under 2 minutes)
suspicious = master_pop[master_pop['nearest_any_min'] < 2].head(10)

print(f"Population points with travel time under 2 minutes: {len(master_pop[master_pop['nearest_any_min'] < 2]):,}")
print(f"\nTesting 10 of them against OSRM directly...\n")

for _, row in suspicious.iterrows():
    # Call OSRM directly
    url = (f"http://localhost:5000/route/v1/driving/"
           f"{row['lon']},{row['lat']};"
           f"{row['nearest_fac_lon']},{row['nearest_fac_lat']}"
           f"?overview=false")
    
    try:
        response = requests.get(url, timeout=5)
        data = response.json()
        
        if data['code'] == 'Ok':
            osrm_time = round(data['routes'][0]['duration'] / 60, 2)
            distance_km = round(data['routes'][0]['distance'] / 1000, 2)
        else:
            osrm_time = None
            distance_km = None
            
        print(f"Pop point: ({row['lat']:.3f}, {row['lon']:.3f})")
        print(f"  Facility: {row['nearest_fac_name']} ({row['nearest_fac_type']})")
        print(f"  Stored time: {round(row['nearest_any_min'], 2)} min")
        print(f"  OSRM live time: {osrm_time} min")
        print(f"  Distance: {distance_km} km")
        print()
        
    except Exception as e:
        print(f"  OSRM error: {e}")

Testing OSRM routing on suspicious short-time population points...

Population points with travel time under 2 minutes: 46,684

Testing 10 of them against OSRM directly...

Pop point: (11.154, -0.205)
  Facility: SALIFU MEMORIAL CLINIC (CLINIC)
  Stored time: 0.98 min
  OSRM live time: 0.98 min
  Distance: 0.55 km

Pop point: (11.154, -0.197)
  Facility: SALIFU MEMORIAL CLINIC (CLINIC)
  Stored time: 1.82 min
  OSRM live time: 1.82 min
  Distance: 0.76 km

Pop point: (11.145, -0.214)
  Facility: DIVINE FAVOUR HEALTH CENTER (HEALTH CENTRE)
  Stored time: 0.74 min
  OSRM live time: 0.74 min
  Distance: 0.31 km

Pop point: (11.145, -0.205)
  Facility: ZION LOVE LIFE CLINIC (CLINIC)
  Stored time: 1.04 min
  OSRM live time: 1.22 min
  Distance: 0.47 km

Pop point: (11.137, -0.255)
  Facility: BADOR CHPS (CHPS)
  Stored time: 1.14 min
  OSRM live time: 1.14 min
  Distance: 0.48 km

Pop point: (11.137, -0.247)
  Facility: BADOR CHPS (CHPS)
  Stored time: 0.18 min
  OSRM live time: 0.18 min
 

In [23]:
import requests

try:
    r = requests.get("http://localhost:5000/", timeout=5)
    print("✅ OSRM is running!")
except:
    print("❌ OSRM not running — need to start the container")

✅ OSRM is running!


In [25]:
import requests
import numpy as np
from scipy.spatial import cKDTree

print("Testing K=10 vs K=20 vs K=50 on sample points...\n")

# Load facilities
master_dataset = pd.read_csv(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\master_dataset_v3.csv")

fac_coords = master_dataset[['Longitude', 'Latitude']].values
fac_tree = cKDTree(fac_coords)

# Sample 200 population points randomly
sample = master_pop.sample(200, random_state=42)
pop_coords = sample[['lon', 'lat']].values

results = []

for i, (_, row) in enumerate(sample.iterrows()):
    if i % 50 == 0:
        print(f"  {i}/200 tested...")
    
    pop_coord = [[row['lon'], row['lat']]]
    times = {}
    
    for k in [10, 20, 50]:
        _, idxs = fac_tree.query(pop_coord, k=k)
        best_time = float('inf')
        
        for idx in idxs[0]:
            fac_lon = fac_coords[idx, 0]
            fac_lat = fac_coords[idx, 1]
            
            url = (f"http://localhost:5000/route/v1/driving/"
                   f"{row['lon']},{row['lat']};"
                   f"{fac_lon},{fac_lat}?overview=false")
            
            try:
                r = requests.get(url, timeout=5)
                d = r.json()
                if d['code'] == 'Ok':
                    t = d['routes'][0]['duration'] / 60
                    if t < best_time:
                        best_time = t
            except:
                pass
        
        times[f'k{k}'] = round(best_time, 2)
    
    results.append(times)

results_df = pd.DataFrame(results)

print(f"\n=== K SENSITIVITY RESULTS ===")
print(f"\nAverage travel time:")
print(f"  K=10: {results_df['k10'].mean():.2f} min")
print(f"  K=20: {results_df['k20'].mean():.2f} min")
print(f"  K=50: {results_df['k50'].mean():.2f} min")

print(f"\nCases where K=20 found shorter time than K=10:")
diff_20 = (results_df['k20'] < results_df['k10']).sum()
print(f"  {diff_20} out of 200 ({diff_20/2:.1f}%)")

print(f"\nCases where K=50 found shorter time than K=10:")
diff_50 = (results_df['k50'] < results_df['k10']).sum()
print(f"  {diff_50} out of 200 ({diff_50/2:.1f}%)")

print(f"\nMax difference K=10 vs K=50:")
print(f"  {(results_df['k10'] - results_df['k50']).max():.2f} min")

print(f"\nSample of cases with biggest differences:")
results_df['diff'] = results_df['k10'] - results_df['k50']
print(results_df.nlargest(10, 'diff').to_string(index=False))

Testing K=10 vs K=20 vs K=50 on sample points...

  0/200 tested...
  50/200 tested...
  100/200 tested...
  150/200 tested...

=== K SENSITIVITY RESULTS ===

Average travel time:
  K=10: 14.07 min
  K=20: 13.22 min
  K=50: 13.19 min

Cases where K=20 found shorter time than K=10:
  7 out of 200 (3.5%)

Cases where K=50 found shorter time than K=10:
  7 out of 200 (3.5%)

Max difference K=10 vs K=50:
  74.88 min

Sample of cases with biggest differences:
   k10    k20    k50  diff
116.67  41.79  41.79 74.88
123.19  71.63  71.63 51.56
106.78  82.94  76.87 29.91
116.17 108.61 108.61  7.56
113.74 106.19 106.19  7.55
 44.07  41.21  41.21  2.86
 18.72  16.80  16.80  1.92
 16.58  16.58  16.58  0.00
  1.44   1.44   1.44  0.00
 10.80  10.80  10.80  0.00


In [26]:
import requests
import time
import numpy as np

print("Testing OSRM Table API speed...\n")

# Take a sample of 100 population points
sample_pop = master_pop.sample(100, random_state=42)

# Take all emergency facilities (726) as destinations
master_dataset = pd.read_csv(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\master_dataset_v3.csv")

emergency_facs = master_dataset[master_dataset['Facility_Type'].isin([
    'Hospital', 'District Hospital', 'Polyclinic', 
    'Regional Hospital', 'Teaching Hospital', 
    'University Hospital'
])][['Longitude', 'Latitude']].values

print(f"Sample population points: {len(sample_pop)}")
print(f"Emergency facilities: {len(emergency_facs)}")

# Build OSRM Table API request
# Format: coordinates are origins first then destinations
# sources = indices of population points
# destinations = indices of facilities

origins = sample_pop[['lon', 'lat']].values
all_coords = np.vstack([origins, emergency_facs])

# Build coordinate string
coord_str = ";".join([f"{lon},{lat}" for lon, lat in all_coords])

# Source indices = 0 to 99 (population points)
sources = ";".join([str(i) for i in range(len(origins))])

# Destination indices = 100 onwards (facilities)
destinations = ";".join([str(i) for i in range(len(origins), len(all_coords))])

url = f"http://localhost:5000/table/v1/driving/{coord_str}?sources={sources}&destinations={destinations}&annotations=duration"

print(f"\nSending request to OSRM Table API...")
print(f"Matrix size: {len(origins)} × {len(emergency_facs)}")

start = time.time()
response = requests.get(url, timeout=120)
elapsed = time.time() - start

print(f"Response received in {elapsed:.1f} seconds")
print(f"Status code: {response.status_code}")

if response.status_code == 200:
    data = response.json()
    if data['code'] == 'Ok':
        # Get minimum time per population point
        durations = np.array(data['durations'])
        min_times = np.nanmin(durations, axis=1) / 60  # convert to minutes
        
        print(f"\n✅ Table API working!")
        print(f"Matrix shape: {durations.shape}")
        print(f"\nSample minimum times (minutes):")
        print(f"  Mean: {min_times.mean():.1f}")
        print(f"  Min: {min_times.min():.1f}")
        print(f"  Max: {min_times.max():.1f}")
        
        # Estimate full computation time
        # Full run: 278,001 population points in batches of 100
        batches = 278001 / 100
        estimated_total = (elapsed * batches) / 60
        print(f"\n=== TIME ESTIMATE ===")
        print(f"Time for 100 points: {elapsed:.1f} seconds")
        print(f"Total batches needed: {batches:.0f}")
        print(f"Estimated total time: {estimated_total:.1f} minutes ({estimated_total/60:.1f} hours)")
    else:
        print(f"❌ OSRM error: {data['code']}")
else:
    print(f"❌ HTTP error: {response.status_code}")
    print(response.text[:500])

Testing OSRM Table API speed...

Sample population points: 100
Emergency facilities: 0

Sending request to OSRM Table API...
Matrix size: 100 × 0
Response received in 0.0 seconds
Status code: 400
❌ HTTP error: 400
{"message":"Query string malformed close to position 4075","code":"InvalidQuery"}


In [27]:
print("Facility types in master dataset:")
print(master_dataset['Facility_Type'].value_counts())

Facility types in master dataset:
Facility_Type
CHPS                          6733
HEALTH CENTRE                 1215
CLINIC                         929
HOSPITAL                       581
MATERNITY HOME                 248
DISTRICT HOSPITAL              145
POLYCLINIC                      93
TEACHING HOSPITAL               10
REGIONAL HOSPITAL               10
UNIVERSITY HOSPITAL/CLINIC       8
PSYCHIATRIC HOSPITAL             5
LEPROSARIUM                      1
Name: count, dtype: int64


In [28]:
import requests
import time
import numpy as np

print("Testing OSRM Table API speed...\n")

# Fix facility type names — all uppercase
emergency_facs = master_dataset[master_dataset['Facility_Type'].isin([
    'HOSPITAL', 'DISTRICT HOSPITAL', 'POLYCLINIC',
    'REGIONAL HOSPITAL', 'TEACHING HOSPITAL',
    'UNIVERSITY HOSPITAL/CLINIC'
])][['Longitude', 'Latitude']].values

print(f"Emergency facilities: {len(emergency_facs)}")

# Use smaller batch to avoid URL length issues
# 50 population points at a time
sample_pop = master_pop.sample(50, random_state=42)
origins = sample_pop[['lon', 'lat']].values

print(f"Sample population points: {len(origins)}")
print(f"Matrix size: {len(origins)} × {len(emergency_facs)}")

# Build coordinates — origins first then destinations
all_coords = np.vstack([origins, emergency_facs])
coord_str = ";".join([f"{lon},{lat}" for lon, lat in all_coords])
sources = ";".join([str(i) for i in range(len(origins))])
destinations = ";".join([str(i) for i in range(len(origins), len(all_coords))])

url = (f"http://localhost:5000/table/v1/driving/{coord_str}"
       f"?sources={sources}&destinations={destinations}&annotations=duration")

print(f"\nURL length: {len(url):,} characters")
print(f"Sending request...")

start = time.time()
response = requests.get(url, timeout=120)
elapsed = time.time() - start

print(f"Response in {elapsed:.1f} seconds — Status: {response.status_code}")

if response.status_code == 200:
    data = response.json()
    if data['code'] == 'Ok':
        durations = np.array(data['durations'])
        min_times = np.nanmin(durations, axis=1) / 60

        print(f"\n✅ Table API working!")
        print(f"Matrix shape: {durations.shape}")
        print(f"\nSample min times:")
        print(f"  Mean: {min_times.mean():.1f} min")
        print(f"  Min:  {min_times.min():.1f} min")
        print(f"  Max:  {min_times.max():.1f} min")

        # Time estimate
        batches = 278001 / 50
        estimated_mins = (elapsed * batches) / 60
        print(f"\n=== TIME ESTIMATE ===")
        print(f"Time for 50 points vs {len(emergency_facs)} facilities: {elapsed:.1f} sec")
        print(f"Total batches: {batches:.0f}")
        print(f"Estimated total: {estimated_mins:.0f} minutes ({estimated_mins/60:.1f} hours)")
    else:
        print(f"❌ OSRM error: {data}")
else:
    print(f"❌ Error: {response.status_code}")
    print(response.text[:300])

Testing OSRM Table API speed...

Emergency facilities: 847
Sample population points: 50
Matrix size: 50 × 847

URL length: 23,126 characters
Sending request...
Response in 0.1 seconds — Status: 400
❌ Error: 400
{"message":"Too many table coordinates","code":"TooBig"}


In [29]:
import requests
import time
import numpy as np

print("Testing with smaller batch...\n")

# Try just 20 population points vs all emergency facilities
sample_pop = master_pop.sample(20, random_state=42)
origins = sample_pop[['lon', 'lat']].values

print(f"Population points: {len(origins)}")
print(f"Emergency facilities: {len(emergency_facs)}")

all_coords = np.vstack([origins, emergency_facs])
coord_str = ";".join([f"{lon},{lat}" for lon, lat in all_coords])
sources = ";".join([str(i) for i in range(len(origins))])
destinations = ";".join([str(i) for i in range(len(origins), len(all_coords))])

url = (f"http://localhost:5000/table/v1/driving/{coord_str}"
       f"?sources={sources}&destinations={destinations}&annotations=duration")

print(f"URL length: {len(url):,} characters")
print(f"Sending...")

start = time.time()
response = requests.get(url, timeout=120)
elapsed = time.time() - start

print(f"Response in {elapsed:.2f} seconds — Status: {response.status_code}")

if response.status_code == 200:
    data = response.json()
    if data['code'] == 'Ok':
        durations = np.array(data['durations'])
        min_times = np.nanmin(durations, axis=1) / 60
        print(f"\n✅ Working!")
        print(f"Matrix shape: {durations.shape}")
        print(f"Mean min time: {min_times.mean():.1f} min")

        # Time estimate
        batches = 278001 / 20
        estimated_mins = (elapsed * batches) / 60
        print(f"\n=== TIME ESTIMATE ===")
        print(f"Time for 20 points: {elapsed:.2f} sec")
        print(f"Total batches: {batches:.0f}")
        print(f"Estimated total: {estimated_mins:.0f} min ({estimated_mins/60:.1f} hours)")
    else:
        print(f"❌ {data}")
else:
    print(f"❌ {response.status_code}: {response.text[:200]}")

Testing with smaller batch...

Population points: 20
Emergency facilities: 847
URL length: 21,886 characters
Sending...
Response in 0.06 seconds — Status: 400
❌ 400: {"message":"Too many table coordinates","code":"TooBig"}


In [30]:
import requests
import time
import numpy as np

print("Finding OSRM Table API coordinate limit...\n")

sample_pop = master_pop.sample(10, random_state=42)
origins = sample_pop[['lon', 'lat']].values

# Try different numbers of facilities
for n_facs in [100, 200, 300, 400, 500]:
    fac_sample = emergency_facs[:n_facs]
    all_coords = np.vstack([origins, fac_sample])
    coord_str = ";".join([f"{lon},{lat}" for lon, lat in all_coords])
    sources = ";".join([str(i) for i in range(len(origins))])
    destinations = ";".join([str(i) for i in range(len(origins), len(all_coords))])
    
    url = (f"http://localhost:5000/table/v1/driving/{coord_str}"
           f"?sources={sources}&destinations={destinations}&annotations=duration")
    
    start = time.time()
    response = requests.get(url, timeout=30)
    elapsed = time.time() - start
    
    status = "✅ OK" if response.status_code == 200 else "❌ FAIL"
    print(f"10 points vs {n_facs} facilities — {status} — {elapsed:.2f}s — URL: {len(url):,} chars")

Finding OSRM Table API coordinate limit...

10 points vs 100 facilities — ✅ OK — 0.38s — URL: 2,850 chars
10 points vs 200 facilities — ✅ OK — 0.48s — URL: 5,328 chars
10 points vs 300 facilities — ✅ OK — 0.71s — URL: 7,810 chars
10 points vs 400 facilities — ✅ OK — 0.99s — URL: 10,291 chars
10 points vs 500 facilities — ✅ OK — 1.14s — URL: 12,765 chars


In [31]:
print("Pushing higher...\n")

for n_facs in [600, 700, 800, 847, 900]:
    fac_sample = emergency_facs[:min(n_facs, len(emergency_facs))]
    all_coords = np.vstack([origins, fac_sample])
    coord_str = ";".join([f"{lon},{lat}" for lon, lat in all_coords])
    sources = ";".join([str(i) for i in range(len(origins))])
    destinations = ";".join([str(i) for i in range(len(origins), len(all_coords))])
    
    url = (f"http://localhost:5000/table/v1/driving/{coord_str}"
           f"?sources={sources}&destinations={destinations}&annotations=duration")
    
    start = time.time()
    response = requests.get(url, timeout=30)
    elapsed = time.time() - start
    
    status = "✅ OK" if response.status_code == 200 else "❌ FAIL"
    print(f"10 points vs {n_facs} facilities — {status} — {elapsed:.2f}s — URL: {len(url):,} chars")

Pushing higher...

10 points vs 600 facilities — ✅ OK — 1.36s — URL: 15,247 chars
10 points vs 700 facilities — ✅ OK — 1.41s — URL: 17,743 chars
10 points vs 800 facilities — ✅ OK — 1.72s — URL: 20,309 chars
10 points vs 847 facilities — ✅ OK — 1.86s — URL: 21,469 chars
10 points vs 900 facilities — ✅ OK — 1.90s — URL: 21,469 chars


In [32]:
print("Finding optimal population point batch size...\n")

# Use all 847 emergency facilities
for n_pop in [10, 20, 30, 40, 50, 75, 100]:
    pop_sample = master_pop.sample(n_pop, random_state=42)
    origins = pop_sample[['lon', 'lat']].values
    
    all_coords = np.vstack([origins, emergency_facs])
    coord_str = ";".join([f"{lon},{lat}" for lon, lat in all_coords])
    sources = ";".join([str(i) for i in range(len(origins))])
    destinations = ";".join([str(i) for i in range(len(origins), len(all_coords))])
    
    url = (f"http://localhost:5000/table/v1/driving/{coord_str}"
           f"?sources={sources}&destinations={destinations}&annotations=duration")
    
    start = time.time()
    response = requests.get(url, timeout=30)
    elapsed = time.time() - start
    
    status = "✅ OK" if response.status_code == 200 else "❌ FAIL"
    
    if response.status_code == 200:
        # Estimate full computation time
        batches = 278001 / n_pop
        est_mins = (elapsed * batches) / 60
        print(f"{n_pop} points vs 847 facs — {status} — {elapsed:.2f}s — Est total: {est_mins:.0f} min ({est_mins/60:.1f} hrs)")
    else:
        print(f"{n_pop} points vs 847 facs — {status} — {elapsed:.2f}s")

Finding optimal population point batch size...

10 points vs 847 facs — ✅ OK — 1.93s — Est total: 893 min (14.9 hrs)
20 points vs 847 facs — ❌ FAIL — 0.06s
30 points vs 847 facs — ❌ FAIL — 0.07s
40 points vs 847 facs — ❌ FAIL — 0.05s
50 points vs 847 facs — ❌ FAIL — 0.05s
75 points vs 847 facs — ❌ FAIL — 0.05s


ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))

In [33]:
import requests
import time
import numpy as np
from scipy.spatial import cKDTree

print("Testing K=10 vs K=20 vs K=50 vs K=100...\n")

fac_coords = master_dataset[['Longitude', 'Latitude']].values
fac_tree = cKDTree(fac_coords)

sample = master_pop.sample(200, random_state=42)

results = []

for i, (_, row) in enumerate(sample.iterrows()):
    if i % 50 == 0:
        print(f"  {i}/200 tested...")
    
    pop_coord = [[row['lon'], row['lat']]]
    times = {}
    
    for k in [10, 20, 50, 100]:
        _, idxs = fac_tree.query(pop_coord, k=k)
        best_time = float('inf')
        
        for idx in idxs[0]:
            fac_lon = fac_coords[idx, 0]
            fac_lat = fac_coords[idx, 1]
            
            url = (f"http://localhost:5000/route/v1/driving/"
                   f"{row['lon']},{row['lat']};"
                   f"{fac_lon},{fac_lat}?overview=false")
            
            try:
                r = requests.get(url, timeout=5)
                d = r.json()
                if d['code'] == 'Ok':
                    t = d['routes'][0]['duration'] / 60
                    if t < best_time:
                        best_time = t
            except:
                pass
        
        times[f'k{k}'] = round(best_time, 2)
    
    results.append(times)

results_df = pd.DataFrame(results)

print(f"\n=== K SENSITIVITY RESULTS ===")
print(f"\nAverage travel time:")
print(f"  K=10:  {results_df['k10'].mean():.2f} min")
print(f"  K=20:  {results_df['k20'].mean():.2f} min")
print(f"  K=50:  {results_df['k50'].mean():.2f} min")
print(f"  K=100: {results_df['k100'].mean():.2f} min")

print(f"\nCases where each K found shorter time than K=10:")
for k in ['k20', 'k50', 'k100']:
    diff = (results_df[k] < results_df['k10']).sum()
    print(f"  {k}: {diff} out of 200 ({diff/2:.1f}%)")

print(f"\nCases where K=100 found shorter than K=20:")
diff = (results_df['k100'] < results_df['k20']).sum()
print(f"  {diff} out of 200 ({diff/2:.1f}%)")

print(f"\nMax differences:")
print(f"  K=10 vs K=20:  {(results_df['k10'] - results_df['k20']).max():.2f} min")
print(f"  K=10 vs K=100: {(results_df['k10'] - results_df['k100']).max():.2f} min")
print(f"  K=20 vs K=100: {(results_df['k20'] - results_df['k100']).max():.2f} min")

Testing K=10 vs K=20 vs K=50 vs K=100...

  0/200 tested...
  50/200 tested...
  100/200 tested...
  150/200 tested...

=== K SENSITIVITY RESULTS ===

Average travel time:
  K=10:  14.07 min
  K=20:  13.22 min
  K=50:  13.19 min
  K=100: 13.19 min

Cases where each K found shorter time than K=10:
  k20: 7 out of 200 (3.5%)
  k50: 7 out of 200 (3.5%)
  k100: 7 out of 200 (3.5%)

Cases where K=100 found shorter than K=20:
  1 out of 200 (0.5%)

Max differences:
  K=10 vs K=20:  74.88 min
  K=10 vs K=100: 74.88 min
  K=20 vs K=100: 6.07 min


In [34]:
import requests
import time
import numpy as np
from scipy.spatial import cKDTree

print("Speed test — K=20 with min, mean, max recording...\n")

# Load facilities
master_dataset = pd.read_csv(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\master_dataset_v3.csv")

# Define groupings
GROUPINGS = {
    'any':        master_dataset,
    'chps':       master_dataset[master_dataset['Facility_Type'] == 'CHPS'],
    'maternity':  master_dataset[master_dataset['Facility_Type'].isin(['CHPS', 'MATERNITY HOME'])],
    'outpatient': master_dataset[master_dataset['Facility_Type'].isin(['CLINIC', 'HEALTH CENTRE', 'HOSPITAL', 'DISTRICT HOSPITAL', 'POLYCLINIC', 'REGIONAL HOSPITAL', 'TEACHING HOSPITAL', 'UNIVERSITY HOSPITAL/CLINIC'])],
    'emergency':  master_dataset[master_dataset['Facility_Type'].isin(['HOSPITAL', 'DISTRICT HOSPITAL', 'POLYCLINIC', 'REGIONAL HOSPITAL', 'TEACHING HOSPITAL', 'UNIVERSITY HOSPITAL/CLINIC'])],
    'specialist': master_dataset[master_dataset['Facility_Type'].isin(['REGIONAL HOSPITAL', 'TEACHING HOSPITAL', 'UNIVERSITY HOSPITAL/CLINIC'])],
    'psychiatric':master_dataset[master_dataset['Facility_Type'] == 'PSYCHIATRIC HOSPITAL'],
}

print("Facility counts per grouping:")
for name, df in GROUPINGS.items():
    print(f"  {name}: {len(df):,}")

# Speed test on 100 points
sample = master_pop.sample(100, random_state=42)
pop_coords = sample[['lon', 'lat']].values

# Test with emergency grouping (middle sized — 847 facilities)
fac_df = GROUPINGS['emergency']
fac_coords = fac_df[['Longitude', 'Latitude']].values
fac_tree = cKDTree(fac_coords)

print(f"\nSpeed testing K=20 on 100 points vs emergency ({len(fac_df)} facilities)...")
start = time.time()

for i, (_, row) in enumerate(sample.iterrows()):
    pop_coord = [[row['lon'], row['lat']]]
    distances, idxs = fac_tree.query(pop_coord, k=min(20, len(fac_coords)))
    
    times = []
    nearest_name = None
    nearest_time = float('inf')
    
    for idx in idxs[0]:
        fac_lon = fac_coords[idx, 0]
        fac_lat = fac_coords[idx, 1]
        
        url = (f"http://localhost:5000/route/v1/driving/"
               f"{row['lon']},{row['lat']};"
               f"{fac_lon},{fac_lat}?overview=false")
        
        try:
            r = requests.get(url, timeout=5)
            d = r.json()
            if d['code'] == 'Ok':
                t = d['routes'][0]['duration'] / 60
                times.append(t)
                if t < nearest_time:
                    nearest_time = t
                    nearest_idx = idx
        except:
            pass

elapsed = time.time() - start

print(f"✅ 100 points took: {round(elapsed, 1)} seconds")
print(f"\n=== TIME ESTIMATES FOR FULL RUN ===")
for name, df in GROUPINGS.items():
    # Scale by facility count relative to emergency
    scale = min(len(df), 20) / min(len(fac_df), 20)
    est_mins = round((elapsed / 100) * 278001 * scale / 60, 1)
    print(f"  {name} ({len(df):,} facs): ~{est_mins} min ({round(est_mins/60,1)} hrs)")

total_est = sum(
    round((elapsed / 100) * 278001 * min(len(df), 20) / min(len(fac_df), 20) / 60, 1)
    for df in GROUPINGS.values()
)
print(f"\n  TOTAL ESTIMATED: ~{round(total_est/60, 1)} hours")

Speed test — K=20 with min, mean, max recording...

Facility counts per grouping:
  any: 9,978
  chps: 6,733
  maternity: 6,981
  outpatient: 2,991
  emergency: 847
  specialist: 28
  psychiatric: 5

Speed testing K=20 on 100 points vs emergency (847 facilities)...
✅ 100 points took: 32.9 seconds

=== TIME ESTIMATES FOR FULL RUN ===
  any (9,978 facs): ~1525.8 min (25.4 hrs)
  chps (6,733 facs): ~1525.8 min (25.4 hrs)
  maternity (6,981 facs): ~1525.8 min (25.4 hrs)
  outpatient (2,991 facs): ~1525.8 min (25.4 hrs)
  emergency (847 facs): ~1525.8 min (25.4 hrs)
  specialist (28 facs): ~1525.8 min (25.4 hrs)
  psychiatric (5 facs): ~381.4 min (6.4 hrs)

  TOTAL ESTIMATED: ~158.9 hours


In [35]:
import time
import requests
from scipy.spatial import cKDTree
import numpy as np

# Test 500 points with K=20 on emergency for accurate ETA
fac_coords = GROUPINGS['emergency'][['Longitude', 'Latitude']].values
fac_tree = cKDTree(fac_coords)

sample = master_pop.sample(500, random_state=42)

print("Accurate speed test — 500 points K=20 emergency...\n")
start = time.time()

for i, (_, row) in enumerate(sample.iterrows()):
    if i % 100 == 0:
        elapsed = time.time() - start
        if i > 0:
            eta = (elapsed / i) * (500 - i) / 60
            print(f"  {i}/500 — ETA: {eta:.1f} min")
    
    pop_coord = [[row['lon'], row['lat']]]
    _, idxs = fac_tree.query(pop_coord, k=min(20, len(fac_coords)))
    
    best_time = float('inf')
    for idx in idxs[0]:
        url = (f"http://localhost:5000/route/v1/driving/"
               f"{row['lon']},{row['lat']};"
               f"{fac_coords[idx,0]},{fac_coords[idx,1]}"
               f"?overview=false")
        try:
            r = requests.get(url, timeout=5)
            d = r.json()
            if d['code'] == 'Ok':
                t = d['routes'][0]['duration'] / 60
                if t < best_time:
                    best_time = t
        except:
            pass

elapsed = time.time() - start
rate = elapsed / 500
est_per_grouping = round(rate * 278001 / 60, 1)
est_total = round(est_per_grouping * 7 / 60, 1)

print(f"\n✅ 500 points took: {round(elapsed/60, 1)} min")
print(f"Rate: {round(rate, 3)} sec/point")
print(f"\nEstimated per grouping: {est_per_grouping} min ({round(est_per_grouping/60,1)} hrs)")
print(f"Estimated total (7 groupings): {est_total} hrs")

Accurate speed test — 500 points K=20 emergency...

  100/500 — ETA: 2.1 min
  200/500 — ETA: 1.6 min
  300/500 — ETA: 1.1 min
  400/500 — ETA: 0.5 min

✅ 500 points took: 2.7 min
Rate: 0.323 sec/point

Estimated per grouping: 1494.6 min (24.9 hrs)
Estimated total (7 groupings): 174.4 hrs


In [36]:
import requests
import time
import numpy as np
from scipy.spatial import cKDTree

print("Speed test — K=20 using Table API (1 call per point)...\n")

fac_coords = GROUPINGS['emergency'][['Longitude', 'Latitude']].values
fac_tree = cKDTree(fac_coords)

sample = master_pop.sample(500, random_state=42)

start = time.time()

for i, (_, row) in enumerate(sample.iterrows()):
    if i % 100 == 0:
        elapsed = time.time() - start
        if i > 0:
            eta = (elapsed / i) * (500 - i) / 60
            print(f"  {i}/500 — ETA: {eta:.1f} min")

    pop_coord = [[row['lon'], row['lat']]]
    _, idxs = fac_tree.query(pop_coord, k=min(20, len(fac_coords)))

    # Get the 20 candidate facility coordinates
    candidates = fac_coords[idxs[0]]

    # Build ONE Table API call — 1 origin + 20 destinations
    all_coords = np.vstack([[row['lon'], row['lat']], candidates])
    coord_str = ";".join([f"{lon},{lat}" for lon, lat in all_coords])
    sources = "0"
    destinations = ";".join([str(i) for i in range(1, len(all_coords))])

    url = (f"http://localhost:5000/table/v1/driving/{coord_str}"
           f"?sources={sources}&destinations={destinations}&annotations=duration")

    try:
        r = requests.get(url, timeout=10)
        d = r.json()
        if d['code'] == 'Ok':
            times = [t/60 if t is not None else float('inf') 
                    for t in d['durations'][0]]
            best_time = min(times)
    except:
        pass

elapsed = time.time() - start
rate = elapsed / 500
est_per_grouping = round(rate * 278001 / 60, 1)
est_total = round(est_per_grouping * 7 / 60, 1)

print(f"\n✅ 500 points took: {round(elapsed/60, 2)} min")
print(f"Rate: {round(rate, 3)} sec/point")
print(f"\nEstimated per grouping: {est_per_grouping} min ({round(est_per_grouping/60,1)} hrs)")
print(f"Estimated total (7 groupings): {est_total} hrs")

Speed test — K=20 using Table API (1 call per point)...

  100/500 — ETA: 0.1 min
  200/500 — ETA: 0.1 min
  300/500 — ETA: 0.1 min
  400/500 — ETA: 0.0 min

✅ 500 points took: 0.18 min
Rate: 0.021 sec/point

Estimated per grouping: 98.0 min (1.6 hrs)
Estimated total (7 groupings): 11.4 hrs


In [37]:
import requests
import time
import numpy as np
import pandas as pd
import pickle
from scipy.spatial import cKDTree

print("=" * 65)
print("GHANA HEALTHCARE TRAVEL TIME RECOMPUTATION")
print("K=20 + OSRM Table API — Population Weighted")
print("=" * 65)

# ── LOAD DATA ──
print("\nLoading data...")
master_pop = pd.read_csv(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\master_population_data.csv")
master_dataset = pd.read_csv(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\master_dataset_v3.csv")
print(f"✅ Population points: {len(master_pop):,}")
print(f"✅ Facilities: {len(master_dataset):,}")

# ── DEFINE GROUPINGS ──
GROUPINGS = {
    'any':        master_dataset,
    'chps':       master_dataset[master_dataset['Facility_Type'] == 'CHPS'],
    'maternity':  master_dataset[master_dataset['Facility_Type'].isin(['CHPS', 'MATERNITY HOME'])],
    'outpatient': master_dataset[master_dataset['Facility_Type'].isin(['CLINIC', 'HEALTH CENTRE', 'HOSPITAL', 'DISTRICT HOSPITAL', 'POLYCLINIC', 'REGIONAL HOSPITAL', 'TEACHING HOSPITAL', 'UNIVERSITY HOSPITAL/CLINIC'])],
    'emergency':  master_dataset[master_dataset['Facility_Type'].isin(['HOSPITAL', 'DISTRICT HOSPITAL', 'POLYCLINIC', 'REGIONAL HOSPITAL', 'TEACHING HOSPITAL', 'UNIVERSITY HOSPITAL/CLINIC'])],
    'specialist': master_dataset[master_dataset['Facility_Type'].isin(['REGIONAL HOSPITAL', 'TEACHING HOSPITAL', 'UNIVERSITY HOSPITAL/CLINIC'])],
    'psychiatric':master_dataset[master_dataset['Facility_Type'] == 'PSYCHIATRIC HOSPITAL'],
}

print(f"\nFacility counts per grouping:")
for name, df in GROUPINGS.items():
    print(f"  {name}: {len(df):,}")

# ── OUTPUT STORAGE ──
# We will store for each grouping and each population point:
# - nearest_min (fastest facility by road)
# - nearest_mean (mean of 20 candidates)
# - nearest_max (max of 20 candidates)
# - nearest_fac_name
# - nearest_fac_type
# - nearest_fac_lon
# - nearest_fac_lat
# - failed (boolean)

PROJECT_PATH = r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project"

K = 20
n_pop = len(master_pop)
pop_coords = master_pop[['lon', 'lat']].values

# ── RUN EACH GROUPING ──
all_results = {}

for grouping_name, fac_df in GROUPINGS.items():
    print(f"\n{'='*65}")
    print(f"  {grouping_name.upper()} — {len(fac_df):,} facilities")
    print(f"{'='*65}")

    fac_coords = fac_df[['Longitude', 'Latitude']].values
    fac_names = fac_df['Name'].values
    fac_types = fac_df['Facility_Type'].values
    k_actual = min(K, len(fac_coords))

    fac_tree = cKDTree(fac_coords)

    # Results arrays
    nearest_min  = np.full(n_pop, np.nan)
    nearest_mean = np.full(n_pop, np.nan)
    nearest_max  = np.full(n_pop, np.nan)
    nearest_name = np.full(n_pop, '', dtype=object)
    nearest_type = np.full(n_pop, '', dtype=object)
    nearest_lon  = np.full(n_pop, np.nan)
    nearest_lat  = np.full(n_pop, np.nan)
    failed       = np.zeros(n_pop, dtype=bool)

    # Check for checkpoint
    ckpt_path = f"{PROJECT_PATH}\\checkpoint_{grouping_name}.pkl"
    start_idx = 0
    try:
        with open(ckpt_path, 'rb') as f:
            ckpt = pickle.load(f)
            nearest_min  = ckpt['nearest_min']
            nearest_mean = ckpt['nearest_mean']
            nearest_max  = ckpt['nearest_max']
            nearest_name = ckpt['nearest_name']
            nearest_type = ckpt['nearest_type']
            nearest_lon  = ckpt['nearest_lon']
            nearest_lat  = ckpt['nearest_lat']
            failed       = ckpt['failed']
            start_idx    = ckpt['next_idx']
            print(f"  ♻️  Resuming from checkpoint at {start_idx:,}")
    except:
        print(f"  Starting fresh...")

    start_time = time.time()

    for i in range(start_idx, n_pop):

        # Progress
        if i % 500 == 0 and i > start_idx:
            elapsed = time.time() - start_time
            rate = elapsed / (i - start_idx)
            eta = rate * (n_pop - i) / 60
            print(f"  {i:,}/{n_pop:,} ({i/n_pop*100:.1f}%) — ETA: {eta:.1f} min")

        # Find K nearest by straight line
        pop_coord = [[pop_coords[i, 0], pop_coords[i, 1]]]
        _, idxs = fac_tree.query(pop_coord, k=k_actual)
        candidates = fac_coords[idxs[0]]

        # ONE Table API call — 1 origin vs K candidates
        all_coords = np.vstack([[pop_coords[i, 0], pop_coords[i, 1]], candidates])
        coord_str = ";".join([f"{lon},{lat}" for lon, lat in all_coords])
        sources = "0"
        dests = ";".join([str(j) for j in range(1, len(all_coords))])

        url = (f"http://localhost:5000/table/v1/driving/{coord_str}"
               f"?sources={sources}&destinations={dests}&annotations=duration")

        try:
            r = requests.get(url, timeout=10)
            d = r.json()

            if d['code'] == 'Ok':
                times = [t/60 if t is not None else np.nan
                        for t in d['durations'][0]]
                times_arr = np.array(times)
                valid = times_arr[~np.isnan(times_arr)]

                if len(valid) > 0:
                    best_idx_local = np.nanargmin(times_arr)
                    nearest_min[i]  = round(np.nanmin(times_arr), 4)
                    nearest_mean[i] = round(np.nanmean(times_arr), 4)
                    nearest_max[i]  = round(np.nanmax(times_arr), 4)
                    nearest_name[i] = fac_names[idxs[0][best_idx_local]]
                    nearest_type[i] = fac_types[idxs[0][best_idx_local]]
                    nearest_lon[i]  = candidates[best_idx_local, 0]
                    nearest_lat[i]  = candidates[best_idx_local, 1]
                else:
                    failed[i] = True
            else:
                failed[i] = True

        except Exception:
            failed[i] = True

        # Save checkpoint every 10,000 points
        if i > 0 and i % 10000 == 0:
            with open(ckpt_path, 'wb') as f:
                pickle.dump({
                    'nearest_min': nearest_min,
                    'nearest_mean': nearest_mean,
                    'nearest_max': nearest_max,
                    'nearest_name': nearest_name,
                    'nearest_type': nearest_type,
                    'nearest_lon': nearest_lon,
                    'nearest_lat': nearest_lat,
                    'failed': failed,
                    'next_idx': i
                }, f)
            print(f"  💾 Checkpoint saved at {i:,}")

    # Save grouping results
    all_results[grouping_name] = {
        'nearest_min':  nearest_min,
        'nearest_mean': nearest_mean,
        'nearest_max':  nearest_max,
        'nearest_name': nearest_name,
        'nearest_type': nearest_type,
        'nearest_lon':  nearest_lon,
        'nearest_lat':  nearest_lat,
        'failed':       failed,
    }

    elapsed_total = round((time.time() - start_time) / 60, 1)
    print(f"\n  ✅ {grouping_name.upper()} DONE in {elapsed_total} min!")
    print(f"  Within 30 min: {(nearest_min <= 30).sum():,} ({(nearest_min <= 30).mean()*100:.1f}%)")
    print(f"  Failed: {failed.sum():,}")

    # Intermediate save
    temp_df = master_pop.copy()
    temp_df[f'nearest_{grouping_name}_min']  = nearest_min
    temp_df[f'nearest_{grouping_name}_mean'] = nearest_mean
    temp_df[f'nearest_{grouping_name}_max']  = nearest_max
    temp_df[f'nearest_{grouping_name}_fac_name'] = nearest_name
    temp_df[f'nearest_{grouping_name}_fac_type'] = nearest_type
    temp_df.to_csv(f"{PROJECT_PATH}\\nearest_{grouping_name}_times_v2.csv", index=False)
    print(f"  💾 Saved nearest_{grouping_name}_times_v2.csv")

# ── FINAL MASTER FILE ──
print(f"\n{'='*65}")
print("BUILDING FINAL MASTER FILE...")
print(f"{'='*65}")

final_df = master_pop.copy()

for grouping_name, results in all_results.items():
    final_df[f'nearest_{grouping_name}_min']      = results['nearest_min']
    final_df[f'nearest_{grouping_name}_mean']     = results['nearest_mean']
    final_df[f'nearest_{grouping_name}_max']      = results['nearest_max']
    final_df[f'nearest_{grouping_name}_fac_name'] = results['nearest_name']
    final_df[f'nearest_{grouping_name}_fac_type'] = results['nearest_type']

out_path = f"{PROJECT_PATH}\\master_population_data_v2.csv"
final_df.to_csv(out_path, index=False)
print(f"✅ Saved master_population_data_v2.csv")
print(f"   Shape: {final_df.shape}")

print(f"\n🎉 ALL DONE!")

# ── NATIONAL SUMMARY ──
print(f"\n=== NATIONAL POPULATION WEIGHTED SUMMARY ===")
total_pop = final_df['population'].sum()
for label, col in [
    ('Any facility', 'nearest_any_min'),
    ('Emergency', 'nearest_emergency_min'),
    ('Outpatient', 'nearest_outpatient_min'),
    ('Specialist', 'nearest_specialist_min'),
]:
    pct = final_df.loc[final_df[col] <= 30, 'population'].sum() / total_pop * 100
    print(f"  {label} within 30 min: {pct:.1f}%")

GHANA HEALTHCARE TRAVEL TIME RECOMPUTATION
K=20 + OSRM Table API — Population Weighted

Loading data...
✅ Population points: 278,001
✅ Facilities: 9,978

Facility counts per grouping:
  any: 9,978
  chps: 6,733
  maternity: 6,981
  outpatient: 2,991
  emergency: 847
  specialist: 28
  psychiatric: 5

  ANY — 9,978 facilities
  Starting fresh...
  500/278,001 (0.2%) — ETA: 88.2 min
  1,000/278,001 (0.4%) — ETA: 85.0 min
  1,500/278,001 (0.5%) — ETA: 82.6 min
  2,000/278,001 (0.7%) — ETA: 89.8 min
  2,500/278,001 (0.9%) — ETA: 91.5 min
  3,000/278,001 (1.1%) — ETA: 94.1 min
  3,500/278,001 (1.3%) — ETA: 97.3 min
  4,000/278,001 (1.4%) — ETA: 98.4 min
  4,500/278,001 (1.6%) — ETA: 98.7 min
  5,000/278,001 (1.8%) — ETA: 99.1 min
  5,500/278,001 (2.0%) — ETA: 97.5 min
  6,000/278,001 (2.2%) — ETA: 97.8 min
  6,500/278,001 (2.3%) — ETA: 95.9 min
  7,000/278,001 (2.5%) — ETA: 95.1 min
  7,500/278,001 (2.7%) — ETA: 94.3 min
  8,000/278,001 (2.9%) — ETA: 94.4 min
  8,500/278,001 (3.1%) — ETA: 9

KeyboardInterrupt: 

In [38]:
import pandas as pd
import numpy as np

print("Loading new results...")

# Load each grouping result
project_path = r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project"

any_df      = pd.read_csv(f"{project_path}\\nearest_any_times_v2.csv")
chps_df     = pd.read_csv(f"{project_path}\\nearest_chps_times_v2.csv")
maternity_df= pd.read_csv(f"{project_path}\\nearest_maternity_times_v2.csv")
outpatient_df=pd.read_csv(f"{project_path}\\nearest_outpatient_times_v2.csv")
emergency_df= pd.read_csv(f"{project_path}\\nearest_emergency_times_v2.csv")
specialist_df=pd.read_csv(f"{project_path}\\nearest_specialist_times_v2.csv")

total_pop = any_df['population'].sum()

print(f"\n=== POPULATION WEIGHTED HEADLINE NUMBERS (K=20) ===\n")

for label, df, col, threshold in [
    ('Any facility',  any_df,       'nearest_any_min',       30),
    ('CHPS',          chps_df,      'nearest_chps_min',      30),
    ('Maternity',     maternity_df, 'nearest_maternity_min', 30),
    ('Outpatient',    outpatient_df,'nearest_outpatient_min',30),
    ('Emergency',     emergency_df, 'nearest_emergency_min', 30),
    ('Specialist',    specialist_df,'nearest_specialist_min',30),
]:
    within = df.loc[df[col] <= threshold, 'population'].sum() / total_pop * 100
    beyond = 100 - within
    median = df[col].median()
    mean   = df[col].mean()
    
    print(f"{label}:")
    print(f"  Within {threshold} min: {within:.1f}%")
    print(f"  Beyond {threshold} min: {beyond:.1f}%")
    print(f"  Mean:   {mean:.1f} min")
    print(f"  Median: {median:.1f} min")
    print()

Loading new results...

=== POPULATION WEIGHTED HEADLINE NUMBERS (K=20) ===

Any facility:
  Within 30 min: 98.9%
  Beyond 30 min: 1.1%
  Mean:   12.2 min
  Median: 6.8 min

CHPS:
  Within 30 min: 98.7%
  Beyond 30 min: 1.3%
  Mean:   13.1 min
  Median: 7.8 min

Maternity:
  Within 30 min: 98.7%
  Beyond 30 min: 1.3%
  Mean:   13.1 min
  Median: 7.8 min

Outpatient:
  Within 30 min: 94.7%
  Beyond 30 min: 5.3%
  Mean:   24.2 min
  Median: 16.7 min

Emergency:
  Within 30 min: 83.0%
  Beyond 30 min: 17.0%
  Mean:   41.2 min
  Median: 32.1 min

Specialist:
  Within 30 min: 45.2%
  Beyond 30 min: 54.8%
  Mean:   117.7 min
  Median: 109.4 min



In [39]:
# Check if K=20 results for specialist make sense
print("Specialist facility distribution:")
specialist_facs = master_dataset[master_dataset['Facility_Type'].isin([
    'REGIONAL HOSPITAL', 'TEACHING HOSPITAL', 'UNIVERSITY HOSPITAL/CLINIC'
])]
print(specialist_facs[['Name', 'Facility_Type', 'Region']].to_string(index=False))

Specialist facility distribution:
                                   Name              Facility_Type        Region
                 BIU ST. MARTINS CLINIC          TEACHING HOSPITAL    UPPER EAST
                BONO  REGIONAL HOSPITAL          REGIONAL HOSPITAL          BONO
           CAPE COAST TEACHING HOSPITAL          TEACHING HOSPITAL       CENTRAL
              EASTERN REGIONAL HOSPITAL          REGIONAL HOSPITAL       EASTERN
        EFFIA NKWANTA REGIONAL HOSPITAL          REGIONAL HOSPITAL       WESTERN
           ENTRANCE UNIVERSITY HOSPITAL UNIVERSITY HOSPITAL/CLINIC GREATER ACCRA
GREATER ACCRA REGIONAL HOSPITAL - RIDGE          REGIONAL HOSPITAL GREATER ACCRA
                   HO TEACHING HOSPITAL          TEACHING HOSPITAL         VOLTA
     KINGS AND QUEENS TEACHING HOSPITAL          TEACHING HOSPITAL       EASTERN
                         KNUST HOSPITAL UNIVERSITY HOSPITAL/CLINIC       ASHANTI
         KOMFO ANOKYE TEACHING HOSPITAL          TEACHING HOSPITAL       AS

In [41]:
import time
import requests
import numpy as np
import pandas as pd

print("=" * 60)
print("BENCHMARK TEST: INWARD NETWORK CONTROLS (ISOCHRONE ENGINE)")
print("=" * 60)

# Fixed variable name from 'master' to 'master_dataset'
grouping_definitions = {
    'any': master_dataset,
    'chps': master_dataset[master_dataset['Facility_Type'].str.upper() == 'CHPS'],
    'maternity': master_dataset[master_dataset['Facility_Type'].str.upper().str.contains('MATERNITY|CLINIC', na=False)],
    'outpatient': master_dataset[master_dataset['Facility_Type'].str.upper().isin(['HEALTH CENTRE', 'CLINIC', 'POLYCLINIC'])],
    'emergency': master_dataset[master_dataset['Facility_Type'].str.upper().isin(['HOSPITAL', 'DISTRICT HOSPITAL'])],
    'specialist': master_dataset[master_dataset['Facility_Type'].str.upper().isin(['REGIONAL HOSPITAL', 'TEACHING HOSPITAL', 'UNIVERSITY HOSPITAL/CLINIC'])],
    'psychiatric': master_dataset[master_dataset['Facility_Type'].str.upper().str.contains('PSYCHIATRIC|MENTAL', na=False)]
}

print(f"Total Population Grid Points to resolve: {len(pop_df):,}")
print("\nFacility counts per grouping:")
for group_name, df in grouping_definitions.items():
    print(f"  {group_name:<12} : {len(df):,} facilities")

print("-" * 60)
print("Running simulation step to measure OSRM server response times...")
print("-" * 60)

# Sample size for speed testing (Test 20 facilities from the dense 'emergency' tier)
test_sample_size = 20
test_facilities = grouping_definitions['emergency'].sample(min(test_sample_size, len(grouping_definitions['emergency'])), random_state=42)

start_time = time.time()
successful_calls = 0

for idx, (_, facility) in enumerate(test_facilities.iterrows()):
    lon_val, lat_val = facility['Longitude'], facility['Latitude']
    
    # Grab a small sample of target points to simulate the table array network cost
    sample_targets = pop_df.sample(50, random_state=idx)
    coords = f"{lon_val},{lat_val}"
    for _, pt in sample_targets.iterrows():
        # Match your dataframe's exact column names 'lon' and 'lat'
        coords += f";{pt['lon']},{pt['lat']}"
        
    url = f"http://localhost:5000/table/v1/driving/{coords}?sources=0"
    
    try:
        response = requests.get(url, timeout=5)
        if response.status_code == 200 and response.json().get('code') == 'Ok':
            successful_calls += 1
    except Exception:
        pass

elapsed_time = time.time() - start_time
avg_time_per_facility = elapsed_time / test_sample_size if successful_calls > 0 else 0

print(f"\n✅ Benchmark Complete!")
print(f"  Processed {successful_calls} network clusters successfully.")
print(f"  Average OSRM network execution time: {avg_time_per_facility:.4f} seconds per facility cluster.")

print("\n" + "=" * 60)
print("ESTIMATED PROCESSING TIMES FOR ALL 7 GROUPINGS USING INWARD ANALYSIS")
print("=" * 60)

total_estimated_seconds = 0

for group_name, df in grouping_definitions.items():
    facility_count = len(df)
    estimated_seconds = facility_count * avg_time_per_facility
    total_estimated_seconds += estimated_seconds
    
    if estimated_seconds < 60:
        time_display = f"{estimated_seconds:.1f} seconds"
    elif estimated_seconds < 3600:
        time_display = f"{estimated_seconds / 60:.1f} minutes"
    else:
        time_display = f"{estimated_seconds / 3600:.1f} hours"
        
    print(f"  {group_name:<12} ({facility_count:>5} facs) ➔ Estimated Runtime: {time_display}")

print("-" * 60)
final_minutes = total_estimated_seconds / 60
print(f"🎉 TOTAL PROJECT ESTIMATED RUNTIME FOR ALL 7 TIERS: {final_minutes:.1f} minutes")
print("=" * 60)

BENCHMARK TEST: INWARD NETWORK CONTROLS (ISOCHRONE ENGINE)


NameError: name 'pop_df' is not defined

In [46]:
import time
import requests
import numpy as np
import pandas as pd

print("=" * 60)
print("BENCHMARK TEST: INWARD NETWORK CONTROLS (ISOCHRONE ENGINE)")
print("=" * 60)

# 100% verified variables from your notebook
grouping_definitions = {
    'any': master_dataset,
    'chps': master_dataset[master_dataset['Facility_Type'].str.upper() == 'CHPS'],
    'maternity': master_dataset[master_dataset['Facility_Type'].str.upper().str.contains('MATERNITY|CLINIC', na=False)],
    'outpatient': master_dataset[master_dataset['Facility_Type'].str.upper().isin(['HEALTH CENTRE', 'CLINIC', 'POLYCLINIC'])],
    'emergency': master_dataset[master_dataset['Facility_Type'].str.upper().isin(['HOSPITAL', 'DISTRICT HOSPITAL'])],
    'specialist': master_dataset[master_dataset['Facility_Type'].str.upper().isin(['REGIONAL HOSPITAL', 'TEACHING HOSPITAL', 'UNIVERSITY HOSPITAL/CLINIC'])],
    'psychiatric': master_dataset[master_dataset['Facility_Type'].str.upper().str.contains('PSYCHIATRIC|MENTAL', na=False)]
}

print(f"Total Population Grid Points to resolve: {len(master_pop):,}")
print("\nFacility counts per grouping:")
for group_name, df in grouping_definitions.items():
    print(f"  {group_name:<12} : {len(df):,} facilities")

print("-" * 60)
print("Running simulation step to measure OSRM server response times...")
print("-" * 60)

# Sample size for speed testing (Test 20 facilities from the dense 'emergency' tier)
test_sample_size = 20
test_facilities = grouping_definitions['emergency'].sample(min(test_sample_size, len(grouping_definitions['emergency'])), random_state=42)

start_time = time.time()
successful_calls = 0

for idx, (_, facility) in enumerate(test_facilities.iterrows()):
    lon_val, lat_val = facility['Longitude'], facility['Latitude']
    
    # Using your exact 'master_pop' dataframe and its 'lon' / 'lat' columns
    sample_targets = master_pop.sample(50, random_state=idx)
    coords = f"{lon_val},{lat_val}"
    for _, pt in sample_targets.iterrows():
        coords += f";{pt['lon']},{pt['lat']}"
        
    url = f"http://localhost:5000/table/v1/driving/{coords}?sources=0"
    
    try:
        response = requests.get(url, timeout=5)
        if response.status_code == 200 and response.json().get('code') == 'Ok':
            successful_calls += 1
    except Exception:
        pass

elapsed_time = time.time() - start_time
avg_time_per_facility = elapsed_time / test_sample_size if successful_calls > 0 else 0

print(f"\n✅ Benchmark Complete!")
print(f"  Processed {successful_calls} network clusters successfully.")
print(f"  Average OSRM network execution time: {avg_time_per_facility:.4f} seconds per facility cluster.")

print("\n" + "=" * 60)
print("ESTIMATED PROCESSING TIMES FOR ALL 7 GROUPINGS USING INWARD ANALYSIS")
print("=" * 60)

total_estimated_seconds = 0

for group_name, df in grouping_definitions.items():
    facility_count = len(df)
    estimated_seconds = facility_count * avg_time_per_facility
    total_estimated_seconds += estimated_seconds
    
    if estimated_seconds < 60:
        time_display = f"{estimated_seconds:.1f} seconds"
    elif estimated_seconds < 3600:
        time_display = f"{estimated_seconds / 60:.1f} minutes"
    else:
        time_display = f"{estimated_seconds / 3600:.1f} hours"
        
    print(f"  {group_name:<12} ({facility_count:>5} facs) ➔ Estimated Runtime: {time_display}")

print("-" * 60)
final_minutes = total_estimated_seconds / 60
print(f"🎉 TOTAL PROJECT ESTIMATED RUNTIME FOR ALL 7 TIERS: {final_minutes:.1f} minutes")
print("=" * 60)

BENCHMARK TEST: INWARD NETWORK CONTROLS (ISOCHRONE ENGINE)
Total Population Grid Points to resolve: 278,001

Facility counts per grouping:
  any          : 9,978 facilities
  chps         : 6,733 facilities
  maternity    : 1,278 facilities
  outpatient   : 2,237 facilities
  emergency    : 726 facilities
  specialist   : 28 facilities
  psychiatric  : 5 facilities
------------------------------------------------------------
Running simulation step to measure OSRM server response times...
------------------------------------------------------------

✅ Benchmark Complete!
  Processed 20 network clusters successfully.
  Average OSRM network execution time: 0.2824 seconds per facility cluster.

ESTIMATED PROCESSING TIMES FOR ALL 7 GROUPINGS USING INWARD ANALYSIS
  any          ( 9978 facs) ➔ Estimated Runtime: 47.0 minutes
  chps         ( 6733 facs) ➔ Estimated Runtime: 31.7 minutes
  maternity    ( 1278 facs) ➔ Estimated Runtime: 6.0 minutes
  outpatient   ( 2237 facs) ➔ Estimated Runti

In [1]:
import pandas as pd

any_v2 = pd.read_csv(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\nearest_any_times_v2.csv")

print(f"Shape: {any_v2.shape}")
print(f"\nColumns: {list(any_v2.columns)}")
print(f"\nSample:")
print(any_v2.head(3).to_string(index=False))

Shape: (278001, 29)

Columns: ['lat', 'lon', 'population', 'nearest_any_min', 'nearest_chps_min', 'nearest_maternity_min', 'nearest_outpatient_min', 'nearest_emergency_min', 'nearest_specialist_min', 'nearest_psychiatric_min', 'node_id', 'nearest_fac_lon', 'nearest_fac_lat', 'nearest_fac_name', 'nearest_fac_type', 'journey_major_road_min', 'journey_connecting_road_min', 'journey_urban_road_min', 'journey_rural_unpaved_min', 'journey_walking_min', 'journey_total_min', 'journey_failed', 'Region', 'District', 'accessibility_score', 'nearest_any_mean', 'nearest_any_max', 'nearest_any_fac_name', 'nearest_any_fac_type']

Sample:
      lat       lon  population  nearest_any_min  nearest_chps_min  nearest_maternity_min  nearest_outpatient_min  nearest_emergency_min  nearest_specialist_min  nearest_psychiatric_min  node_id  nearest_fac_lon  nearest_fac_lat nearest_fac_name nearest_fac_type  journey_major_road_min  journey_connecting_road_min  journey_urban_road_min  journey_rural_unpaved_min  j

In [2]:
# Compare nearest_emergency_min in v2 file vs original
original = pd.read_csv(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\master_population_data.csv")
v2 = any_v2

print("Original K=10 emergency times (first 5):")
print(original['nearest_emergency_min'].head())

print("\nV2 file emergency times (first 5):")
print(v2['nearest_emergency_min'].head())

print("\nAre they the same?")
print((original['nearest_emergency_min'] == v2['nearest_emergency_min']).all())

Original K=10 emergency times (first 5):
0    31.205000
1    30.815000
2    30.261667
3    29.095000
4    28.176667
Name: nearest_emergency_min, dtype: float64

V2 file emergency times (first 5):
0    31.205000
1    30.815000
2    30.261667
3    29.095000
4    28.176667
Name: nearest_emergency_min, dtype: float64

Are they the same?
True


In [3]:
import pandas as pd

print("Loading all K=20 grouping files...")
project = r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project"

any_v2       = pd.read_csv(f"{project}\\nearest_any_times_v2.csv")
chps_v2      = pd.read_csv(f"{project}\\nearest_chps_times_v2.csv")
maternity_v2 = pd.read_csv(f"{project}\\nearest_maternity_times_v2.csv")
outpatient_v2= pd.read_csv(f"{project}\\nearest_outpatient_times_v2.csv")
emergency_v2 = pd.read_csv(f"{project}\\nearest_emergency_times_v2.csv")
specialist_v2= pd.read_csv(f"{project}\\nearest_specialist_times_v2.csv")

print("Checking which column each file updated...")
for name, df in [
    ('any', any_v2),
    ('chps', chps_v2),
    ('maternity', maternity_v2),
    ('outpatient', outpatient_v2),
    ('emergency', emergency_v2),
    ('specialist', specialist_v2),
]:
    # Find columns with _mean or _max — those are the K=20 ones
    k20_cols = [c for c in df.columns if '_mean' in c or '_max' in c]
    print(f"  {name}: K=20 columns = {k20_cols}")

Loading all K=20 grouping files...
Checking which column each file updated...
  any: K=20 columns = ['nearest_any_mean', 'nearest_any_max']
  chps: K=20 columns = ['nearest_chps_mean', 'nearest_chps_max']
  maternity: K=20 columns = ['nearest_maternity_mean', 'nearest_maternity_max']
  outpatient: K=20 columns = ['nearest_outpatient_mean', 'nearest_outpatient_max']
  emergency: K=20 columns = ['nearest_emergency_mean', 'nearest_emergency_max']
  specialist: K=20 columns = ['nearest_specialist_mean', 'nearest_specialist_max']


In [4]:
# Compare emergency min in emergency_v2 vs original
original = pd.read_csv(f"{project}\\master_population_data.csv")

print("Comparing emergency travel times:")
print(f"\nOriginal K=10 emergency mean: {original['nearest_emergency_min'].mean():.2f}")
print(f"V2 emergency mean: {emergency_v2['nearest_emergency_min'].mean():.2f}")

print(f"\nAre they identical?")
print((original['nearest_emergency_min'].round(4) == emergency_v2['nearest_emergency_min'].round(4)).all())

print(f"\nCheck nearest_emergency_mean (the K=20 minimum):")
print(f"Emergency mean of means: {emergency_v2['nearest_emergency_mean'].mean():.2f}")
print(f"Emergency mean of max: {emergency_v2['nearest_emergency_max'].mean():.2f}")

print(f"\nFirst 5 rows comparison:")
print(pd.DataFrame({
    'k10_min': original['nearest_emergency_min'].head(),
    'v2_min': emergency_v2['nearest_emergency_min'].head(),
    'v2_mean': emergency_v2['nearest_emergency_mean'].head(),
    'v2_max': emergency_v2['nearest_emergency_max'].head(),
}).to_string())

Comparing emergency travel times:

Original K=10 emergency mean: 45.42
V2 emergency mean: 41.24

Are they identical?
False

Check nearest_emergency_mean (the K=20 minimum):
Emergency mean of means: 102.43
Emergency mean of max: 169.30

First 5 rows comparison:
     k10_min   v2_min  v2_mean    v2_max
0  31.205000  31.2050  67.7262  114.3450
1  30.815000  30.8150  67.3362  113.9550
2  30.261667  30.2617  66.7828  113.4017
3  29.095000  29.0950  65.6162  112.2350
4  28.176667  28.1767  64.6978  111.3167


In [5]:
import pandas as pd
import numpy as np

print("Building clean master population dataset v2...")

project = r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project"

# Load all K=20 files
print("Loading K=20 grouping files...")
any_v2        = pd.read_csv(f"{project}\\nearest_any_times_v2.csv")
chps_v2       = pd.read_csv(f"{project}\\nearest_chps_times_v2.csv")
maternity_v2  = pd.read_csv(f"{project}\\nearest_maternity_times_v2.csv")
outpatient_v2 = pd.read_csv(f"{project}\\nearest_outpatient_times_v2.csv")
emergency_v2  = pd.read_csv(f"{project}\\nearest_emergency_times_v2.csv")
specialist_v2 = pd.read_csv(f"{project}\\nearest_specialist_times_v2.csv")

print("✅ All files loaded")

# Start fresh — take only base columns from any_v2
base_cols = [
    'lat', 'lon', 'population', 'Region', 'District',
    'journey_major_road_min', 'journey_connecting_road_min',
    'journey_urban_road_min', 'journey_rural_unpaved_min',
    'journey_walking_min', 'journey_total_min', 'journey_failed',
    'accessibility_score',
    'nearest_fac_lon', 'nearest_fac_lat',
    'nearest_fac_name', 'nearest_fac_type'
]

master_v2 = any_v2[base_cols].copy()

# Add K=20 min times for all groupings
print("\nAdding K=20 travel times...")
master_v2['nearest_any_min']        = any_v2['nearest_any_min']
master_v2['nearest_any_mean']       = any_v2['nearest_any_mean']
master_v2['nearest_any_max']        = any_v2['nearest_any_max']

master_v2['nearest_chps_min']       = chps_v2['nearest_chps_min']
master_v2['nearest_chps_mean']      = chps_v2['nearest_chps_mean']
master_v2['nearest_chps_max']       = chps_v2['nearest_chps_max']

master_v2['nearest_maternity_min']  = maternity_v2['nearest_maternity_min']
master_v2['nearest_maternity_mean'] = maternity_v2['nearest_maternity_mean']
master_v2['nearest_maternity_max']  = maternity_v2['nearest_maternity_max']

master_v2['nearest_outpatient_min'] = outpatient_v2['nearest_outpatient_min']
master_v2['nearest_outpatient_mean']= outpatient_v2['nearest_outpatient_mean']
master_v2['nearest_outpatient_max'] = outpatient_v2['nearest_outpatient_max']

master_v2['nearest_emergency_min']  = emergency_v2['nearest_emergency_min']
master_v2['nearest_emergency_mean'] = emergency_v2['nearest_emergency_mean']
master_v2['nearest_emergency_max']  = emergency_v2['nearest_emergency_max']

master_v2['nearest_specialist_min'] = specialist_v2['nearest_specialist_min']
master_v2['nearest_specialist_mean']= specialist_v2['nearest_specialist_mean']
master_v2['nearest_specialist_max'] = specialist_v2['nearest_specialist_max']

print(f"✅ Shape: {master_v2.shape}")
print(f"✅ Columns: {list(master_v2.columns)}")

# Save
out_path = f"{project}\\master_population_data_v2.csv"
master_v2.to_csv(out_path, index=False)
print(f"\n✅ Saved master_population_data_v2.csv")

# Sense check
print(f"\n=== SENSE CHECK (cell weighted) ===")
for label, col in [
    ('Any facility', 'nearest_any_min'),
    ('Emergency',    'nearest_emergency_min'),
    ('Outpatient',   'nearest_outpatient_min'),
    ('Specialist',   'nearest_specialist_min'),
]:
    pct = (master_v2[col] <= 30).sum() / len(master_v2) * 100
    mean = master_v2[col].mean()
    print(f"  {label}: {pct:.1f}% within 30 min | mean: {mean:.1f} min")

Building clean master population dataset v2...
Loading K=20 grouping files...
✅ All files loaded

Adding K=20 travel times...
✅ Shape: (278001, 35)
✅ Columns: ['lat', 'lon', 'population', 'Region', 'District', 'journey_major_road_min', 'journey_connecting_road_min', 'journey_urban_road_min', 'journey_rural_unpaved_min', 'journey_walking_min', 'journey_total_min', 'journey_failed', 'accessibility_score', 'nearest_fac_lon', 'nearest_fac_lat', 'nearest_fac_name', 'nearest_fac_type', 'nearest_any_min', 'nearest_any_mean', 'nearest_any_max', 'nearest_chps_min', 'nearest_chps_mean', 'nearest_chps_max', 'nearest_maternity_min', 'nearest_maternity_mean', 'nearest_maternity_max', 'nearest_outpatient_min', 'nearest_outpatient_mean', 'nearest_outpatient_max', 'nearest_emergency_min', 'nearest_emergency_mean', 'nearest_emergency_max', 'nearest_specialist_min', 'nearest_specialist_mean', 'nearest_specialist_max']

✅ Saved master_population_data_v2.csv

=== SENSE CHECK (cell weighted) ===
  Any faci

In [6]:
import pandas as pd
import numpy as np

print("Rebuilding district accessibility scores from K=20 master...")

master_v2 = pd.read_csv(f"{project}\\master_population_data_v2.csv")

def emergency_band(series):
    scores = np.zeros(len(series))
    scores[series <= 30] = 100
    scores[(series > 30) & (series <= 60)] = 70
    scores[(series > 60) & (series <= 90)] = 40
    scores[(series > 90) & (series <= 120)] = 15
    scores[series > 120] = 0
    return scores

def specialist_band(series):
    scores = np.zeros(len(series))
    scores[series <= 60] = 100
    scores[(series > 60) & (series <= 120)] = 70
    scores[(series > 120) & (series <= 180)] = 40
    scores[series > 180] = 0
    return scores

def pop_pct(group, col, lower, upper=None):
    total = len(group)
    if total == 0:
        return 0
    if upper is None:
        mask = group[col] > lower
    elif lower == 0:
        mask = group[col] <= upper
    else:
        mask = (group[col] > lower) & (group[col] <= upper)
    return round(mask.sum() / total * 100, 1)

rows = []
total = master_v2.groupby(['Region', 'District']).ngroups
count = 0

for (region, district), group in master_v2.groupby(['Region', 'District']):
    if pd.isna(region) or pd.isna(district):
        continue
    count += 1
    if count % 50 == 0:
        print(f"  {count}/{total} districts...")

    row = {
        'Region': region,
        'District': district,
        'total_population': round(group['population'].sum()),
        'population_points': len(group),

        # Emergency bands
        'emergency_pct_within_30': pop_pct(group, 'nearest_emergency_min', 0, 30),
        'emergency_pct_30_60':     pop_pct(group, 'nearest_emergency_min', 30, 60),
        'emergency_pct_60_90':     pop_pct(group, 'nearest_emergency_min', 60, 90),
        'emergency_pct_90_120':    pop_pct(group, 'nearest_emergency_min', 90, 120),
        'emergency_pct_120plus':   pop_pct(group, 'nearest_emergency_min', 120),
        'emergency_band_score':    round(emergency_band(group['nearest_emergency_min']).mean(), 2),
        'emergency_mean_min':      round(group['nearest_emergency_min'].mean(), 1),
        'emergency_min_min':       round(group['nearest_emergency_min'].min(), 1),
        'emergency_max_min':       round(group['nearest_emergency_min'].max(), 1),

        # Any facility bands
        'any_pct_within_30': pop_pct(group, 'nearest_any_min', 0, 30),
        'any_pct_30_60':     pop_pct(group, 'nearest_any_min', 30, 60),
        'any_pct_60_90':     pop_pct(group, 'nearest_any_min', 60, 90),
        'any_pct_90_120':    pop_pct(group, 'nearest_any_min', 90, 120),
        'any_pct_120plus':   pop_pct(group, 'nearest_any_min', 120),
        'any_band_score':    round(emergency_band(group['nearest_any_min']).mean(), 2),
        'any_mean_min':      round(group['nearest_any_min'].mean(), 1),
        'any_min_min':       round(group['nearest_any_min'].min(), 1),
        'any_max_min':       round(group['nearest_any_min'].max(), 1),

        # Specialist bands
        'specialist_pct_within_60':  pop_pct(group, 'nearest_specialist_min', 0, 60),
        'specialist_pct_60_120':     pop_pct(group, 'nearest_specialist_min', 60, 120),
        'specialist_pct_120_180':    pop_pct(group, 'nearest_specialist_min', 120, 180),
        'specialist_pct_180plus':    pop_pct(group, 'nearest_specialist_min', 180),
        'specialist_band_score_wide':round(specialist_band(group['nearest_specialist_min']).mean(), 2),
        'specialist_mean_min':       round(group['nearest_specialist_min'].mean(), 1),
        'specialist_min_min':        round(group['nearest_specialist_min'].min(), 1),
        'specialist_max_min':        round(group['nearest_specialist_min'].max(), 1),

        # Outpatient bands
        'outpatient_pct_within_30': pop_pct(group, 'nearest_outpatient_min', 0, 30),
        'outpatient_pct_30_60':     pop_pct(group, 'nearest_outpatient_min', 30, 60),
        'outpatient_pct_60_90':     pop_pct(group, 'nearest_outpatient_min', 60, 90),
        'outpatient_pct_90_120':    pop_pct(group, 'nearest_outpatient_min', 90, 120),
        'outpatient_pct_120plus':   pop_pct(group, 'nearest_outpatient_min', 120),
        'outpatient_band_score':    round(emergency_band(group['nearest_outpatient_min']).mean(), 2),
        'outpatient_mean_min':      round(group['nearest_outpatient_min'].mean(), 1),
        'outpatient_min_min':       round(group['nearest_outpatient_min'].min(), 1),
        'outpatient_max_min':       round(group['nearest_outpatient_min'].max(), 1),

        # Road breakdown
        'road_pct_major_road':      round(group['journey_major_road_min'].mean() / group['journey_total_min'].mean() * 100, 1) if group['journey_total_min'].mean() > 0 else 0,
        'road_pct_urban_road':      round(group['journey_urban_road_min'].mean() / group['journey_total_min'].mean() * 100, 1) if group['journey_total_min'].mean() > 0 else 0,
        'road_pct_connecting_road': round(group['journey_connecting_road_min'].mean() / group['journey_total_min'].mean() * 100, 1) if group['journey_total_min'].mean() > 0 else 0,
        'road_pct_rural_unpaved':   round(group['journey_rural_unpaved_min'].mean() / group['journey_total_min'].mean() * 100, 1) if group['journey_total_min'].mean() > 0 else 0,
        'road_pct_walking':         round(group['journey_walking_min'].mean() / group['journey_total_min'].mean() * 100, 1) if group['journey_total_min'].mean() > 0 else 0,
        'road_quality_score':       round(
            (group['journey_major_road_min'].mean() * 100 +
             group['journey_urban_road_min'].mean() * 85 +
             group['journey_connecting_road_min'].mean() * 60 +
             group['journey_rural_unpaved_min'].mean() * 15 +
             group['journey_walking_min'].mean() * 0) /
            group['journey_total_min'].mean(), 2
        ) if group['journey_total_min'].mean() > 0 else 0,

        # E2SFCA
        'avg_accessibility_score': round(group['accessibility_score'].mean(), 6),
    }
    rows.append(row)

district_scores = pd.DataFrame(rows)

# Normalize E2SFCA
e_min = district_scores['avg_accessibility_score'].min()
e_max = district_scores['avg_accessibility_score'].max()
district_scores['e2sfca_normalized'] = round(
    (district_scores['avg_accessibility_score'] - e_min) /
    (e_max - e_min) * 100, 2
)

# Composite scores
district_scores['journey_access_score'] = round(
    (district_scores['emergency_band_score']       * 0.50) +
    (district_scores['specialist_band_score_wide'] * 0.35) +
    (district_scores['any_band_score']             * 0.15),
    2
)
district_scores['supply_adequacy_score'] = district_scores['e2sfca_normalized']
district_scores['composite_score'] = round(
    (district_scores['journey_access_score']  * 0.70) +
    (district_scores['supply_adequacy_score'] * 0.30),
    2
)

def categorize(score):
    if score >= 80:   return 'Thriving'
    elif score >= 60: return 'Decent'
    elif score >= 40: return 'Getting By'
    elif score >= 20: return 'Struggling'
    else:             return 'Dire'

district_scores['category'] = district_scores['composite_score'].apply(categorize)

# Save
out_path = f"{project}\\district_accessibility_scores.csv"
district_scores.to_csv(out_path, index=False)
print(f"\n✅ Saved district_accessibility_scores.csv — {district_scores.shape}")

print(f"\n=== CATEGORY DISTRIBUTION ===")
print(district_scores['category'].value_counts())

print(f"\n=== TOP 10 ===")
print(district_scores.nlargest(10, 'composite_score')[
    ['District', 'Region', 'journey_access_score',
     'supply_adequacy_score', 'composite_score', 'category']
].to_string(index=False))

print(f"\n=== BOTTOM 10 ===")
print(district_scores.nsmallest(10, 'composite_score')[
    ['District', 'Region', 'journey_access_score',
     'supply_adequacy_score', 'composite_score', 'category']
].to_string(index=False))

Rebuilding district accessibility scores from K=20 master...
  50/260 districts...
  100/260 districts...
  150/260 districts...
  200/260 districts...
  250/260 districts...

✅ Saved district_accessibility_scores.csv — (260, 51)

=== CATEGORY DISTRIBUTION ===
category
Decent        164
Getting By     58
Thriving       27
Struggling     10
Dire            1
Name: count, dtype: int64

=== TOP 10 ===
     District     Region  journey_access_score  supply_adequacy_score  composite_score category
   Bolga East Upper East                100.00                 100.00           100.00 Thriving
 Wa Municipal Upper West                 99.15                  90.82            96.65 Thriving
Nadowli-Kaleo Upper West                 97.65                  83.47            93.40 Thriving
        Lawra Upper West                 90.50                  94.35            91.65 Thriving
       Nabdam Upper East                 98.69                  73.45            91.12 Thriving
   Bolgatanga Upper Ea

In [8]:
import requests

try:
    r = requests.get("http://localhost:5000/", timeout=5)
    print("✅ OSRM is running!")
except:
    print("❌ OSRM not running — start Docker first")

✅ OSRM is running!


In [9]:
import requests
import time
import numpy as np
from scipy.spatial import cKDTree

print("Speed test — journey breakdown for emergency and specialist...\n")

# Load master dataset for facility coordinates
master_dataset = pd.read_csv(f"{project}\\master_dataset_v3.csv")

# Emergency facilities
emergency_facs = master_dataset[master_dataset['Facility_Type'].isin([
    'HOSPITAL', 'DISTRICT HOSPITAL', 'POLYCLINIC',
    'REGIONAL HOSPITAL', 'TEACHING HOSPITAL',
    'UNIVERSITY HOSPITAL/CLINIC'
])][['Longitude', 'Latitude', 'Name', 'Facility_Type']].reset_index(drop=True)

# Specialist facilities
specialist_facs = master_dataset[master_dataset['Facility_Type'].isin([
    'REGIONAL HOSPITAL', 'TEACHING HOSPITAL',
    'UNIVERSITY HOSPITAL/CLINIC'
])][['Longitude', 'Latitude', 'Name', 'Facility_Type']].reset_index(drop=True)

print(f"Emergency facilities: {len(emergency_facs)}")
print(f"Specialist facilities: {len(specialist_facs)}")

# Build road KD-Tree for road type matching
from scipy.spatial import cKDTree as ckd
import geopandas as gpd

print("\nLoading road network for journey matching...")
roads = gpd.read_file(f"{project}\\ghana-260322-free.shp\\gis_osm_roads_free_1.shp")

mode_map = {
    'motorway': 'major_road', 'motorway_link': 'major_road',
    'trunk': 'major_road', 'trunk_link': 'major_road',
    'primary': 'major_road', 'primary_link': 'major_road',
    'secondary': 'connecting_road', 'secondary_link': 'connecting_road',
    'tertiary': 'connecting_road', 'tertiary_link': 'connecting_road',
    'residential': 'urban_road', 'living_street': 'urban_road',
    'service': 'urban_road', 'busway': 'urban_road',
    'unclassified': 'rural_unpaved', 'track': 'rural_unpaved',
    'track_grade1': 'rural_unpaved', 'track_grade2': 'rural_unpaved',
    'track_grade3': 'rural_unpaved', 'track_grade4': 'rural_unpaved',
    'track_grade5': 'rural_unpaved', 'cycleway': 'rural_unpaved',
    'unknown': 'rural_unpaved', 'path': 'walking', 'footway': 'walking',
    'pedestrian': 'walking', 'steps': 'walking', 'bridleway': 'walking'
}
roads['mode'] = roads['fclass'].map(mode_map).fillna('rural_unpaved')

print("Building road KD-Tree...")
road_midpoints = []
road_modes = []
for _, road in roads.iterrows():
    coords = list(road.geometry.coords)
    for i in range(len(coords) - 1):
        mid_lon = (coords[i][0] + coords[i+1][0]) / 2
        mid_lat = (coords[i][1] + coords[i+1][1]) / 2
        road_midpoints.append([mid_lon, mid_lat])
        road_modes.append(road['mode'])

road_midpoints = np.array(road_midpoints)
road_modes = np.array(road_modes)
road_kd = cKDTree(road_midpoints)
print(f"✅ Road KD-Tree built on {len(road_midpoints):,} segments")

# Speed test on 100 points
sample = master_v2.sample(100, random_state=42)
emerg_coords = emergency_facs[['Longitude', 'Latitude']].values
emerg_tree = cKDTree(emerg_coords)

start = time.time()

for _, row in sample.iterrows():
    # Find nearest emergency facility using K=20
    pop_coord = [[row['lon'], row['lat']]]
    _, idxs = emerg_tree.query(pop_coord, k=min(20, len(emerg_coords)))
    candidates = emerg_coords[idxs[0]]

    # Table API call
    all_coords = np.vstack([[row['lon'], row['lat']], candidates])
    coord_str = ";".join([f"{lon},{lat}" for lon, lat in all_coords])
    sources = "0"
    dests = ";".join([str(j) for j in range(1, len(all_coords))])
    url = (f"http://localhost:5000/table/v1/driving/{coord_str}"
           f"?sources={sources}&destinations={dests}&annotations=duration")

    try:
        r = requests.get(url, timeout=10)
        d = r.json()
        if d['code'] == 'Ok':
            times = [t/60 if t is not None else np.nan for t in d['durations'][0]]
            best_idx = np.nanargmin(times)
            best_fac_lon = candidates[best_idx, 0]
            best_fac_lat = candidates[best_idx, 1]

            # Get route geometry
            route_url = (f"http://localhost:5000/route/v1/driving/"
                        f"{row['lon']},{row['lat']};"
                        f"{best_fac_lon},{best_fac_lat}"
                        f"?geometries=geojson&overview=full&annotations=duration")
            route_r = requests.get(route_url, timeout=10)
            route_d = route_r.json()

            if route_d['code'] == 'Ok':
                coords = route_d['routes'][0]['geometry']['coordinates']
                seg_durations = route_d['routes'][0]['legs'][0]['annotation']['duration']

                if len(coords) >= 2:
                    mids = np.array([
                        [(coords[j][0] + coords[j+1][0]) / 2,
                         (coords[j][1] + coords[j+1][1]) / 2]
                        for j in range(len(coords) - 1)
                    ])
                    _, road_idxs = road_kd.query(mids)
                    modes = road_modes[road_idxs]

                    breakdown = {'major_road': 0, 'connecting_road': 0,
                                'urban_road': 0, 'rural_unpaved': 0, 'walking': 0}
                    for j, mode in enumerate(modes):
                        if j < len(seg_durations):
                            breakdown[mode] += seg_durations[j] / 60
    except:
        pass

elapsed = time.time() - start
est_mins = round((elapsed / 100) * 278001 / 60, 1)
est_hrs = round(est_mins / 60, 1)

print(f"\n100 points took: {round(elapsed, 1)} seconds")
print(f"Estimated for 278,001 points: {est_mins} min ({est_hrs} hrs)")
print(f"For both emergency + specialist: {round(est_hrs*2, 1)} hrs")

Speed test — journey breakdown for emergency and specialist...

Emergency facilities: 847
Specialist facilities: 28

Loading road network for journey matching...
Building road KD-Tree...
✅ Road KD-Tree built on 4,542,352 segments

100 points took: 9.9 seconds
Estimated for 278,001 points: 458.7 min (7.6 hrs)
For both emergency + specialist: 15.2 hrs


In [10]:
import requests
import time
import numpy as np
import pandas as pd
import pickle
from scipy.spatial import cKDTree

print("=" * 65)
print("JOURNEY BREAKDOWN — EMERGENCY + SPECIALIST")
print("=" * 65)

project = r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project"

# Load data
master_v2 = pd.read_csv(f"{project}\\master_population_data_v2.csv")
master_dataset = pd.read_csv(f"{project}\\master_dataset_v3.csv")

# Define facility groups
GROUPS = {
    'emergency': master_dataset[master_dataset['Facility_Type'].isin([
        'HOSPITAL', 'DISTRICT HOSPITAL', 'POLYCLINIC',
        'REGIONAL HOSPITAL', 'TEACHING HOSPITAL',
        'UNIVERSITY HOSPITAL/CLINIC'
    ])],
    'specialist': master_dataset[master_dataset['Facility_Type'].isin([
        'REGIONAL HOSPITAL', 'TEACHING HOSPITAL',
        'UNIVERSITY HOSPITAL/CLINIC'
    ])],
}

for name, df in GROUPS.items():
    print(f"{name}: {len(df)} facilities")

# Road KD-Tree
print("\nLoading road network...")
import geopandas as gpd

roads = gpd.read_file(f"{project}\\ghana-260322-free.shp\\gis_osm_roads_free_1.shp")
mode_map = {
    'motorway': 'major_road', 'motorway_link': 'major_road',
    'trunk': 'major_road', 'trunk_link': 'major_road',
    'primary': 'major_road', 'primary_link': 'major_road',
    'secondary': 'connecting_road', 'secondary_link': 'connecting_road',
    'tertiary': 'connecting_road', 'tertiary_link': 'connecting_road',
    'residential': 'urban_road', 'living_street': 'urban_road',
    'service': 'urban_road', 'busway': 'urban_road',
    'unclassified': 'rural_unpaved', 'track': 'rural_unpaved',
    'track_grade1': 'rural_unpaved', 'track_grade2': 'rural_unpaved',
    'track_grade3': 'rural_unpaved', 'track_grade4': 'rural_unpaved',
    'track_grade5': 'rural_unpaved', 'cycleway': 'rural_unpaved',
    'unknown': 'rural_unpaved', 'path': 'walking', 'footway': 'walking',
    'pedestrian': 'walking', 'steps': 'walking', 'bridleway': 'walking'
}
roads['mode'] = roads['fclass'].map(mode_map).fillna('rural_unpaved')

print("Building road KD-Tree...")
road_midpoints = []
road_modes = []
for _, road in roads.iterrows():
    coords = list(road.geometry.coords)
    for i in range(len(coords) - 1):
        mid_lon = (coords[i][0] + coords[i+1][0]) / 2
        mid_lat = (coords[i][1] + coords[i+1][1]) / 2
        road_midpoints.append([mid_lon, mid_lat])
        road_modes.append(road['mode'])

road_midpoints = np.array(road_midpoints)
road_modes = np.array(road_modes)
road_kd = cKDTree(road_midpoints)
print(f"✅ Road KD-Tree built on {len(road_midpoints):,} segments")

n_pop = len(master_v2)
pop_coords = master_v2[['lon', 'lat']].values

# Run for each grouping
for grouping_name, fac_df in GROUPS.items():
    print(f"\n{'='*65}")
    print(f"  {grouping_name.upper()} JOURNEY BREAKDOWN")
    print(f"{'='*65}")

    fac_coords = fac_df[['Longitude', 'Latitude']].values
    fac_tree = cKDTree(fac_coords)
    k_actual = min(20, len(fac_coords))

    # Result arrays
    major_road   = np.zeros(n_pop)
    connecting   = np.zeros(n_pop)
    urban        = np.zeros(n_pop)
    rural_unpaved= np.zeros(n_pop)
    walking      = np.zeros(n_pop)
    total_time   = np.zeros(n_pop)
    failed       = np.zeros(n_pop, dtype=bool)

    # Check checkpoint
    ckpt_path = f"{project}\\journey_{grouping_name}_checkpoint.pkl"
    start_idx = 0
    try:
        with open(ckpt_path, 'rb') as f:
            ckpt = pickle.load(f)
            major_road    = ckpt['major_road']
            connecting    = ckpt['connecting']
            urban         = ckpt['urban']
            rural_unpaved = ckpt['rural_unpaved']
            walking       = ckpt['walking']
            total_time    = ckpt['total_time']
            failed        = ckpt['failed']
            start_idx     = ckpt['next_idx']
            print(f"  ♻️  Resuming from {start_idx:,}")
    except:
        print(f"  Starting fresh...")

    start_time = time.time()

    for i in range(start_idx, n_pop):

        # Progress
        if i % 500 == 0 and i > start_idx:
            elapsed = time.time() - start_time
            rate = elapsed / (i - start_idx)
            eta = rate * (n_pop - i) / 60
            print(f"  {i:,}/{n_pop:,} ({i/n_pop*100:.1f}%) — ETA: {eta:.1f} min")

        pop_lon = pop_coords[i, 0]
        pop_lat = pop_coords[i, 1]

        # Step 1 — Find nearest facility via K=20 Table API
        _, idxs = fac_tree.query([[pop_lon, pop_lat]], k=k_actual)
        candidates = fac_coords[idxs[0]]

        all_coords = np.vstack([[pop_lon, pop_lat], candidates])
        coord_str = ";".join([f"{lon},{lat}" for lon, lat in all_coords])
        sources = "0"
        dests = ";".join([str(j) for j in range(1, len(all_coords))])

        url = (f"http://localhost:5000/table/v1/driving/{coord_str}"
               f"?sources={sources}&destinations={dests}&annotations=duration")

        try:
            r = requests.get(url, timeout=10)
            d = r.json()

            if d['code'] == 'Ok':
                times = [t/60 if t is not None else np.nan
                        for t in d['durations'][0]]
                best_idx = int(np.nanargmin(times))
                best_lon = candidates[best_idx, 0]
                best_lat = candidates[best_idx, 1]

                # Step 2 — Get route geometry for breakdown
                route_url = (f"http://localhost:5000/route/v1/driving/"
                            f"{pop_lon},{pop_lat};"
                            f"{best_lon},{best_lat}"
                            f"?geometries=geojson&overview=full&annotations=duration")

                route_r = requests.get(route_url, timeout=10)
                route_d = route_r.json()

                if route_d['code'] == 'Ok':
                    coords = route_d['routes'][0]['geometry']['coordinates']
                    seg_durations = route_d['routes'][0]['legs'][0]['annotation']['duration']

                    if len(coords) >= 2 and len(seg_durations) > 0:
                        mids = np.array([
                            [(coords[j][0] + coords[j+1][0]) / 2,
                             (coords[j][1] + coords[j+1][1]) / 2]
                            for j in range(len(coords) - 1)
                        ])

                        _, road_idxs = road_kd.query(mids)
                        modes = road_modes[road_idxs]

                        for j, mode in enumerate(modes):
                            if j < len(seg_durations):
                                t = seg_durations[j] / 60
                                if mode == 'major_road':    major_road[i] += t
                                elif mode == 'connecting_road': connecting[i] += t
                                elif mode == 'urban_road':  urban[i] += t
                                elif mode == 'rural_unpaved': rural_unpaved[i] += t
                                elif mode == 'walking':     walking[i] += t

                        total_time[i] = sum(seg_durations) / 60
                    else:
                        failed[i] = True
                else:
                    failed[i] = True
            else:
                failed[i] = True

        except Exception:
            failed[i] = True

        # Checkpoint every 10,000
        if i > 0 and i % 10000 == 0:
            with open(ckpt_path, 'wb') as f:
                pickle.dump({
                    'major_road': major_road,
                    'connecting': connecting,
                    'urban': urban,
                    'rural_unpaved': rural_unpaved,
                    'walking': walking,
                    'total_time': total_time,
                    'failed': failed,
                    'next_idx': i
                }, f)
            print(f"  💾 Checkpoint saved at {i:,}")

    # Add to master_v2
    master_v2[f'journey_{grouping_name}_major_road_min']   = major_road
    master_v2[f'journey_{grouping_name}_connecting_min']   = connecting
    master_v2[f'journey_{grouping_name}_urban_min']        = urban
    master_v2[f'journey_{grouping_name}_rural_unpaved_min']= rural_unpaved
    master_v2[f'journey_{grouping_name}_walking_min']      = walking
    master_v2[f'journey_{grouping_name}_total_min']        = total_time
    master_v2[f'journey_{grouping_name}_failed']           = failed

    elapsed_total = round((time.time() - start_time) / 60, 1)
    print(f"\n  ✅ {grouping_name.upper()} DONE in {elapsed_total} min!")
    print(f"  Failed: {failed.sum():,}")

    # National summary
    valid = total_time[total_time > 0]
    if len(valid) > 0:
        print(f"\n  === {grouping_name.upper()} JOURNEY BREAKDOWN ===")
        for col, arr in [
            ('rural_unpaved', rural_unpaved),
            ('connecting', connecting),
            ('major_road', major_road),
            ('urban', urban),
            ('walking', walking),
        ]:
            pct = round(arr.sum() / total_time.sum() * 100, 1)
            avg = round(arr.mean(), 1)
            print(f"  {col:<20}: {avg} min avg ({pct}% of journey)")

    # Save intermediate
    master_v2.to_csv(f"{project}\\master_population_data_v2.csv", index=False)
    print(f"  💾 Saved to master_population_data_v2.csv")

print(f"\n🎉 ALL DONE!")

JOURNEY BREAKDOWN — EMERGENCY + SPECIALIST
emergency: 847 facilities
specialist: 28 facilities

Loading road network...
Building road KD-Tree...
✅ Road KD-Tree built on 4,542,352 segments

  EMERGENCY JOURNEY BREAKDOWN
  Starting fresh...
  500/278,001 (0.2%) — ETA: 193.0 min
  1,000/278,001 (0.4%) — ETA: 194.2 min
  1,500/278,001 (0.5%) — ETA: 191.3 min
  2,000/278,001 (0.7%) — ETA: 190.1 min
  2,500/278,001 (0.9%) — ETA: 184.1 min
  3,000/278,001 (1.1%) — ETA: 180.7 min
  3,500/278,001 (1.3%) — ETA: 177.8 min
  4,000/278,001 (1.4%) — ETA: 176.2 min
  4,500/278,001 (1.6%) — ETA: 174.1 min
  5,000/278,001 (1.8%) — ETA: 173.6 min
  5,500/278,001 (2.0%) — ETA: 171.8 min
  6,000/278,001 (2.2%) — ETA: 171.2 min
  6,500/278,001 (2.3%) — ETA: 170.1 min
  7,000/278,001 (2.5%) — ETA: 169.0 min
  7,500/278,001 (2.7%) — ETA: 169.0 min
  8,000/278,001 (2.9%) — ETA: 168.5 min
  8,500/278,001 (3.1%) — ETA: 168.1 min
  9,000/278,001 (3.2%) — ETA: 168.2 min
  9,500/278,001 (3.4%) — ETA: 167.3 min
  1

In [1]:
import pandas as pd
project = r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project"
master_v2 = pd.read_csv(f"{project}\\master_population_data_v2.csv", nrows=1)
journey_cols = [c for c in master_v2.columns if 'journey' in c.lower()]
print(journey_cols)

['journey_major_road_min', 'journey_connecting_road_min', 'journey_urban_road_min', 'journey_rural_unpaved_min', 'journey_walking_min', 'journey_total_min', 'journey_failed', 'journey_emergency_major_road_min', 'journey_emergency_connecting_min', 'journey_emergency_urban_min', 'journey_emergency_rural_unpaved_min', 'journey_emergency_walking_min', 'journey_emergency_total_min', 'journey_emergency_failed', 'journey_specialist_major_road_min', 'journey_specialist_connecting_min', 'journey_specialist_urban_min', 'journey_specialist_rural_unpaved_min', 'journey_specialist_walking_min', 'journey_specialist_total_min', 'journey_specialist_failed']


In [3]:
import pandas as pd
import numpy as np

print("Building district scores with new three-score framework...")

project = r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project"
master_v2 = pd.read_csv(f"{project}\\master_population_data_v2.csv")
print(f"✅ Loaded: {master_v2.shape}")

# ── BAND SCORE FUNCTIONS ──

def emergency_bands(series):
    scores = np.zeros(len(series))
    scores[series <= 30] = 100
    scores[(series > 30) & (series <= 60)] = 70
    scores[(series > 60) & (series <= 90)] = 40
    scores[(series > 90) & (series <= 120)] = 15
    scores[series > 120] = 0
    return scores

def specialist_bands(series):
    scores = np.zeros(len(series))
    scores[series <= 60] = 100
    scores[(series > 60) & (series <= 120)] = 70
    scores[(series > 120) & (series <= 180)] = 40
    scores[series > 180] = 0
    return scores

def road_quality_score(good_pct, bad_pct):
    if good_pct <= 10:   good_score = 10
    elif good_pct <= 30: good_score = 40
    elif good_pct <= 50: good_score = 70
    else:                good_score = 100

    if bad_pct <= 10:   bad_score = 100
    elif bad_pct <= 30: bad_score = 70
    elif bad_pct <= 50: bad_score = 30
    else:               bad_score = 0

    return (good_score + bad_score) / 2

def pct(group, col, lower, upper=None):
    total = len(group)
    if total == 0:
        return 0
    if upper is None:
        mask = group[col] > lower
    elif lower == 0:
        mask = group[col] <= upper
    else:
        mask = (group[col] > lower) & (group[col] <= upper)
    return round(mask.sum() / total * 100, 1)

# ── BUILD DISTRICT SCORES ──
rows = []
total_groups = master_v2.groupby(['Region', 'District']).ngroups
count = 0

for (region, district), group in master_v2.groupby(['Region', 'District']):
    if pd.isna(region) or pd.isna(district):
        continue
    count += 1
    if count % 50 == 0:
        print(f"  {count}/{total_groups} districts...")

    # ── JOURNEY ACCESS SCORE ──
    emergency_band_score  = round(emergency_bands(group['nearest_emergency_min']).mean(), 2)
    specialist_band_score = round(specialist_bands(group['nearest_specialist_min']).mean(), 2)
    any_band_score        = round(emergency_bands(group['nearest_any_min']).mean(), 2)

    journey_access_score = round(
        (emergency_band_score  * 0.50) +
        (specialist_band_score * 0.35) +
        (any_band_score        * 0.15), 2
    )

    # ── JOURNEY QUALITY SCORE ──

    # Any facility road quality
    any_total = group['journey_total_min'].mean()
    if any_total > 0:
        any_good = (group['journey_major_road_min'].mean() +
                    group['journey_connecting_road_min'].mean() +
                    group['journey_urban_road_min'].mean()) / any_total * 100
        any_bad  = (group['journey_rural_unpaved_min'].mean() +
                    group['journey_walking_min'].mean()) / any_total * 100
        any_quality = road_quality_score(any_good, any_bad)
    else:
        any_quality = 0

    # Emergency road quality
    emerg_total = group['journey_emergency_total_min'].mean()
    if emerg_total > 0:
        emerg_good = (group['journey_emergency_major_road_min'].mean() +
                      group['journey_emergency_connecting_min'].mean() +
                      group['journey_emergency_urban_min'].mean()) / emerg_total * 100
        emerg_bad  = (group['journey_emergency_rural_unpaved_min'].mean() +
                      group['journey_emergency_walking_min'].mean()) / emerg_total * 100
        emerg_quality = road_quality_score(emerg_good, emerg_bad)
    else:
        emerg_quality = 0

    # Specialist road quality
    spec_total = group['journey_specialist_total_min'].mean()
    if spec_total > 0:
        spec_good = (group['journey_specialist_major_road_min'].mean() +
                     group['journey_specialist_connecting_min'].mean() +
                     group['journey_specialist_urban_min'].mean()) / spec_total * 100
        spec_bad  = (group['journey_specialist_rural_unpaved_min'].mean() +
                     group['journey_specialist_walking_min'].mean()) / spec_total * 100
        spec_quality = road_quality_score(spec_good, spec_bad)
    else:
        spec_quality = 0

    journey_quality_score = round(
        (emerg_quality * 0.50) +
        (spec_quality  * 0.35) +
        (any_quality   * 0.15), 2
    )

    row = {
        'Region': region,
        'District': district,
        'total_population': round(group['population'].sum()),
        'population_points': len(group),

        # Emergency bands
        'emergency_pct_within_30': pct(group, 'nearest_emergency_min', 0, 30),
        'emergency_pct_30_60':     pct(group, 'nearest_emergency_min', 30, 60),
        'emergency_pct_60_90':     pct(group, 'nearest_emergency_min', 60, 90),
        'emergency_pct_90_120':    pct(group, 'nearest_emergency_min', 90, 120),
        'emergency_pct_120plus':   pct(group, 'nearest_emergency_min', 120),
        'emergency_band_score':    emergency_band_score,
        'emergency_mean_min':      round(group['nearest_emergency_min'].mean(), 1),
        'emergency_min_min':       round(group['nearest_emergency_min'].min(), 1),
        'emergency_max_min':       round(group['nearest_emergency_min'].max(), 1),

        # Any facility bands
        'any_pct_within_30': pct(group, 'nearest_any_min', 0, 30),
        'any_pct_30_60':     pct(group, 'nearest_any_min', 30, 60),
        'any_pct_60_90':     pct(group, 'nearest_any_min', 60, 90),
        'any_pct_90_120':    pct(group, 'nearest_any_min', 90, 120),
        'any_pct_120plus':   pct(group, 'nearest_any_min', 120),
        'any_band_score':    any_band_score,
        'any_mean_min':      round(group['nearest_any_min'].mean(), 1),
        'any_min_min':       round(group['nearest_any_min'].min(), 1),
        'any_max_min':       round(group['nearest_any_min'].max(), 1),

        # Specialist bands
        'specialist_pct_within_60':  pct(group, 'nearest_specialist_min', 0, 60),
        'specialist_pct_60_120':     pct(group, 'nearest_specialist_min', 60, 120),
        'specialist_pct_120_180':    pct(group, 'nearest_specialist_min', 120, 180),
        'specialist_pct_180plus':    pct(group, 'nearest_specialist_min', 180),
        'specialist_band_score':     specialist_band_score,
        'specialist_mean_min':       round(group['nearest_specialist_min'].mean(), 1),
        'specialist_min_min':        round(group['nearest_specialist_min'].min(), 1),
        'specialist_max_min':        round(group['nearest_specialist_min'].max(), 1),

        # Outpatient bands
        'outpatient_pct_within_30': pct(group, 'nearest_outpatient_min', 0, 30),
        'outpatient_pct_30_60':     pct(group, 'nearest_outpatient_min', 30, 60),
        'outpatient_mean_min':      round(group['nearest_outpatient_min'].mean(), 1),

        # Road breakdown — any facility
        'road_pct_major_road':      round(group['journey_major_road_min'].mean() / any_total * 100, 1) if any_total > 0 else 0,
        'road_pct_connecting_road': round(group['journey_connecting_road_min'].mean() / any_total * 100, 1) if any_total > 0 else 0,
        'road_pct_urban_road':      round(group['journey_urban_road_min'].mean() / any_total * 100, 1) if any_total > 0 else 0,
        'road_pct_rural_unpaved':   round(group['journey_rural_unpaved_min'].mean() / any_total * 100, 1) if any_total > 0 else 0,
        'road_pct_walking':         round(group['journey_walking_min'].mean() / any_total * 100, 1) if any_total > 0 else 0,

        # Road breakdown — emergency
        'emerg_road_pct_major':      round(group['journey_emergency_major_road_min'].mean() / emerg_total * 100, 1) if emerg_total > 0 else 0,
        'emerg_road_pct_connecting': round(group['journey_emergency_connecting_min'].mean() / emerg_total * 100, 1) if emerg_total > 0 else 0,
        'emerg_road_pct_urban':      round(group['journey_emergency_urban_min'].mean() / emerg_total * 100, 1) if emerg_total > 0 else 0,
        'emerg_road_pct_unpaved':    round(group['journey_emergency_rural_unpaved_min'].mean() / emerg_total * 100, 1) if emerg_total > 0 else 0,
        'emerg_road_pct_walking':    round(group['journey_emergency_walking_min'].mean() / emerg_total * 100, 1) if emerg_total > 0 else 0,

        # Road breakdown — specialist
        'spec_road_pct_major':      round(group['journey_specialist_major_road_min'].mean() / spec_total * 100, 1) if spec_total > 0 else 0,
        'spec_road_pct_connecting': round(group['journey_specialist_connecting_min'].mean() / spec_total * 100, 1) if spec_total > 0 else 0,
        'spec_road_pct_urban':      round(group['journey_specialist_urban_min'].mean() / spec_total * 100, 1) if spec_total > 0 else 0,
        'spec_road_pct_unpaved':    round(group['journey_specialist_rural_unpaved_min'].mean() / spec_total * 100, 1) if spec_total > 0 else 0,
        'spec_road_pct_walking':    round(group['journey_specialist_walking_min'].mean() / spec_total * 100, 1) if spec_total > 0 else 0,

        # Journey quality sub-scores
        'any_quality_score':   round(any_quality, 2),
        'emerg_quality_score': round(emerg_quality, 2),
        'spec_quality_score':  round(spec_quality, 2),

        # E2SFCA
        'avg_accessibility_score': round(group['accessibility_score'].mean(), 6),
    }
    rows.append(row)

district_scores = pd.DataFrame(rows)

# ── NORMALIZE E2SFCA ──
e_min = district_scores['avg_accessibility_score'].min()
e_max = district_scores['avg_accessibility_score'].max()
district_scores['supply_adequacy_score'] = round(
    (district_scores['avg_accessibility_score'] - e_min) /
    (e_max - e_min) * 100, 2
)

# ── THREE SCORES ──
district_scores['journey_access_score']  = round(
    (district_scores['emergency_band_score']  * 0.50) +
    (district_scores['specialist_band_score'] * 0.35) +
    (district_scores['any_band_score']        * 0.15), 2
)

district_scores['journey_quality_score'] = round(
    (district_scores['emerg_quality_score'] * 0.50) +
    (district_scores['spec_quality_score']  * 0.35) +
    (district_scores['any_quality_score']   * 0.15), 2
)

# ── FINAL COMPOSITE ──
district_scores['composite_score'] = round(
    (district_scores['journey_access_score']  * 0.40) +
    (district_scores['journey_quality_score'] * 0.30) +
    (district_scores['supply_adequacy_score'] * 0.30), 2
)

# ── CATEGORIES ──
def categorize(score):
    if score >= 80:   return 'Thriving'
    elif score >= 60: return 'Decent'
    elif score >= 40: return 'Getting By'
    elif score >= 20: return 'Struggling'
    else:             return 'Dire'

district_scores['category'] = district_scores['composite_score'].apply(categorize)

# ── SAVE ──
out_path = f"{project}\\district_accessibility_scores.csv"
district_scores.to_csv(out_path, index=False)
print(f"\n✅ Saved! {district_scores.shape}")

# ── RESULTS ──
print(f"\n=== CATEGORY DISTRIBUTION ===")
print(district_scores['category'].value_counts())

print(f"\n=== TOP 10 ===")
print(district_scores.nlargest(10, 'composite_score')[
    ['District', 'Region', 'journey_access_score',
     'journey_quality_score', 'supply_adequacy_score',
     'composite_score', 'category']
].to_string(index=False))

print(f"\n=== BOTTOM 10 ===")
print(district_scores.nsmallest(10, 'composite_score')[
    ['District', 'Region', 'journey_access_score',
     'journey_quality_score', 'supply_adequacy_score',
     'composite_score', 'category']
].to_string(index=False))

Building district scores with new three-score framework...
✅ Loaded: (278001, 49)
  50/260 districts...
  100/260 districts...
  150/260 districts...
  200/260 districts...
  250/260 districts...

✅ Saved! (260, 57)

=== CATEGORY DISTRIBUTION ===
category
Decent        171
Getting By     68
Struggling     10
Thriving       10
Dire            1
Name: count, dtype: int64

=== TOP 10 ===
       District        Region  journey_access_score  journey_quality_score  supply_adequacy_score  composite_score category
     Bolga East    Upper East                100.00                  87.25                 100.00            96.18 Thriving
   Wa Municipal    Upper West                 99.15                  77.50                  90.82            90.16 Thriving
  Nadowli-Kaleo    Upper West                 97.65                  77.50                  83.47            87.35 Thriving
         Nabdam    Upper East                 98.69                  82.75                  73.45            86.34 T

In [4]:
# Recompute composite with new weights
# Access 25%, Quality 25%, Supply 50%

district_scores['composite_score_new'] = round(
    (district_scores['journey_access_score']  * 0.25) +
    (district_scores['journey_quality_score'] * 0.25) +
    (district_scores['supply_adequacy_score'] * 0.50), 2
)

def categorize(score):
    if score >= 80:   return 'Thriving'
    elif score >= 60: return 'Decent'
    elif score >= 40: return 'Getting By'
    elif score >= 20: return 'Struggling'
    else:             return 'Dire'

district_scores['category_new'] = district_scores['composite_score_new'].apply(categorize)

print("=== CATEGORY DISTRIBUTION COMPARISON ===\n")
print(f"{'Category':<15} {'Old (40/30/30)':<20} {'New (25/25/50)':<20}")
print("-" * 55)
for cat in ['Thriving', 'Decent', 'Getting By', 'Struggling', 'Dire']:
    old = (district_scores['category'] == cat).sum()
    new = (district_scores['category_new'] == cat).sum()
    print(f"{cat:<15} {old:<20} {new:<20}")

print(f"\n=== TOP 10 (NEW WEIGHTS) ===")
print(district_scores.nlargest(10, 'composite_score_new')[
    ['District', 'Region', 'journey_access_score',
     'journey_quality_score', 'supply_adequacy_score',
     'composite_score_new', 'category_new']
].to_string(index=False))

print(f"\n=== BOTTOM 10 (NEW WEIGHTS) ===")
print(district_scores.nsmallest(10, 'composite_score_new')[
    ['District', 'Region', 'journey_access_score',
     'journey_quality_score', 'supply_adequacy_score',
     'composite_score_new', 'category_new']
].to_string(index=False))

print(f"\n=== SPECIFIC DISTRICTS TO WATCH ===")
watch = ['Ayawaso Central', 'East Gonja', 'Accra', 'Bolga East', 'Savannah']
for d in watch:
    row = district_scores[district_scores['District'] == d]
    if len(row) > 0:
        r = row.iloc[0]
        print(f"\n{d}:")
        print(f"  Old score: {r['composite_score']} ({r['category']})")
        print(f"  New score: {r['composite_score_new']} ({r['category_new']})")

=== CATEGORY DISTRIBUTION COMPARISON ===

Category        Old (40/30/30)       New (25/25/50)      
-------------------------------------------------------
Thriving        10                   5                   
Decent          171                  57                  
Getting By      68                   163                 
Struggling      10                   34                  
Dire            1                    1                   

=== TOP 10 (NEW WEIGHTS) ===
     District     Region  journey_access_score  journey_quality_score  supply_adequacy_score  composite_score_new category_new
   Bolga East Upper East                100.00                  87.25                 100.00                96.81     Thriving
 Wa Municipal Upper West                 99.15                  77.50                  90.82                89.57     Thriving
        Lawra Upper West                 90.50                  70.50                  94.35                87.42     Thriving
Nadowli-Kaleo Up

In [5]:
# Update categories with new names
def categorize(score):
    if score >= 80:   return 'Top Tier'
    elif score >= 60: return 'Almost There'
    elif score >= 40: return 'Managing'
    elif score >= 20: return 'Struggling'
    else:             return 'Crisis'

# Apply new weights and new category names
district_scores['composite_score'] = round(
    (district_scores['journey_access_score']  * 0.25) +
    (district_scores['journey_quality_score'] * 0.25) +
    (district_scores['supply_adequacy_score'] * 0.50), 2
)

district_scores['category'] = district_scores['composite_score'].apply(categorize)

# Save
out_path = f"{project}\\district_accessibility_scores.csv"
district_scores.to_csv(out_path, index=False)
print(f"✅ Saved! {district_scores.shape}")

# Quick check
print(f"\n=== CATEGORY DISTRIBUTION ===")
print(district_scores['category'].value_counts())

print(f"\n=== TOP 5 ===")
print(district_scores.nlargest(5, 'composite_score')[
    ['District', 'Region', 'composite_score', 'category']
].to_string(index=False))

print(f"\n=== BOTTOM 5 ===")
print(district_scores.nsmallest(5, 'composite_score')[
    ['District', 'Region', 'composite_score', 'category']
].to_string(index=False))

✅ Saved! (260, 59)

=== CATEGORY DISTRIBUTION ===
category
Managing        163
Almost There     57
Struggling       34
Top Tier          5
Crisis            1
Name: count, dtype: int64

=== TOP 5 ===
     District     Region  composite_score category
   Bolga East Upper East            96.81 Top Tier
 Wa Municipal Upper West            89.57 Top Tier
        Lawra Upper West            87.42 Top Tier
Nadowli-Kaleo Upper West            85.52 Top Tier
       Nabdam Upper East            82.08 Top Tier

=== BOTTOM 5 ===
                District    Region  composite_score   category
              East Gonja  Savannah            13.74     Crisis
Kwahu Afram Plains South   Eastern            25.55 Struggling
               Sene West Bono East            26.12 Struggling
          Kintampo North Bono East            28.83 Struggling
                   Banda      Bono            29.66 Struggling


In [6]:
import json

# Export district_scores.json
district_json = []
for _, row in district_scores.iterrows():
    district_json.append({
        k: (None if str(v) == 'nan' else v)
        for k, v in row.items()
    })

out_path = f"{project}\\district_scores.json"
with open(out_path, 'w') as f:
    json.dump(district_json, f)

print(f"✅ Saved district_scores.json — {len(district_json)} districts")

✅ Saved district_scores.json — 260 districts


In [7]:
import json

# Export district_scores.json
district_json = []
for _, row in district_scores.iterrows():
    district_json.append({
        k: (None if str(v) == 'nan' else v)
        for k, v in row.items()
    })

out_path = f"{project}\\district_scores.json"
with open(out_path, 'w') as f:
    json.dump(district_json, f)

print(f"✅ Saved district_scores.json — {len(district_json)} districts")

✅ Saved district_scores.json — 260 districts


In [8]:
import pandas as pd
import numpy as np
import json

print("Building region summary...")

# ── REGION LEVEL AGGREGATION ──
region_rows = []

for region, group in master_v2.groupby('Region'):
    if pd.isna(region):
        continue

    # Get district scores for this region
    region_districts = district_scores[district_scores['Region'] == region]

    # Journey totals
    any_total   = group['journey_total_min'].mean()
    emerg_total = group['journey_emergency_total_min'].mean()
    spec_total  = group['journey_specialist_total_min'].mean()

    # Road quality scores
    if any_total > 0:
        any_good = (group['journey_major_road_min'].mean() +
                    group['journey_connecting_road_min'].mean() +
                    group['journey_urban_road_min'].mean()) / any_total * 100
        any_bad  = (group['journey_rural_unpaved_min'].mean() +
                    group['journey_walking_min'].mean()) / any_total * 100
        any_quality = road_quality_score(any_good, any_bad)
    else:
        any_quality = 0

    if emerg_total > 0:
        emerg_good = (group['journey_emergency_major_road_min'].mean() +
                      group['journey_emergency_connecting_min'].mean() +
                      group['journey_emergency_urban_min'].mean()) / emerg_total * 100
        emerg_bad  = (group['journey_emergency_rural_unpaved_min'].mean() +
                      group['journey_emergency_walking_min'].mean()) / emerg_total * 100
        emerg_quality = road_quality_score(emerg_good, emerg_bad)
    else:
        emerg_quality = 0

    if spec_total > 0:
        spec_good = (group['journey_specialist_major_road_min'].mean() +
                     group['journey_specialist_connecting_min'].mean() +
                     group['journey_specialist_urban_min'].mean()) / spec_total * 100
        spec_bad  = (group['journey_specialist_rural_unpaved_min'].mean() +
                     group['journey_specialist_walking_min'].mean()) / spec_total * 100
        spec_quality = road_quality_score(spec_good, spec_bad)
    else:
        spec_quality = 0

    journey_quality_score = round(
        (emerg_quality * 0.50) +
        (spec_quality  * 0.35) +
        (any_quality   * 0.15), 2
    )

    # Band scores
    emergency_band  = round(emergency_bands(group['nearest_emergency_min']).mean(), 2)
    specialist_band = round(specialist_bands(group['nearest_specialist_min']).mean(), 2)
    any_band        = round(emergency_bands(group['nearest_any_min']).mean(), 2)

    journey_access_score = round(
        (emergency_band  * 0.50) +
        (specialist_band * 0.35) +
        (any_band        * 0.15), 2
    )

    # E2SFCA
    avg_e2sfca = group['accessibility_score'].mean()

    # Normalize using same min/max as districts
    e2sfca_normalized = round(
        (avg_e2sfca - e_min) / (e_max - e_min) * 100, 2
    )

    supply_adequacy_score = e2sfca_normalized

    composite_score = round(
        (journey_access_score  * 0.25) +
        (journey_quality_score * 0.25) +
        (supply_adequacy_score * 0.50), 2
    )

    def categorize(score):
        if score >= 80:   return 'Top Tier'
        elif score >= 60: return 'Almost There'
        elif score >= 40: return 'Managing'
        elif score >= 20: return 'Struggling'
        else:             return 'Crisis'

    # Best and worst districts
    best_idx  = region_districts['composite_score'].idxmax()
    worst_idx = region_districts['composite_score'].idxmin()

    row = {
        'Region': region,
        'total_population': round(group['population'].sum()),
        'population_points': len(group),
        'num_districts': len(region_districts),

        # Emergency bands
        'emergency_pct_within_30': round((group['nearest_emergency_min'] <= 30).mean() * 100, 1),
        'emergency_pct_30_60':     round(((group['nearest_emergency_min'] > 30) & (group['nearest_emergency_min'] <= 60)).mean() * 100, 1),
        'emergency_pct_60_90':     round(((group['nearest_emergency_min'] > 60) & (group['nearest_emergency_min'] <= 90)).mean() * 100, 1),
        'emergency_pct_90_120':    round(((group['nearest_emergency_min'] > 90) & (group['nearest_emergency_min'] <= 120)).mean() * 100, 1),
        'emergency_pct_120plus':   round((group['nearest_emergency_min'] > 120).mean() * 100, 1),
        'emergency_mean_min':      round(group['nearest_emergency_min'].mean(), 1),
        'emergency_band_score':    emergency_band,

        # Any facility bands
        'any_pct_within_30': round((group['nearest_any_min'] <= 30).mean() * 100, 1),
        'any_mean_min':      round(group['nearest_any_min'].mean(), 1),
        'any_band_score':    any_band,

        # Specialist bands
        'specialist_pct_within_60': round((group['nearest_specialist_min'] <= 60).mean() * 100, 1),
        'specialist_mean_min':      round(group['nearest_specialist_min'].mean(), 1),
        'specialist_band_score':    specialist_band,

        # Outpatient
        'outpatient_pct_within_30': round((group['nearest_outpatient_min'] <= 30).mean() * 100, 1),
        'outpatient_mean_min':      round(group['nearest_outpatient_min'].mean(), 1),

        # Road breakdown — any
        'road_pct_major_road':      round(group['journey_major_road_min'].mean() / any_total * 100, 1) if any_total > 0 else 0,
        'road_pct_connecting_road': round(group['journey_connecting_road_min'].mean() / any_total * 100, 1) if any_total > 0 else 0,
        'road_pct_urban_road':      round(group['journey_urban_road_min'].mean() / any_total * 100, 1) if any_total > 0 else 0,
        'road_pct_rural_unpaved':   round(group['journey_rural_unpaved_min'].mean() / any_total * 100, 1) if any_total > 0 else 0,
        'road_pct_walking':         round(group['journey_walking_min'].mean() / any_total * 100, 1) if any_total > 0 else 0,

        # Road breakdown — emergency
        'emerg_road_pct_major':      round(group['journey_emergency_major_road_min'].mean() / emerg_total * 100, 1) if emerg_total > 0 else 0,
        'emerg_road_pct_connecting': round(group['journey_emergency_connecting_min'].mean() / emerg_total * 100, 1) if emerg_total > 0 else 0,
        'emerg_road_pct_urban':      round(group['journey_emergency_urban_min'].mean() / emerg_total * 100, 1) if emerg_total > 0 else 0,
        'emerg_road_pct_unpaved':    round(group['journey_emergency_rural_unpaved_min'].mean() / emerg_total * 100, 1) if emerg_total > 0 else 0,
        'emerg_road_pct_walking':    round(group['journey_emergency_walking_min'].mean() / emerg_total * 100, 1) if emerg_total > 0 else 0,

        # Road breakdown — specialist
        'spec_road_pct_major':      round(group['journey_specialist_major_road_min'].mean() / spec_total * 100, 1) if spec_total > 0 else 0,
        'spec_road_pct_connecting': round(group['journey_specialist_connecting_min'].mean() / spec_total * 100, 1) if spec_total > 0 else 0,
        'spec_road_pct_urban':      round(group['journey_specialist_urban_min'].mean() / spec_total * 100, 1) if spec_total > 0 else 0,
        'spec_road_pct_unpaved':    round(group['journey_specialist_rural_unpaved_min'].mean() / spec_total * 100, 1) if spec_total > 0 else 0,
        'spec_road_pct_walking':    round(group['journey_specialist_walking_min'].mean() / spec_total * 100, 1) if spec_total > 0 else 0,

        # Quality sub-scores
        'any_quality_score':   round(any_quality, 2),
        'emerg_quality_score': round(emerg_quality, 2),
        'spec_quality_score':  round(spec_quality, 2),

        # Three scores
        'journey_access_score':   journey_access_score,
        'journey_quality_score':  journey_quality_score,
        'supply_adequacy_score':  supply_adequacy_score,
        'composite_score':        composite_score,
        'category':               categorize(composite_score),

        # Best and worst districts
        'best_district':        region_districts.loc[best_idx, 'District'],
        'best_district_score':  region_districts.loc[best_idx, 'composite_score'],
        'worst_district':       region_districts.loc[worst_idx, 'District'],
        'worst_district_score': region_districts.loc[worst_idx, 'composite_score'],

        # Category counts
        'num_top_tier':     (region_districts['category'] == 'Top Tier').sum(),
        'num_almost_there': (region_districts['category'] == 'Almost There').sum(),
        'num_managing':     (region_districts['category'] == 'Managing').sum(),
        'num_struggling':   (region_districts['category'] == 'Struggling').sum(),
        'num_crisis':       (region_districts['category'] == 'Crisis').sum(),
    }
    region_rows.append(row)

region_summary = pd.DataFrame(region_rows)

# Save CSV
region_summary.to_csv(f"{project}\\region_summary.csv", index=False)
print(f"✅ Saved region_summary.csv — {region_summary.shape}")

# Export JSON
region_json = []
for _, row in region_summary.iterrows():
    region_json.append({
        k: (None if str(v) == 'nan' else v)
        for k, v in row.items()
    })

with open(f"{project}\\region_scores.json", 'w') as f:
    json.dump(region_json, f)
print(f"✅ Saved region_scores.json — {len(region_json)} regions")

# Show results
print(f"\n=== REGIONAL SCORES ===")
print(region_summary[['Region', 'journey_access_score',
                        'journey_quality_score',
                        'supply_adequacy_score',
                        'composite_score',
                        'category']].sort_values(
    'composite_score', ascending=False).to_string(index=False))

Building region summary...
✅ Saved region_summary.csv — (16, 51)
✅ Saved region_scores.json — 16 regions

=== REGIONAL SCORES ===
       Region  journey_access_score  journey_quality_score  supply_adequacy_score  composite_score     category
   Upper East                 86.55                  67.50                  46.74            61.88 Almost There
   Upper West                 72.36                  65.25                  50.23            59.52     Managing
Greater Accra                 97.15                  87.25                  18.84            55.52     Managing
Western North                 65.91                  72.75                  34.48            51.90     Managing
        Ahafo                 85.33                  67.50                  26.39            51.40     Managing
         Bono                 78.46                  67.50                  28.40            50.69     Managing
      Western                 74.67                  67.50                  25.90     

In [9]:
import os
from datetime import datetime

files = [
    f"{project}\\district_accessibility_scores.csv",
    f"{project}\\region_summary.csv",
    f"{project}\\district_scores.json",
    f"{project}\\region_scores.json",
]

print("File timestamps:\n")
for f in files:
    modified = os.path.getmtime(f)
    dt = datetime.fromtimestamp(modified)
    size = os.path.getsize(f) / 1024
    print(f"{f.split(chr(92))[-1]}")
    print(f"  Last modified: {dt.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"  Size: {size:.1f} KB")
    print()

File timestamps:

district_accessibility_scores.csv
  Last modified: 2026-06-05 20:21:42
  Size: 80.1 KB

region_summary.csv
  Last modified: 2026-06-05 20:32:15
  Size: 5.1 KB

district_scores.json
  Last modified: 2026-06-05 20:31:03
  Size: 429.1 KB

region_scores.json
  Last modified: 2026-06-05 20:32:15
  Size: 22.7 KB



In [10]:
import os
path = f"{project}\\district_travel_times.csv"
if os.path.exists(path):
    df = pd.read_csv(path)
    print(f"EXISTS — {df.shape}")
    print(df.columns.tolist())
else:
    print("Does not exist")

EXISTS — (260, 6)
['Region', 'District', 'emergency_median_min', 'any_median_min', 'specialist_median_min', 'psychiatric_median_min']


In [11]:
# Rebuild district_travel_times.csv with mean times and outpatient added
travel_times = master_v2.groupby(['Region', 'District']).agg(
    any_mean_min=('nearest_any_min', 'mean'),
    outpatient_mean_min=('nearest_outpatient_min', 'mean'),
    emergency_mean_min=('nearest_emergency_min', 'mean'),
    specialist_mean_min=('nearest_specialist_min', 'mean'),
).round(1).reset_index()

out_path = f"{project}\\district_travel_times.csv"
travel_times.to_csv(out_path, index=False)
print(f"✅ Saved district_travel_times.csv — {travel_times.shape}")
print(travel_times.head(5).to_string(index=False))

✅ Saved district_travel_times.csv — (260, 6)
Region      District  any_mean_min  outpatient_mean_min  emergency_mean_min  specialist_mean_min
 Ahafo Asunafo North           9.1                 16.1                35.8                103.4
 Ahafo Asunafo South           7.4                 16.7                24.1                104.7
 Ahafo Asutifi North          10.0                 19.4                30.2                 65.2
 Ahafo Asutifi South           7.0                 12.4                18.9                 73.8
 Ahafo    Tano North           7.2                 12.2                19.6                 43.5
